In [ ]:
# -*- coding: utf-8 -*-

# # ── v13 Install dependencies ───────────────────────────────────────────────
# import subprocess, sys
# pkgs = ['librosa','soundfile','scipy','torch','torchaudio','scikit-learn',
#         'tqdm','requests','pydub','remotezip']
# subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade']+pkgs)
# try:
#     import yt_dlp
# except ImportError:
#     subprocess.check_call([sys.executable,'-m','pip','install','-q','yt-dlp'])
# print('✅ Dependencies installed')

# ──────────────────────────────────────────────────────────────────────────────
# Installs (run once in Colab):
#   !pip install -q soundfile librosa torch torchvision torchaudio \
#                   scikit-learn matplotlib tqdm requests pydub remotezip yt-dlp
# ──────────────────────────────────────────────────────────────────────────────

import os, subprocess, sys, json, math, time, random, shutil, zipfile, tempfile
import urllib.request, urllib.parse, re, warnings
from pathlib import Path
from datetime import datetime
from collections import deque
from dataclasses import dataclass
from typing import List, Optional, Tuple, Dict, Any

import numpy as np
import soundfile as sf
import librosa
import scipy.signal
import scipy.optimize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import classification_report, confusion_matrix
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
from matplotlib.colors import Normalize
import requests
from tqdm.auto import tqdm

try:
    from pydub import AudioSegment
    _PYDUB_OK = True
except ImportError:
    _PYDUB_OK = False

try:
    import yt_dlp as ytdlp
    _YTDLP_OK = True
except ImportError:
    _YTDLP_OK = False


# ══════════════════════════════════════════════════════════════════════════════
# Config
# ══════════════════════════════════════════════════════════════════════════════

class Config:
    def __init__(self):
        self.IN_COLAB = "google.colab" in sys.modules
        if self.IN_COLAB:
            self._mount_drive()
        self._setup_paths()

        self.SR              = 22050
        self.TARGET_DURATION = 3.0
        self.N_MELS          = 64
        self.HOP_LENGTH      = 256
        self.N_FFT           = 1024

        self.MIC_POSITIONS = np.array(
            [[0.00, 0.00], [0.20, 0.00], [0.10, 0.1732]], dtype=np.float32
        )
        self.SPEED_OF_SOUND        = 343.0
        self.ARRAY_CENTER          = self.MIC_POSITIONS.mean(axis=0)
        self.MAX_LOCALIZATION_DIST = 25.0

        self.UAVIRBASE_ZIP_URL     = (
            "https://zenodo.org/records/15391924/files/"
            "Microphone_array.zip?download=1"
        )
        self.UAVIRBASE_MIC_INDICES = [0, 1, 2]
        self.UAVIRBASE_ORIG_SR     = 96000
        self.UAVIRBASE_FULL        = False
        self.UAVIRBASE_N_SESSIONS  = 300

        self.DRONEDS_ZIP_URL = (
            "https://github.com/saraalemadi/DroneAudioDataset/"
            "archive/refs/heads/master.zip"
        )

        # Scraping
        self.FREESOUND_API_KEY    = ""   # optional
        self.SCRAPE_MAX_PER_QUERY = 50
        self.SCRAPE_MIN_DURATION  = 2.0
        self.SCRAPE_MAX_DURATION  = 15.0
        self.SCRAPE_YTDLP_URLS    = [
            "https://www.youtube.com/watch?v=5PSNL5qB3xQ",
            "https://www.youtube.com/watch?v=ZKvZv8sLPAo",
        ]
        self.SCRAPE_YTDLP_ENABLED = False

        self.SYNTHETIC_DET_SAMPLES = 500
        self.SYNTHETIC_SAMPLES     = 1000

        self.BATCH_SIZE  = 32
        self.NUM_EPOCHS  = 30
        self.LR          = 5e-4
        self.SEED        = 42
        self.DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
        self.USE_AMP     = True

        self._gpu_mem_gb  = self._detect_gpu_mem()
        self.USE_LITE_LOC = (self._gpu_mem_gb < 4.0)

        self.DETECTION_THRESHOLD      = 0.70
        self.DETECTION_THRESHOLD_LOW  = 0.35
        self.DETECTION_THRESHOLD_WEAK = 0.20
        self.CNN_WEIGHT               = 0.65
        self.HEURISTIC_WEIGHT         = 0.35

        self.EXTERNAL_INFER_THRESHOLD  = 0.35
        self.EXTERNAL_MIN_POS_SEGMENTS = 1
        self.EXTERNAL_SEGMENT_SEC      = 1.0
        self.EXTERNAL_SEGMENT_OVERLAP  = 0.5
        self.EXTERNAL_AGG_MODE         = "mean_topk"
        self.EXTERNAL_TOPK             = 3

        self.MIXED_DRONE_SAMPLES   = 1200
        self.MIXED_DRONE_VAL_FRAC  = 0.15
        self.MIX_SNR_DB_RANGE      = (-5.0, 15.0)
        self.MIX_GAIN_RANGE_DB     = (-8.0, 8.0)
        self.MIX_BG_GAIN_RANGE_DB  = (-6.0, 6.0)
        self.MIX_BACKGROUND_LABELS = ["speech","crowd","wind","traffic","non_drone"]
        self.MIX_CACHE_PREFIX      = "mixdrone"

        self.LOC_CONFIDENCE_CAP = 5.0
        self.KF_PROCESS_NOISE   = 0.5
        self.KF_MEASURE_NOISE   = 0.3
        self.KF_MAX_COAST       = 5
        self.KF_MIN_HITS        = 2
        self.KF_MATCH_GATE      = 2.0
        self.MAX_DRONES         = 3
        self.TDOA_DEDUP_MS      = 0.05e-3

    @staticmethod
    def _detect_gpu_mem():
        if not torch.cuda.is_available(): return 0.0
        try: return torch.cuda.get_device_properties(0).total_memory / (1024**3)
        except: return 0.0

    def _setup_paths(self):
        base  = "/content/drone_v14" if self.IN_COLAB else "/tmp/drone_v14"
        drive = "/content/drive/MyDrive/drone_v14" if self.IN_COLAB else "/tmp/drone_v14"
        B, D  = Path(base), Path(drive)
        self.LOCAL_BASE    = B;  self.RAW_DIR       = B/"raw"
        self.PROCESSED_DIR = B/"processed"; self.MEL_CACHE_DIR = B/"mel_cache"
        self.DRIVE_ROOT    = D;  self.DRIVE_MODELS  = D/"models"
        self.DRIVE_LOGS    = D/"logs"; self.DRIVE_TRACKS = D/"tracks"
        self.DRIVE_PLOTS   = D/"logs/plots"
        self.UAVIRBASE_RAW = B/"uavirbase"; self.DRONEDS_RAW = B/"droneds"

    def _mount_drive(self):
        try:
            from google.colab import drive
            drive.mount("/content/drive", force_remount=False)
        except Exception as e: print(f"Drive mount failed: {e}")

    def ensure_dirs(self):
        for p in [self.RAW_DIR, self.PROCESSED_DIR, self.MEL_CACHE_DIR,
                  self.UAVIRBASE_RAW, self.DRONEDS_RAW]:
            os.makedirs(str(p), exist_ok=True)
        for p in [self.DRIVE_ROOT, self.DRIVE_MODELS, self.DRIVE_LOGS,
                  self.DRIVE_TRACKS, self.DRIVE_PLOTS]:
            try: os.makedirs(str(p), exist_ok=True)
            except OSError as e: print(f"⚠️  {p}: {e}")


config = Config()


# ══════════════════════════════════════════════════════════════════════════════
# Utilities
# ══════════════════════════════════════════════════════════════════════════════

def sigmoid(x):    return float(1.0/(1.0+math.exp(-x)))
def wrap_angle_deg(a): return float((a+180.0)%360.0-180.0)
def safe_slug(n):
    return "".join(c if c.isalnum() or c in("-","_") else "_" for c in n).strip("_") or "out"
def rms_energy(y): return float(np.sqrt(np.mean(np.asarray(y,np.float32)**2))+1e-8)
def db_to_gain(db): return float(10**(db/20.0))
def normalize_peak(y, peak=0.98):
    y=np.asarray(y,np.float32); m=float(np.max(np.abs(y))+1e-8)
    return (y*(peak/m)).astype(np.float32) if m>0 else y

def angular_error_deg(p,t):
    diff=(np.asarray(p)-np.asarray(t)); diff=(diff+180.0)%360.0-180.0; return np.abs(diff)

def classify_detection_score(s,cfg):
    if s>=cfg.DETECTION_THRESHOLD: return "drone"
    if s>=cfg.DETECTION_THRESHOLD_LOW: return "possible_drone"
    if s>=cfg.DETECTION_THRESHOLD_WEAK: return "weak_possible_drone"
    return "non_drone"

def xy_to_azimuth_deg(xy,center):
    return wrap_angle_deg(np.degrees(np.arctan2(float(xy[1]-center[1]),float(xy[0]-center[0]))))

def azimuth_deg_to_xy(az,dist,center):
    r=np.radians(az)
    return np.array([center[0]+dist*np.cos(r), center[1]+dist*np.sin(r)],dtype=np.float32)

def _set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def _ensure_remotezip():
    try: import remotezip
    except ImportError:
        subprocess.check_call([sys.executable,"-m","pip","install","-q","remotezip"])

def moving_average(x,k):
    if k<=1 or len(x)<2: return x.astype(np.float32)
    k=max(1,int(k)); pad=k//2
    xp=np.pad(x,(pad,pad),mode="edge"); w=np.ones(k)/k
    return np.convolve(xp,w,mode="valid")[:len(x)].astype(np.float32)

def safe_prob_average(values, default=0.0):
    vals = [float(v) for v in values if v is not None and not math.isnan(float(v))]
    return float(np.mean(vals)) if vals else float(default)

def robust_diff(x: np.ndarray, dt: float) -> np.ndarray:
    if len(x) < 2: return np.zeros_like(x, dtype=np.float32)
    return np.gradient(x.astype(np.float64), dt).astype(np.float32)

def random_crop_or_loop(y: np.ndarray, target_n: int) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if len(y) == 0: return np.zeros(target_n, dtype=np.float32)
    if len(y) == target_n: return y
    if len(y) > target_n:
        start = random.randint(0, len(y) - target_n)
        return y[start:start+target_n]
    reps = int(np.ceil(target_n / max(len(y), 1)))
    return np.tile(y, reps)[:target_n]

def mix_at_snr(drone_y: np.ndarray, bg_y: np.ndarray, snr_db: float) -> np.ndarray:
    """Mix drone + background at a target SNR (dB)."""
    drone_y = np.asarray(drone_y, dtype=np.float32)
    bg_y    = np.asarray(bg_y, dtype=np.float32)
    d_rms   = rms_energy(drone_y); b_rms = rms_energy(bg_y)
    if b_rms < 1e-8: return drone_y
    scale = (d_rms / (10 ** (snr_db / 20.0))) / b_rms
    return normalize_peak(drone_y + bg_y * scale)

def _parabolic_peak(y: np.ndarray, x: int) -> float:
    if x <= 0 or x >= len(y)-1: return float(x)
    y1, y2, y3 = y[x-1], y[x], y[x+1]
    d = y1 - 2*y2 + y3
    if abs(d) < 1e-12: return float(x)
    return x + 0.5*(y1-y3)/d

def gcc_phat(sig: np.ndarray, ref: np.ndarray, fs: int, max_tau: float, interp: int = 4):
    n   = len(sig) + len(ref)
    S   = np.fft.rfft(sig, n=n); R = np.fft.rfft(ref, n=n)
    if np.max(np.abs(S)) < 1e-10 or np.max(np.abs(R)) < 1e-10:
        nlags = 2*int(interp*fs*max_tau)+1
        return 0.0, np.zeros(nlags), np.zeros(nlags)
    X   = S * np.conj(R); den = np.abs(X); den[den < 1e-10] = 1e-10; X /= den
    cc  = np.fft.irfft(X, n=interp*n)
    ms  = min(int(interp*n/2), int(interp*fs*max_tau))
    cc  = np.concatenate((cc[-ms:], cc[:ms+1]))
    pk  = int(np.argmax(np.abs(cc))); pk_f = _parabolic_peak(np.abs(cc), pk)
    tau = (pk_f - ms) / (interp * fs)
    lags = np.arange(-ms, ms+1) / (interp*fs)
    return tau, lags, np.abs(cc)

def _fractional_delay(signal: np.ndarray, delay_samples: float) -> np.ndarray:
    if delay_samples < 0: delay_samples = 0.0
    int_d = int(np.floor(delay_samples)); frac = delay_samples - int_d
    taps  = np.arange(-3, 5)
    h     = np.sinc(taps - frac) * np.hanning(len(taps)); h /= h.sum() + 1e-12
    filt  = np.convolve(signal.astype(np.float64), h, mode="full")[:len(signal)]
    if int_d > 0:
        return np.concatenate([np.zeros(int_d), filt])[:len(signal)].astype(np.float32)
    return filt.astype(np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# Audio Processing
# ══════════════════════════════════════════════════════════════════════════════

class AudioProcessor:
    def __init__(self, cfg=None): self.cfg = cfg or config

    def load(self, path, mono=True):
        path=str(path)
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            try:
                y,_=librosa.load(path,sr=self.cfg.SR,mono=mono); return y.astype(np.float32)
            except Exception: pass
        if _PYDUB_OK:
            tmp=Path(tempfile.mktemp(suffix=".wav"))
            try:
                AudioSegment.from_file(path).export(str(tmp),format="wav")
                y,_=librosa.load(str(tmp),sr=self.cfg.SR,mono=mono)
                return y.astype(np.float32)
            except Exception as e: raise RuntimeError(f"Cannot load {path}: {e}")
            finally: tmp.unlink(missing_ok=True)
        raise RuntimeError(f"Cannot load {path} (pydub not installed)")

    def load_channels(self, path, channel_indices=None):
        with warnings.catch_warnings():
            warnings.filterwarnings("ignore")
            y,_=librosa.load(str(path),sr=self.cfg.SR,mono=False)
        if y.ndim==1: y=y[np.newaxis,:]
        idx=channel_indices if channel_indices is not None else list(range(y.shape[0]))
        return [y[i].astype(np.float32) for i in idx if i<y.shape[0]]

    def pad_or_truncate(self, y):
        n=int(self.cfg.SR*self.cfg.TARGET_DURATION)
        if len(y)>=n: return y[:n].astype(np.float32)
        return np.pad(y,(0,n-len(y))).astype(np.float32)

    def mel(self, y):
        M=librosa.feature.melspectrogram(
            y=y,sr=self.cfg.SR,n_fft=self.cfg.N_FFT,
            hop_length=self.cfg.HOP_LENGTH,n_mels=self.cfg.N_MELS,
            fmin=20,fmax=8000)
        return librosa.power_to_db(M,ref=np.max).astype(np.float32)

    def add_noise(self, y, snr_db):
        sr=rms_energy(y); n=np.random.randn(len(y)).astype(np.float32)
        n/=(rms_energy(n)+1e-8); scale=sr/(10**(snr_db/20.0))
        return np.clip(y+n*scale,-1.0,1.0).astype(np.float32)


def load_audio_any(path, sr):
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore")
        try: y,_=librosa.load(str(path),sr=sr,mono=True); return y.astype(np.float32)
        except Exception: pass
    if _PYDUB_OK:
        tmp=Path(tempfile.mktemp(suffix=".wav"))
        try:
            AudioSegment.from_file(str(path)).export(str(tmp),format="wav")
            y,_=librosa.load(str(tmp),sr=sr,mono=True); return y.astype(np.float32)
        finally: tmp.unlink(missing_ok=True)
    raise RuntimeError(f"Cannot load {path}")

def convert_to_wav(src, dst):
    if src.suffix.lower()==".wav": shutil.copy2(str(src),str(dst))
    elif _PYDUB_OK: AudioSegment.from_file(str(src)).export(str(dst),format="wav")
    else: raise RuntimeError(f"pydub not available for {src.suffix}")


# ══════════════════════════════════════════════════════════════════════════════
# IPD / TDOA / Signal helpers
# ══════════════════════════════════════════════════════════════════════════════

def compute_ipd_features(channels, cfg):
    sr=cfg.SR; pairs=[(0,1),(0,2),(1,2)]; ipds=[]
    for i,j in pairs:
        xi=channels[i].astype(np.float64); xj=channels[j].astype(np.float64)
        n=max(len(xi),len(xj))
        Xi=np.fft.rfft(xi,n=n); Xj=np.fft.rfft(xj,n=n)
        cc=Xi*np.conj(Xj); cc/=(np.abs(cc)+1e-8)
        tau=np.fft.irfft(cc,n=n)
        pk=int(np.argmax(np.abs(tau)))
        if pk>n//2: pk-=n
        ipds.append(float(pk)/sr)
    return np.array(ipds,dtype=np.float32)

def bandpass(y, sr, lo, hi):
    sos=scipy.signal.butter(4,[lo/(sr/2),hi/(sr/2)],btype="band",output="sos")
    return scipy.signal.sosfilt(sos,y).astype(np.float32)

def gcc_phat_peaks(x1, x2, sr, max_tau, n_peaks):
    n=len(x1)+len(x2)-1
    X1=np.fft.rfft(x1,n=n); X2=np.fft.rfft(x2,n=n)
    cc=X1*np.conj(X2); cc/=(np.abs(cc)+1e-8)
    r=np.fft.irfft(cc,n=n); ml=int(max_tau*sr)
    rt=np.concatenate([r[-ml:],r[:ml+1]]); lags=np.arange(-ml,ml+1)/sr
    peaks=[]; rw=rt.copy()
    for _ in range(n_peaks):
        idx=int(np.argmax(rw)); peaks.append((float(lags[idx]),float(rw[idx])))
        lo=max(0,idx-3); hi=min(len(rw),idx+4); rw[lo:hi]=0.0
    return peaks

def synthesise_drone(mic_positions, src_xy, fundamental=100, noise_level=0.03,
                     duration=None, sr=None):
    sr=sr or config.SR; dur=duration or config.TARGET_DURATION
    n=int(sr*dur); t=np.linspace(0,dur,n,endpoint=False); c=config.SPEED_OF_SOUND
    src=np.asarray(src_xy,dtype=np.float64); channels=[]
    for mic in mic_positions:
        dist=max(float(np.linalg.norm(src-mic)),0.01)
        sd=int(dist/c*sr); y=np.zeros(n,dtype=np.float64)
        for k in range(1,9):
            amp=1.0/(k**1.3)*(0.9+0.2*random.random()); ph=random.uniform(0,2*np.pi)
            jit=1.0+0.003*np.sin(2*np.pi*0.5*t)
            y+=amp*np.sin(2*np.pi*fundamental*k*jit*t+ph)
        y/=(dist**0.6+0.1)
        wn=np.random.randn(n); b,a=scipy.signal.butter(1,0.05)
        pink=scipy.signal.lfilter(b,a,wn)
        y+=noise_level*pink+noise_level*0.5*np.random.randn(n)
        if sd>0: y=np.concatenate([np.zeros(sd),y[:-sd]])
        channels.append(y.astype(np.float32))
    return channels


# ══════════════════════════════════════════════════════════════════════════════
# Enhancement ① — Multi-Source Audio Web Scraper
# ══════════════════════════════════════════════════════════════════════════════

class AudioWebScraper:
    """
    Downloads drone and non-drone audio from 5 free public sources:
      - BBC Sound Effects (no key)
      - xeno-canto (no key)
      - SoundBible (no key)
      - FreeSound.io (no key)
      - Freesound.org (optional API key extends results)
      - yt-dlp YouTube extraction (opt-in)
    """
    AUDIO_EXTS = (".mp3",".wav",".ogg",".flac")
    _BBC_DRONE  = ["drone","buzz","propeller","rotor"]
    _BBC_NON    = ["wind","crowd","traffic","rain","birds","urban"]
    _SOUNDBIBLE = {"non_drone": ["1575","1480","1350","1288","1192"]}

    def __init__(self, cfg):
        self.cfg=cfg; self.api_key=getattr(cfg,"FREESOUND_API_KEY","")
        self.out_root=cfg.RAW_DIR/"scraped_audio"
        self.sess=requests.Session()
        self.sess.headers.update({"User-Agent":"DroneDetectionResearch/1.0"})

    def download(self, force=False):
        for label in ["drone","non_drone"]:
            (self.out_root/label).mkdir(parents=True,exist_ok=True)
        n=self._count()
        if n>0 and not force:
            print(f"✅ Scraped audio already exists ({n} files) — skipping."); return
        print("🌐 Multi-source audio scraping …")
        self._scrape_freesound()
        self._scrape_freesound_io()
        self._scrape_bbc()
        self._scrape_xeno_canto()
        self._scrape_soundbible()
        if getattr(self.cfg,"SCRAPE_YTDLP_ENABLED",False) and _YTDLP_OK:
            self._scrape_youtube()
        print(f"✅ Scraping done — {self._count()} files collected.")

    # ── per-source scrapers ──────────────────────────────────────────────────

    def _scrape_freesound(self):
        if not self.api_key:
            print("  ℹ️  No FREESOUND_API_KEY — skipping Freesound.org"); return
        print("  🔎 Freesound.org …")
        base="https://freesound.org/apiv2/search/text/"
        queries={
            "drone":["drone flying","quadcopter","uav sound","drone propeller","multirotor"],
            "non_drone":["wind","car passing","crowd noise","bird chirping","construction"],
        }
        for label,terms in queries.items():
            sd=self.out_root/label
            for term in terms:
                try:
                    data=self.sess.get(base,params={
                        "query":term,"filter":"duration:[2 TO 15]",
                        "fields":"id,name,previews,duration",
                        "page_size":self.cfg.SCRAPE_MAX_PER_QUERY,
                        "token":self.api_key},timeout=10).json()
                except Exception as e: print(f"    ⚠️  FS ({term}): {e}"); continue
                for s in data.get("results",[]):
                    d=s.get("duration",0)
                    if not(self.cfg.SCRAPE_MIN_DURATION<=d<=self.cfg.SCRAPE_MAX_DURATION): continue
                    fp=sd/f"fs_{s['id']}.mp3"
                    if fp.exists(): continue
                    try:
                        c=self.sess.get(s["previews"]["preview-hq-mp3"],timeout=10).content
                        if len(c)>5000: fp.write_bytes(c)
                    except Exception: pass

    def _scrape_freesound_io(self):
        print("  🔎 FreeSound.io (key-free) …")
        base="https://freesound.io/api/sounds/search"
        queries={"drone":["drone","uav","quadcopter"],"non_drone":["wind","crowd","ambient"]}
        for label,terms in queries.items():
            sd=self.out_root/label
            for term in terms:
                try:
                    resp=self.sess.get(base,params={"query":term,"limit":20},timeout=8)
                    data=resp.json()
                    items=data.get("results",data if isinstance(data,list) else [])
                    for item in items:
                        url=item.get("download_url") or item.get("url","")
                        if not url or not any(url.endswith(e) for e in self.AUDIO_EXTS): continue
                        fid=re.sub(r"[^\w]","_",url[-20:]); ext=Path(url).suffix or ".mp3"
                        fp=sd/f"fsio_{fid}{ext}"
                        if fp.exists(): continue
                        try:
                            c=self.sess.get(url,timeout=10).content
                            if len(c)>5000: fp.write_bytes(c)
                        except Exception: pass
                except Exception as e: print(f"    ⚠️  FSio ({term}): {e}")

    def _scrape_bbc(self):
        print("  🔎 BBC Sound Effects …")
        base="https://sound-effects.bbcrewind.co.uk/api/search"
        qs={"drone":self._BBC_DRONE,"non_drone":self._BBC_NON}
        for label,terms in qs.items():
            sd=self.out_root/label
            for term in terms:
                try:
                    resp=self.sess.get(base,params={"q":term,"limit":15},timeout=8)
                    data=resp.json(); sounds=data.get("results",data.get("sounds",[]))
                    for s in sounds:
                        sid=s.get("id",s.get("assetId",""))
                        if not sid: continue
                        url=f"https://sound-effects-media.bbcrewind.co.uk/zip/{sid}.wav"
                        fp=sd/f"bbc_{sid}.wav"
                        if fp.exists(): continue
                        try:
                            c=self.sess.get(url,timeout=12).content
                            if len(c)>10000: fp.write_bytes(c)
                        except Exception: pass
                except Exception as e: print(f"    ⚠️  BBC ({term}): {e}")

    def _scrape_xeno_canto(self):
        print("  🔎 xeno-canto …")
        sd=self.out_root/"non_drone"
        for term in ["wind","rain","stream","ambient"]:
            try:
                data=self.sess.get("https://xeno-canto.org/api/2/recordings",
                                   params={"query":term,"page":1},timeout=8).json()
                for rec in data.get("recordings",[])[:10]:
                    url="https:"+rec.get("file","")
                    if url=="https:": continue
                    fp=sd/f"xc_{rec.get('id','')}.mp3"
                    if fp.exists(): continue
                    try:
                        c=self.sess.get(url,timeout=12).content
                        if len(c)>5000: fp.write_bytes(c)
                    except Exception: pass
            except Exception as e: print(f"    ⚠️  XC ({term}): {e}")

    def _scrape_soundbible(self):
        print("  🔎 SoundBible …")
        sd=self.out_root/"non_drone"
        for sid in self._SOUNDBIBLE["non_drone"]:
            fp=sd/f"sb_{sid}.mp3"
            if fp.exists(): continue
            try:
                c=self.sess.get(f"https://soundbible.com/grab.php?id={sid}&type=mp3",
                                timeout=10).content
                if len(c)>5000: fp.write_bytes(c)
            except Exception: pass

    def _scrape_youtube(self):
        if not _YTDLP_OK: print("  ⚠️  yt-dlp not installed."); return
        print("  🔎 yt-dlp (YouTube) …"); sd=self.out_root/"drone"
        opts={"format":"bestaudio/best","outtmpl":str(sd/"yt_%(id)s.%(ext)s"),
              "postprocessors":[{"key":"FFmpegExtractAudio","preferredcodec":"wav"}],
              "download_sections":{"*":"00:00:00-00:00:30"},"quiet":True,"no_warnings":True}
        for url in getattr(self.cfg,"SCRAPE_YTDLP_URLS",[]):
            try:
                with ytdlp.YoutubeDL(opts) as ydl: ydl.download([url])
            except Exception as e: print(f"    ⚠️  yt-dlp ({url}): {e}")

    def _count(self):
        n=0
        for label in ["drone","non_drone"]:
            d=self.out_root/label
            if d.exists():
                for ext in self.AUDIO_EXTS: n+=len(list(d.glob(f"*{ext}")))
        return n


def _incorporate_scraped_audio(cfg, force=False):
    scraped=cfg.RAW_DIR/"scraped_audio"
    if not scraped.exists():
        print("ℹ️  No scraped audio — skipping."); return
    det=cfg.PROCESSED_DIR/"detection"; ap=AudioProcessor(cfg)
    for label in ["drone","non_drone"]:
        src=scraped/label
        if not src.exists(): continue
        files=[]
        for ext in AudioWebScraper.AUDIO_EXTS: files.extend(src.glob(f"*{ext}"))
        if not files: continue
        random.shuffle(files); si=int(len(files)*0.85)
        sm={"train":files[:si],"val":files[si:]}
        added=skipped=failed=0
        for split,fl in sm.items():
            dst=det/split/label; dst.mkdir(parents=True,exist_ok=True)
            for f in fl:
                out=dst/f"{f.stem}.wav"
                if out.exists() and not force: skipped+=1; continue
                try:
                    y=ap.pad_or_truncate(load_audio_any(f,cfg.SR))
                    sf.write(str(out),y,cfg.SR); added+=1
                except Exception as e: failed+=1; print(f"   ⚠️  {f.name}: {e}")
        print(f"   ✅ {label}: added={added} skip={skipped} fail={failed}")


def collect_background_pool(cfg):
    pool={k:[] for k in ["speech","crowd","wind","traffic","non_drone"]}
    for split in ["train","val","test"]:
        d=cfg.PROCESSED_DIR/"detection"/split/"non_drone"
        if not d.exists(): continue
        for f in d.glob("*.wav"):
            pool["non_drone"].append(f); nm=f.stem.lower()
            for kws,bkt in [
                (["speech","talk","voice"],"speech"),
                (["crowd","market"],"crowd"),
                (["wind","breeze"],"wind"),
                (["traffic","car","road"],"traffic")]:
                if any(k in nm for k in kws): pool[bkt].append(f)
    scraped=cfg.RAW_DIR/"scraped_audio"/"non_drone"
    if scraped.exists():
        for f in scraped.glob("*.*"):
            if f.suffix.lower() not in AudioWebScraper.AUDIO_EXTS: continue
            pool["non_drone"].append(f); nm=f.stem.lower()
            for kws,bkt in [
                (["speech","talk"],"speech"),(["crowd"],"crowd"),
                (["wind"],"wind"),(["traffic","car"],"traffic")]:
                if any(k in nm for k in kws): pool[bkt].append(f)
    for k in ["speech","crowd","wind","traffic"]:
        if not pool[k]: pool[k]=list(pool["non_drone"])
    for k in pool:
        seen,uniq=set(),[]
        for p in pool[k]:
            s=str(p)
            if s not in seen: uniq.append(p); seen.add(s)
        pool[k]=uniq
    print("📚 Background pool:")
    for k in pool: print(f"   {k:10s}: {len(pool[k])}")
    return pool


# ══════════════════════════════════════════════════════════════════════════════
# Mel cache pipeline
# ══════════════════════════════════════════════════════════════════════════════

class MelCacheManager:
    def __init__(self, cfg): self.cfg=cfg; self.ap=AudioProcessor(cfg)

    def build(self, force: bool = False):
        """Build the full mel cache from all processed WAV splits."""
        cache_root = self.cfg.MEL_CACHE_DIR
        n_existing = len(list(cache_root.rglob("*.npy")))
        if not force and n_existing > 100:
            print(f"✅ Mel cache already exists ({n_existing} files) — skipping.")
            return
        if force and cache_root.exists():
            shutil.rmtree(str(cache_root))
        print("🎵 Building mel cache from processed WAVs …")
        det_root = self.cfg.PROCESSED_DIR / "detection"
        wavs = []
        for split in ["train","val","test"]:
            for label in ["drone","non_drone"]:
                src = det_root/split/label
                if src.exists():
                    for wav in src.glob("*.wav"):
                        wavs.append((split, label, wav))
        total = 0
        for split, label, wav in tqdm(wavs, desc="Mel cache"):
            dst = cache_root/split/label; dst.mkdir(parents=True, exist_ok=True)
            out = dst/f"{wav.stem}.npy"
            if out.exists() and not force: continue
            try:
                y = self.ap.pad_or_truncate(self.ap.load(wav))
                m = self.ap.mel(y)
                np.save(str(out), np.stack([m,m,m], axis=0)); total += 1
            except Exception as e: print(f"   ⚠️  {wav.name}: {e}")
        print(f"✅ Mel cache built ({total} new files).")

    def count(self) -> dict:
        out = {}
        for split in ["train","val","test"]:
            for label in ["drone","non_drone"]:
                d = self.cfg.MEL_CACHE_DIR/split/label
                out[f"{split}/{label}"] = len(list(d.glob("*.npy"))) if d.exists() else 0
        return out

    def build_detection_cache(self, force=False):
        det=self.cfg.PROCESSED_DIR/"detection"
        cache=self.cfg.MEL_CACHE_DIR/"detection"
        for split in ["train","val","test"]:
            for label in ["drone","non_drone"]:
                src=det/split/label; dst=cache/split/label
                if not src.exists(): continue
                dst.mkdir(parents=True,exist_ok=True)
                wavs=list(src.glob("*.wav"))
                print(f"  Caching {split}/{label}: {len(wavs)} files …")
                for wav in tqdm(wavs,desc=f"{split}/{label}",leave=False):
                    out=dst/(wav.stem+".npy")
                    if out.exists() and not force: continue
                    try:
                        y=self.ap.pad_or_truncate(self.ap.load(wav))
                        m=self.ap.mel(y); arr=np.stack([m,m,m],axis=0)
                        np.save(str(out),arr)
                    except Exception as e: print(f"    ⚠️  {wav.name}: {e}")
        self._inject_synthetic(cache,force=force)
        print("✅ Detection mel cache built.")

    def _inject_synthetic(self, cache, force=False):
        n=self.cfg.SYNTHETIC_DET_SAMPLES
        if n<=0: return
        out=cache/"train"/"drone"; out.mkdir(parents=True,exist_ok=True)
        if len(list(out.glob("synth_det_*.npy")))>=n and not force:
            print(f"  ✅ Synthetic injection already done."); return
        print(f"  🔬 Injecting {n} synthetic mels …")
        rng=np.random.default_rng(self.cfg.SEED)
        cx,cy=self.cfg.ARRAY_CENTER
        for i in tqdm(range(n),desc="SynthInject",leave=False):
            r=rng.uniform(0.5,self.cfg.MAX_LOCALIZATION_DIST)
            theta=rng.uniform(0,2*np.pi)
            xy=[cx+r*np.cos(theta),cy+r*np.sin(theta)]
            fund=int(rng.choice([80,90,100,110,120,130]))
            chs=synthesise_drone(self.cfg.MIC_POSITIONS,xy,fundamental=fund,
                                  noise_level=float(rng.uniform(0.01,0.08)))
            y=AudioProcessor(self.cfg).pad_or_truncate(chs[0])
            m=AudioProcessor(self.cfg).mel(y); arr=np.stack([m,m,m],axis=0)
            np.save(str(out/f"synth_det_{i:06d}.npy"),arr)


class MelCachedDataset(Dataset):
    def __init__(self, cache_root, split, augment=False):
        self.augment=augment; self.files=[]; self.labels=[]
        for idx,cls in enumerate(["non_drone","drone"]):
            d=cache_root/split/cls
            if d.exists():
                for f in d.glob("*.npy"): self.files.append(f); self.labels.append(idx)
        if not self.files: raise RuntimeError(f"No cached mels in {cache_root}/{split}")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        arr=torch.tensor(np.load(str(self.files[idx])),dtype=torch.float32)
        if self.augment: arr=self._spec_augment(arr)
        return arr, torch.tensor(self.labels[idx],dtype=torch.long)

    @staticmethod
    def _spec_augment(arr):
        _,F,T=arr.shape
        if random.random()<0.7:
            f0=random.randint(0,F-1); fw=random.randint(1,max(1,F//8))
            arr[:,f0:f0+fw,:]=0.0
        if random.random()<0.7:
            t0=random.randint(0,T-1); tw=random.randint(1,max(1,T//8))
            arr[:,:,t0:t0+tw]=0.0
        return arr

# ══════════════════════════════════════════════════════════════════════════════
# CELL 3 (continued) – Dataset Managers & Helpers
# ══════════════════════════════════════════════════════════════════════════════

class DroneAudioDatasetManager:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.ap  = AudioProcessor(cfg)

    def prepare(self):
        dest = self.cfg.DRONEDS_RAW
        proc = self.cfg.PROCESSED_DIR / "detection"

        def _count_wavs(p: Path) -> int:
            return len(list(p.glob("*.wav"))) if p.exists() else 0

        counts = {k: _count_wavs(proc / s / c)
                  for s, c in [("train","drone"),("train","non_drone"),
                                ("val","drone"),("val","non_drone"),
                                ("test","drone"),("test","non_drone")]
                  for k in [f"{s}_{c.replace('non_','non')}"]}
        counts = {
            "train_drone":     _count_wavs(proc/"train"/"drone"),
            "train_non_drone": _count_wavs(proc/"train"/"non_drone"),
            "val_drone":       _count_wavs(proc/"val"/"drone"),
            "val_non_drone":   _count_wavs(proc/"val"/"non_drone"),
            "test_drone":      _count_wavs(proc/"test"/"drone"),
            "test_non_drone":  _count_wavs(proc/"test"/"non_drone"),
        }
        ready = (counts["train_drone"] > 20 and counts["train_non_drone"] > 20
                 and counts["val_drone"] > 0 and counts["val_non_drone"] > 0
                 and counts["test_drone"] > 0 and counts["test_non_drone"] > 0)
        if ready:
            total = sum(counts.values())
            print(f"✅ Detection dataset ready ({total} files)")
            return True
        print("⚠️ Detection dataset incomplete. Rebuilding …")
        if proc.exists(): shutil.rmtree(proc)
        self._download(dest)
        self._process(dest, proc)
        return True

    def _download(self, dest: Path):
        archive = dest / "drone_dataset.zip"
        dest.mkdir(parents=True, exist_ok=True)
        if not archive.exists():
            print("📥 Downloading DroneAudioDataset …")
            urllib.request.urlretrieve(self.cfg.DRONEDS_ZIP_URL, str(archive))
        binary_exists = any(d.is_dir() and d.name == "Binary_Drone_Audio" for d in dest.rglob("*"))
        if binary_exists: return
        print("📦 Extracting …")
        with zipfile.ZipFile(archive) as z: z.extractall(str(dest))

    def _process(self, src: Path, dst: Path):
        binary = next((d for d in src.rglob("Binary_Drone_Audio") if d.is_dir()), None)
        if binary is None:
            print("❌ Binary_Drone_Audio not found"); return
        mapping = {"yes_drone":"drone","unknown":"non_drone","Drone":"drone","noDrone":"non_drone"}
        all_files = {"drone": [], "non_drone": []}
        for cls_dir in binary.iterdir():
            if not cls_dir.is_dir(): continue
            label = mapping.get(cls_dir.name, "non_drone")
            all_files[label].extend([f for f in cls_dir.glob("*.*") if f.is_file()])
        for label, files in all_files.items():
            random.shuffle(files); n = len(files)
            if n == 0: continue
            splits = {"train": files[:int(n*0.70)], "val": files[int(n*0.70):int(n*0.85)], "test": files[int(n*0.85):]}
            for split, flist in splits.items():
                out = dst / split / label; out.mkdir(parents=True, exist_ok=True)
                for f in flist:
                    tgt = out / f"{f.stem}.wav"
                    if tgt.exists(): continue
                    try:
                        y, _ = librosa.load(str(f), sr=self.cfg.SR, mono=True)
                        sf.write(str(tgt), y, self.cfg.SR)
                    except Exception as e:
                        print(f"   ⚠️  {f.name}: {e}")
        print("✅ Detection dataset processed")


def report_detection_split_counts(cfg: Config):
    root = cfg.PROCESSED_DIR / "detection"
    print("\n📊 Detection WAV split counts")
    for split in ["train","val","test"]:
        for label in ["drone","non_drone"]:
            d = root / split / label
            n = len(list(d.glob("*.wav"))) if d.exists() else 0
            print(f"   {split:5s} / {label:10s}: {n}")


def generate_mixed_drone_training_audio(cfg: Config, force: bool = False):
    ap = AudioProcessor(cfg)
    train_drone_dir = cfg.PROCESSED_DIR / "detection" / "train" / "drone"
    val_drone_dir   = cfg.PROCESSED_DIR / "detection" / "val" / "drone"
    if not train_drone_dir.exists():
        print("⚠️ No train/drone directory found. Run dataset preparation first."); return
    existing = (list(train_drone_dir.glob(f"{cfg.MIX_CACHE_PREFIX}_*.wav")) +
                list(val_drone_dir.glob(f"{cfg.MIX_CACHE_PREFIX}_*.wav")))
    if len(existing) >= cfg.MIXED_DRONE_SAMPLES and not force:
        print(f"✅ Mixed drone audio already exists ({len(existing)} files) — skipping."); return
    if force:
        for d in [train_drone_dir, val_drone_dir]:
            if d.exists():
                for f in d.glob(f"{cfg.MIX_CACHE_PREFIX}_*.wav"):
                    try: f.unlink()
                    except: pass
    drone_files = [f for f in train_drone_dir.glob("*.wav") if not f.stem.startswith(cfg.MIX_CACHE_PREFIX)]
    if not drone_files:
        print("⚠️ No clean drone WAVs found for mixing."); return
    bg_pool = collect_background_pool(cfg)
    usable_bg_labels = [k for k in cfg.MIX_BACKGROUND_LABELS if len(bg_pool.get(k, [])) > 0]
    if not usable_bg_labels:
        print("⚠️ No usable background pool found. Skipping mixed-drone generation."); return
    n_total = int(cfg.MIXED_DRONE_SAMPLES)
    n_val   = int(n_total * cfg.MIXED_DRONE_VAL_FRAC)
    n_train = n_total - n_val
    print(f"🎛️ Generating mixed drone audio: total={n_total} train={n_train} val={n_val}")
    for i in tqdm(range(n_total), desc="Mixing drone+background"):
        split = "val" if i < n_val else "train"
        out_dir = val_drone_dir if split == "val" else train_drone_dir
        out_dir.mkdir(parents=True, exist_ok=True)
        drone_path = random.choice(drone_files)
        bg_label   = random.choice(usable_bg_labels)
        bg_path    = random.choice(bg_pool[bg_label])
        out_path   = out_dir / f"{cfg.MIX_CACHE_PREFIX}_{i:06d}_{bg_label}_{drone_path.stem}.wav"
        if out_path.exists() and not force: continue
        try:
            drone_y = ap.pad_or_truncate(load_audio_any(drone_path, cfg.SR))
            bg_y    = random_crop_or_loop(load_audio_any(bg_path, cfg.SR), len(drone_y))
            drone_y = np.clip(drone_y * db_to_gain(random.uniform(*cfg.MIX_GAIN_RANGE_DB)), -1.0, 1.0).astype(np.float32)
            bg_y    = np.clip(bg_y * db_to_gain(random.uniform(*cfg.MIX_BG_GAIN_RANGE_DB)), -1.0, 1.0).astype(np.float32)
            mixed   = mix_at_snr(drone_y, bg_y, random.uniform(*cfg.MIX_SNR_DB_RANGE))
            if random.random() < 0.35: mixed = np.roll(mixed, random.randint(0, len(mixed)//10)).astype(np.float32)
            if random.random() < 0.40:
                noise = np.random.randn(len(mixed)).astype(np.float32)
                noise /= (np.max(np.abs(noise)) + 1e-8)
                mixed = normalize_peak(mixed + 0.01 * noise)
            sf.write(str(out_path), mixed, cfg.SR)
        except Exception as e:
            print(f"   ⚠️ Failed mix {i}: {e}")
    made = (len(list(train_drone_dir.glob(f"{cfg.MIX_CACHE_PREFIX}_*.wav"))) +
            len(list(val_drone_dir.glob(f"{cfg.MIX_CACHE_PREFIX}_*.wav"))))
    print(f"✅ Mixed drone generation complete ({made} files).")


def inject_synthetic_det_data(cfg: Config, n_samples: int = None, force: bool = False):
    n_samples = n_samples or cfg.SYNTHETIC_DET_SAMPLES
    cache_dir = cfg.MEL_CACHE_DIR / "train" / "drone"
    cache_dir.mkdir(parents=True, exist_ok=True)
    existing = len(list(cache_dir.glob("synth_det_*.npy")))
    if existing >= n_samples and not force:
        print(f"✅ Synthetic detection cache already present ({existing} files) — skipping."); return
    ap  = AudioProcessor(cfg)
    rng = np.random.default_rng(cfg.SEED + 1)
    r      = rng.uniform(0.3, cfg.MAX_LOCALIZATION_DIST, n_samples)
    theta  = rng.uniform(0, 2*np.pi, n_samples)
    funds  = rng.choice([80,90,100,110,120,130], n_samples)
    noises = rng.uniform(0.02, 0.10, n_samples)
    cx, cy = cfg.ARRAY_CENTER
    positions = np.stack([cx + r*np.cos(theta), cy + r*np.sin(theta)], axis=1)
    print(f"🔬 Injecting {n_samples} synthetic drone mels into detection cache …")
    for i in tqdm(range(n_samples)):
        out_path = cache_dir / f"synth_det_{i:06d}.npy"
        if out_path.exists(): continue
        chs = synthesise_drone(cfg.MIC_POSITIONS, positions[i], fundamental=int(funds[i]), noise_level=float(noises[i]))
        mono = np.mean(np.stack(chs, axis=0), axis=0)
        mel  = ap.mel(ap.pad_or_truncate(mono))
        np.save(str(out_path), np.stack([mel, mel, mel], axis=0))
    print("✅ Synthetic injection done.")


def get_det_dataloaders(cfg: Config):
    cache = cfg.MEL_CACHE_DIR
    tr_ds = MelCachedDataset(cache, "train", augment=True)
    va_ds = MelCachedDataset(cache, "val",   augment=False)
    try:
        te_ds = MelCachedDataset(cache, "test", augment=False)
    except RuntimeError:
        print("⚠️ No mel cache test split — using val as fallback.")
        te_ds = MelCachedDataset(cache, "val", augment=False)
    labels  = np.array(tr_ds.labels)
    counts  = np.bincount(labels); counts[counts == 0] = 1
    weights = (1.0 / counts)[labels]
    sampler = WeightedRandomSampler(weights, len(weights), replacement=True)
    def _col(batch):
        xs, ys = zip(*batch); return torch.stack(xs), torch.stack(ys)
    bs = cfg.BATCH_SIZE; nw = min(4, os.cpu_count() or 2); pin = (cfg.DEVICE == "cuda")
    kw = dict(collate_fn=_col, num_workers=nw, pin_memory=pin,
              persistent_workers=(nw>0), prefetch_factor=2 if nw>0 else None)
    tr_l = DataLoader(tr_ds, batch_size=bs, sampler=sampler, **kw)
    va_l = DataLoader(va_ds, batch_size=bs, shuffle=False, **kw)
    te_l = DataLoader(te_ds, batch_size=bs, shuffle=False, **kw)
    return tr_l, va_l, te_l


# ── label helpers ─────────────────────────────────────────────────────────────

_AZ_KEYS   = ["azimuth_deg","azimuth","az","Azimuth","AZ","bearing","heading","direction_deg","direction"]
_DIST_KEYS = ["distance_m","distance","dist","Distance","range","range_m","horizontal_distance","slant_range"]
_HT_KEYS   = ["height_m","height","alt","altitude","Height","z","elevation","Elevation","altitude_m","z_m","height_agl"]

def _get_scalar(d: dict, keys: list):
    for k in keys:
        if k in d:
            v = d[k]
            if isinstance(v, (int, float)) and not math.isnan(float(v)): return float(v)
            if isinstance(v, list) and len(v) > 0:
                arr = [float(x) for x in v if x is not None and not math.isnan(float(x))]
                return float(np.median(arr)) if arr else None
    return None

def _cartesian_to_az_dist_ht(obj: dict):
    def _gv(d, keys):
        for k in keys:
            if k in d:
                v = d[k]
                if isinstance(v, (int, float)) and not math.isnan(float(v)): return float(v)
                if isinstance(v, list) and v:
                    vals = [float(x) for x in v if x is not None and not math.isnan(float(x))]
                    return float(np.median(vals)) if vals else None
        return None
    x = _gv(obj, ["x","X","east","East","east_m","pos_x","x_m"])
    y = _gv(obj, ["y","Y","north","North","north_m","pos_y","y_m"])
    z = _gv(obj, ["z","Z","up","Up","up_m","pos_z","z_m","height","height_m","altitude","alt"])
    if x is not None and y is not None and z is not None:
        return wrap_angle_deg(float(math.degrees(math.atan2(y, x)))), float(math.sqrt(x**2+y**2)), float(abs(z))
    return None

def parse_label_json(raw: bytes):
    try: data = json.loads(raw.decode("utf-8"))
    except Exception: return None
    if not isinstance(data, dict): return None
    if "drone" in data and isinstance(data["drone"], dict):
        drone = data["drone"]
        sound_source = drone.get("sound_source", "")
        if isinstance(sound_source, str) and "ambient" in sound_source.lower(): return None
        try:
            az = drone.get("azimuth"); di = drone.get("distance"); ht = drone.get("height")
            if az is not None and di is not None and ht is not None:
                az, di, ht = float(az), float(di), float(ht)
                if not (math.isnan(az) or math.isnan(di) or math.isnan(ht)): return az, di, ht
        except (TypeError, ValueError): pass
        result = _cartesian_to_az_dist_ht(drone)
        if result: return result
    az = _get_scalar(data, _AZ_KEYS); di = _get_scalar(data, _DIST_KEYS); ht = _get_scalar(data, _HT_KEYS)
    if az is not None and di is not None and ht is not None: return float(az), float(di), float(ht)
    for sk in ["uav","target","labels","annotation","data","position"]:
        if sk not in data: continue
        sub = data[sk]
        if isinstance(sub, dict):
            r = _cartesian_to_az_dist_ht(sub)
            if r: return r
    return None

def _probe_label_schema(raw: bytes) -> str:
    try: data = json.loads(raw.decode())
    except Exception: return "invalid JSON"
    if isinstance(data, list):
        sample = data[0] if data else {}
        return f"list[{len(data)}] of dict | sample keys={list(sample.keys())[:8] if isinstance(sample,dict) else '?'}"
    if isinstance(data, dict): return f"dict | top-level keys={list(data.keys())[:8]}"
    return f"unknown ({type(data).__name__})"


class UaVirBASEDatasetManager:
    AUDIO_FILENAME = "output.wav"
    LABEL_FILENAME = "label.json"

    def __init__(self, cfg: Config):
        self.cfg = cfg; self.ap = AudioProcessor(cfg)

    def prepare(self):
        proc = self.cfg.PROCESSED_DIR / "localization"
        if (proc/"train").exists():
            n = len(list(proc.rglob("*_label.json")))
            if n > 20:
                print(f"✅ Localization dataset ready ({n} sessions)"); return True
        url = self.cfg.UAVIRBASE_ZIP_URL
        if url is None:
            print("⚠️  UAVIRBASE_ZIP_URL is None → using synthetic fallback.")
            self._write_synthetic(proc); return True
        if getattr(self.cfg, "UAVIRBASE_FULL", False):
            print("📥 FULL download mode …")
            try:
                self._download_full(self.cfg.UAVIRBASE_RAW)
                self._process_local(self.cfg.UAVIRBASE_RAW, proc)
            except Exception as e:
                print(f"⚠️  Full download failed ({e}) → synthetic fallback.")
                self._write_synthetic(proc)
        else:
            n_sess = getattr(self.cfg, "UAVIRBASE_N_SESSIONS", 300)
            print(f"📥 PARTIAL download: {n_sess} sessions via remotezip …")
            try:
                _ensure_remotezip(); self._download_partial(url, proc, n_sess)
            except Exception as e:
                print(f"⚠️  Partial download failed ({e}) → synthetic fallback.")
                self._write_synthetic(proc)
        return True

    def _download_partial(self, url: str, proc: Path, n_sessions: int):
        from remotezip import RemoteZip
        RZ_KWARGS = {"initial_buffer_size": 64*1024*1024}
        AUDIO_CANDIDATES = {"output.wav","audio.wav"}
        LABEL_CANDIDATES = {"label.json","annotation.json"}
        print("   Reading remote ZIP central directory …")
        with RemoteZip(url, **RZ_KWARGS) as rz: all_names = rz.namelist()
        norm_names = [n.replace("\\","/") for n in all_names]
        session_map = {}
        for path in norm_names:
            p = Path(path)
            if len(p.parts) < 2: continue
            session_dir = str(Path(*p.parts[:-1]))
            session_map.setdefault(session_dir, []).append(p.name)
        paired = []
        for session_dir, files in session_map.items():
            audio_name = next((f for f in files if f in AUDIO_CANDIDATES), None)
            label_name = next((f for f in files if f in LABEL_CANDIDATES), None)
            if audio_name and label_name:
                paired.append((f"{session_dir}/{audio_name}", f"{session_dir}/{label_name}"))
        print(f"   Candidate paired sessions: {len(paired)}")
        if len(paired) == 0: raise RuntimeError("No paired audio/json sessions found.")
        print("   Validating labels …")
        usable = []; ambient_skipped = 0; parse_failed = 0
        with RemoteZip(url, **RZ_KWARGS) as rz:
            for audio_path, label_path in tqdm(paired, desc="Validating sessions"):
                try:
                    raw = rz.read(label_path); parsed = parse_label_json(raw)
                    if parsed is None:
                        try:
                            content = json.loads(raw.decode("utf-8"))
                            source = content.get("drone", {}).get("sound_source", "")
                            if isinstance(source, str) and "ambient" in source.lower():
                                ambient_skipped += 1; continue
                        except: pass
                        parse_failed += 1; continue
                    usable.append((audio_path, label_path, parsed))
                except Exception: parse_failed += 1
        print(f"   Usable sessions: {len(usable)} | Ambient skipped: {ambient_skipped} | Failed: {parse_failed}")
        if len(usable) == 0: raise RuntimeError("No usable sessions after validation.")
        random.shuffle(usable); usable = usable[:min(n_sessions, len(usable))]
        n = len(usable); idx_tr = int(n*0.70); idx_val = int(n*0.85)
        split_map = ([(u,"train") for u in usable[:idx_tr]] +
                     [(u,"val") for u in usable[idx_tr:idx_val]] +
                     [(u,"test") for u in usable[idx_val:]])
        for s in ["train","val","test"]: (proc/s).mkdir(parents=True, exist_ok=True)
        downloaded = 0; failed_audio = 0
        with RemoteZip(url, **RZ_KWARGS) as rz:
            for (audio_path, label_path, parsed), split in tqdm(split_map, desc="Downloading sessions"):
                session_id = Path(audio_path).parent.name
                try:
                    az, di, ht = parsed
                    audio_bytes = rz.read(audio_path)
                    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tf:
                        tf.write(audio_bytes); tmp_path = tf.name
                    try:
                        channels = self.ap.load_channels(tmp_path, channel_indices=self.cfg.UAVIRBASE_MIC_INDICES)
                    finally:
                        if os.path.exists(tmp_path): os.unlink(tmp_path)
                    out_dir = proc / split
                    for i, ch in enumerate(channels):
                        sf.write(str(out_dir/f"{session_id}_ch{i}.wav"), self.ap.pad_or_truncate(ch), self.cfg.SR)
                    (out_dir/f"{session_id}_label.json").write_text(json.dumps({"azimuth_deg":az,"distance_m":di,"height_m":ht}))
                    downloaded += 1
                except Exception as e:
                    failed_audio += 1; print(f"   ⚠️  {session_id}: {e}")
        print(f"✅ Downloaded {downloaded} sessions.")
        for split in ["train","val","test"]:
            print(f"   {split}: {len(list((proc/split).glob('*_label.json')))} sessions")
        if downloaded == 0: raise RuntimeError("Zero sessions saved after filtering.")

    def _download_full(self, dest: Path):
        archive = dest / "uavirbase.zip"
        if not archive.exists(): urllib.request.urlretrieve(self.cfg.UAVIRBASE_ZIP_URL, str(archive))
        with zipfile.ZipFile(archive) as z: z.extractall(str(dest))
        archive.unlink()

    def _process_local(self, src: Path, dst: Path):
        AUDIO = self.AUDIO_FILENAME; LABEL = self.LABEL_FILENAME
        sessions = [p.parent for p in src.rglob(LABEL)] or [p.parent for p in src.rglob("annotation.json")]
        if not sessions: print("⚠️  No label files found after extraction."); return
        random.shuffle(sessions); n = len(sessions)
        splits = {"train": sessions[:int(n*.70)], "val": sessions[int(n*.70):int(n*.85)], "test": sessions[int(n*.85):]}
        print(f"🎵 Processing {n} UaVirBASE sessions …")
        for split, sess_list in splits.items():
            out = dst/split; out.mkdir(parents=True, exist_ok=True); failed = 0
            for sess in tqdm(sess_list, desc=split):
                audio_file = next((sess/x for x in [AUDIO,"audio.wav"] if (sess/x).exists()), None)
                if audio_file is None:
                    wavs = list(sess.glob("*.wav"))
                    if not wavs: continue
                    audio_file = wavs[0]
                label_file = next((sess/x for x in [LABEL,"annotation.json"] if (sess/x).exists()), None)
                if label_file is None: continue
                try:
                    parsed = parse_label_json(label_file.read_bytes())
                    if parsed is None: failed += 1; continue
                    az, di, ht = parsed
                    channels = self.ap.load_channels(audio_file, channel_indices=self.cfg.UAVIRBASE_MIC_INDICES)
                    sid = sess.name
                    for i, ch in enumerate(channels):
                        sf.write(str(out/f"{sid}_ch{i}.wav"), self.ap.pad_or_truncate(ch), self.cfg.SR)
                    (out/f"{sid}_label.json").write_text(json.dumps({"azimuth_deg":az,"distance_m":di,"height_m":ht}))
                except Exception as e:
                    print(f"   ⚠️  {sess.name}: {e}"); failed += 1
            if failed: print(f"   ⚠️  {failed} sessions skipped in '{split}'")
        print("✅ Full dataset processed")

    def _write_synthetic(self, proc: Path, n_total: int = 2400):
        print(f"🔬 Generating {n_total} synthetic localization samples …")
        rng    = np.random.default_rng(self.cfg.SEED)
        dists  = np.exp(rng.uniform(np.log(0.3), np.log(self.cfg.MAX_LOCALIZATION_DIST), n_total))
        az_rads= rng.uniform(-np.pi, np.pi, n_total)
        heights= rng.uniform(0.5, 5.0, n_total)
        funds  = rng.choice([80,90,100,110,120,130], n_total)
        noises = rng.uniform(0.01, 0.08, n_total)
        cx, cy = self.cfg.ARRAY_CENTER
        xs = cx + dists*np.cos(az_rads); ys = cy + dists*np.sin(az_rads)
        idx_tr = int(n_total*0.70); idx_val = int(n_total*0.85)
        splits = ["train"]*idx_tr + ["val"]*(idx_val-idx_tr) + ["test"]*(n_total-idx_val)
        for s in ["train","val","test"]: (proc/s).mkdir(parents=True, exist_ok=True)
        for i in tqdm(range(n_total), desc="Synthetic loc"):
            sid = f"synth_{i:06d}"
            chs = synthesise_drone(self.cfg.MIC_POSITIONS, [xs[i],ys[i]], fundamental=int(funds[i]), noise_level=float(noises[i]))
            out = proc/splits[i]
            for j, ch in enumerate(chs):
                sf.write(str(out/f"{sid}_ch{j}.wav"), AudioProcessor(self.cfg).pad_or_truncate(ch), self.cfg.SR)
            (out/f"{sid}_label.json").write_text(json.dumps({"azimuth_deg":float(np.degrees(az_rads[i])),"distance_m":float(dists[i]),"height_m":float(heights[i])}))
        print(f"✅ Synthetic fallback: {len(list(proc.rglob('*_label.json')))} sessions written.")

# ══════════════════════════════════════════════════════════════════════════════
# CELL 4 – PyTorch Datasets
# ══════════════════════════════════════════════════════════════════════════════

class DetectionDataset(Dataset):
    def __init__(self, root: Path, split: str, augment=False, cfg: Config = None):
        self.ap = AudioProcessor(cfg or config); self.cfg = cfg or config
        self.augment = augment; self.files = []; self.labels = []
        for idx, cls in enumerate(["non_drone","drone"]):
            d = root/split/cls
            if d.exists():
                for f in d.glob("*.wav"): self.files.append(f); self.labels.append(idx)
        if not self.files: raise RuntimeError(f"No files in {root}/{split}")

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        y = self.ap.pad_or_truncate(self.ap.load(self.files[idx]))
        if self.augment:
            y = y * (10 ** (random.uniform(-12, 12) / 20))
            y = np.clip(y, -1, 1).astype(np.float32)
            if random.random() < 0.6: y = self.ap.add_noise(y, random.uniform(3, 25))
            if random.random() < 0.5: y = np.roll(y, random.randint(0, len(y)//4)).astype(np.float32)
            if random.random() < 0.4:
                tilt = random.uniform(-0.3, 0.3)
                freqs = np.fft.rfftfreq(len(y)); freqs[0] = 1e-6
                mag = 10 ** (tilt * np.log2(np.maximum(freqs, 1e-6)))
                y = np.fft.irfft(np.fft.rfft(y)*mag, n=len(y)).astype(np.float32)
                y = np.clip(y, -1, 1).astype(np.float32)
        m = self.ap.mel(y)
        return torch.tensor(np.stack([m,m,m],axis=0), dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)


class LocalizationDataset(Dataset):
    def __init__(self, root: Path, split: str, augment=False, cfg: Config = None):
        self.cfg = cfg or config; self.ap = AudioProcessor(self.cfg)
        self.augment = augment; self.sessions = []
        d = root/split
        if not d.exists(): raise RuntimeError(f"Localization split not found: {d}")
        for lf in d.glob("*_label.json"):
            sid = lf.stem.replace("_label","")
            chs = [d/f"{sid}_ch{i}.wav" for i in range(3)]
            if all(c.exists() for c in chs): self.sessions.append((chs, lf))
        if not self.sessions: raise RuntimeError(f"No complete sessions in {d}")

    def __len__(self): return len(self.sessions)

    def __getitem__(self, idx):
        chs_paths, lf = self.sessions[idx]
        channels = [self.ap.pad_or_truncate(self.ap.load(p)) for p in chs_paths]
        ipd_cache = lf.parent / (lf.stem.replace("_label","") + "_ipd.npy")
        if not self.augment and ipd_cache.exists():
            ipd = np.load(str(ipd_cache))
        else:
            if self.augment and random.random() < 0.35:
                snr = random.uniform(5, 20)
                channels = [self.ap.add_noise(c, snr) for c in channels]
            ipd = compute_ipd_features(channels, self.cfg)
            if not self.augment:
                try: np.save(str(ipd_cache), ipd)
                except: pass
        mels  = [self.ap.mel(c) for c in channels]
        mel_t = torch.tensor(np.stack(mels, axis=0), dtype=torch.float32)
        ipd_t = torch.tensor(ipd, dtype=torch.float32)
        label = json.loads(lf.read_text())
        az_deg = float(label["azimuth_deg"]); di_m = float(label["distance_m"]); ht_m = float(label["height_m"])
        if self.augment:
            az_deg = wrap_angle_deg(az_deg + random.gauss(0, 15.0))
            di_m = max(0.5, di_m + random.gauss(0, 1.5))
            ht_m = max(0.5, ht_m + random.gauss(0, 1.0))
        az_rad = math.radians(az_deg)
        max_dist = self.cfg.MAX_LOCALIZATION_DIST
        lbl_t = torch.tensor([math.sin(az_rad), math.cos(az_rad),
                               np.clip(di_m/max_dist, 0, 1.5), np.clip(ht_m/max_dist, 0, 1.5)], dtype=torch.float32)
        return mel_t, ipd_t, lbl_t


class SyntheticLocDataset(Dataset):
    def __init__(self, cfg: Config, n_samples=500, augment=True):
        self.cfg = cfg; self.ap = AudioProcessor(cfg); self.n = n_samples; self.augment = augment
        rng = np.random.default_rng(cfg.SEED)
        r = rng.uniform(0.3, cfg.MAX_LOCALIZATION_DIST, n_samples)
        theta = rng.uniform(0, 2*np.pi, n_samples)
        height = rng.uniform(0.5, 5.0, n_samples)
        cx, cy = cfg.ARRAY_CENTER
        self.positions  = np.stack([cx + r*np.cos(theta), cy + r*np.sin(theta), height], axis=1)
        self.fundamentals = rng.choice([80,90,100,110,120,130], n_samples)

    def __len__(self): return self.n

    def __getitem__(self, idx):
        pos = self.positions[idx]; fund = int(self.fundamentals[idx])
        chs = synthesise_drone(self.cfg.MIC_POSITIONS, pos[:2], fundamental=fund,
                               noise_level=0.04 if self.augment else 0.01)
        chs = [self.ap.pad_or_truncate(c) for c in chs]
        if self.augment and random.random() < 0.3:
            snr = random.uniform(5, 20); chs = [self.ap.add_noise(c, snr) for c in chs]
        mels  = [self.ap.mel(c) for c in chs]
        mel_t = torch.tensor(np.stack(mels, axis=0), dtype=torch.float32)
        ipd_t = torch.tensor(compute_ipd_features(chs, self.cfg), dtype=torch.float32)
        az_deg = xy_to_azimuth_deg(pos[:2], self.cfg.ARRAY_CENTER)
        az_rad = math.radians(az_deg)
        cx, cy = self.cfg.ARRAY_CENTER
        dist_m = math.sqrt((pos[0]-cx)**2 + (pos[1]-cy)**2)
        max_dist = self.cfg.MAX_LOCALIZATION_DIST
        lbl_t = torch.tensor([math.sin(az_rad), math.cos(az_rad),
                               np.clip(dist_m/max_dist, 0, 1.5), np.clip(pos[2]/max_dist, 0, 1.5)], dtype=torch.float32)
        return mel_t, ipd_t, lbl_t

# ══════════════════════════════════════════════════════════════════════════════
# CELL 5 – Models (DetectionCNN + LocalizationCNNLite + LocalizationCNN)
# ══════════════════════════════════════════════════════════════════════════════

class DetectionCNN(nn.Module):
    """Frequency-axis attention pooling for domain-robust detection."""
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            self._block(3,32), self._block(32,64), self._block(64,128), self._block(128,256))
        self.freq_attn = nn.Sequential(
            nn.Conv2d(256,64,kernel_size=(1,1)), nn.ReLU(),
            nn.Conv2d(64,1,kernel_size=(1,1)),  nn.Softmax(dim=2))
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.Linear(256,256), nn.ReLU(), nn.Dropout(0.4), nn.Linear(256,2))

    @staticmethod
    def _block(cin, cout):
        return nn.Sequential(nn.Conv2d(cin,cout,3,padding=1), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))

    def forward(self, x):
        feat = self.encoder(x)
        attn = self.freq_attn(feat)
        feat = (feat * attn).sum(dim=2, keepdim=True)
        feat = self.gap(feat).flatten(1)
        return self.classifier(feat)


class LocalizationCNNLite(nn.Module):
    """
    v13 Enhancement ②: MobileNet-style depthwise-separable localization model.
    ~4× fewer parameters than LocalizationCNN — ideal for resource-constrained
    GPU environments (< 4 GB VRAM) and Colab free tier.
    """
    def __init__(self, n_mels: int = 64):
        super().__init__()
        def _ds_block(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cin, 3, padding=1, groups=cin, bias=False),
                nn.Conv2d(cin, cout, 1, bias=False),
                nn.BatchNorm2d(cout), nn.ReLU(),
                nn.MaxPool2d(2))
        self.mel_enc = nn.Sequential(
            _ds_block(3,   16),
            _ds_block(16,  32),
            _ds_block(32,  64),
            _ds_block(64,  128),
            nn.AdaptiveAvgPool2d((2, 2)),
        )
        self.ipd_fc = nn.Sequential(
            nn.Linear(3, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
        )
        fused = 128 * 2 * 2 + 16
        self.head = nn.Sequential(
            nn.Linear(fused, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 4),
        )

    def forward(self, mel, ipd):
        mel_feat = self.mel_enc(mel).flatten(1)
        ipd_feat = self.ipd_fc(ipd)
        return self.head(torch.cat([mel_feat, ipd_feat], dim=1))


class LocalizationCNN(nn.Module):
    """Full-capacity localization model for high-VRAM environments."""
    def __init__(self, n_mels: int = 64):
        super().__init__()
        self.mel_enc = nn.Sequential(
            self._block(3,32), self._block(32,64), self._block(64,128), self._block(128,256),
            nn.AdaptiveAvgPool2d((4,4)))
        self.ipd_fc = nn.Sequential(nn.Linear(3,32), nn.ReLU(), nn.Linear(32,32), nn.ReLU())
        fused = 256*4*4 + 32
        self.head = nn.Sequential(
            nn.Linear(fused,512), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(512,128),   nn.ReLU(), nn.Linear(128,4))

    @staticmethod
    def _block(cin, cout):
        return nn.Sequential(nn.Conv2d(cin,cout,3,padding=1), nn.BatchNorm2d(cout), nn.ReLU(), nn.MaxPool2d(2))

    def forward(self, mel, ipd):
        mel_feat = self.mel_enc(mel).flatten(1)
        ipd_feat = self.ipd_fc(ipd)
        return self.head(torch.cat([mel_feat, ipd_feat], dim=1))


def make_localization_model(cfg: Config):
    """
    v13: Factory that selects Lite vs Full model based on available GPU VRAM.
    Use cfg.USE_LITE_LOC = True/False to override the auto-selection.
    """
    use_lite = getattr(cfg, "USE_LITE_LOC", False)
    if use_lite:
        print("🔧 Using LocalizationCNNLite (resource-constrained mode)")
        return LocalizationCNNLite(cfg.N_MELS)
    else:
        print("🔧 Using LocalizationCNN (full-capacity mode)")
        return LocalizationCNN(cfg.N_MELS)


def localization_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    loss_sin  = F.mse_loss(pred[:,0], target[:,0])
    loss_cos  = F.mse_loss(pred[:,1], target[:,1])
    loss_dist = F.smooth_l1_loss(pred[:,2], target[:,2])
    loss_ht   = F.smooth_l1_loss(pred[:,3], target[:,3])
    return 2.0*(loss_sin+loss_cos) + 1.0*loss_dist + 0.7*loss_ht

# ══════════════════════════════════════════════════════════════════════════════
# CELL 6 – Training Managers
# ══════════════════════════════════════════════════════════════════════════════


class TrainingLogger:
    def __init__(self, path: Path, columns: list):
        self.path = path; self.columns = columns; self.rows = []
        try:
            path.parent.mkdir(parents=True, exist_ok=True)
        except OSError as e:
            raise RuntimeError(f"Cannot create log directory {path.parent}: {e}")
        test_file = path.parent / ".write_test"
        try: test_file.write_text("ok"); test_file.unlink()
        except OSError as e: raise RuntimeError(f"Log directory {path.parent} not writable: {e}")
        if path.exists():
            import csv
            with open(path, newline="") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    parsed = {}
                    for k, v in row.items():
                        try: parsed[k] = int(v) if v.lstrip("-").isdigit() else float(v)
                        except: parsed[k] = v
                    self.rows.append(parsed)
            print(f"   📋 Resumed log from {path.name} ({len(self.rows)} existing rows)")
        else:
            print(f"   📋 New log: {path}")

    def log(self, **kwargs):
        self.rows.append(kwargs); self._flush()

    def _flush(self):
        import csv
        tmp = self.path.parent / (self.path.name + ".tmp")
        with open(tmp, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.columns)
            writer.writeheader(); writer.writerows(self.rows)
        tmp.replace(self.path)

    def to_lists(self) -> dict:
        out = {c: [] for c in self.columns}
        for row in self.rows:
            for c in self.columns: out[c].append(row.get(c, float("nan")))
        return out


# ── v13 Enhanced visualization ─────────────────────────────────────────────

PLOT_STYLE = {
    "bg":      "#0f172a",
    "panel":   "#1e293b",
    "accent":  "#00d4ff",
    "warn":    "#f59e0b",
    "ok":      "#22c55e",
    "err":     "#ef4444",
    "grid":    "#334155",
    "text":    "#e2e8f0",
    "muted":   "#64748b",
    "purple":  "#8b5cf6",
    "plasma":  "plasma",
}

def _apply_dark_style(fig, axes_flat):
    fig.patch.set_facecolor(PLOT_STYLE["bg"])
    for ax in axes_flat:
        ax.set_facecolor(PLOT_STYLE["panel"])
        ax.tick_params(colors=PLOT_STYLE["text"])
        ax.xaxis.label.set_color(PLOT_STYLE["text"])
        ax.yaxis.label.set_color(PLOT_STYLE["text"])
        ax.title.set_color(PLOT_STYLE["accent"])
        for spine in ax.spines.values(): spine.set_color(PLOT_STYLE["grid"])
        ax.grid(color=PLOT_STYLE["grid"], alpha=0.4)


def plot_training_logs(cfg: Config = None, save: bool = True):
    """v13: Dark-themed training curves for detection and localization."""
    cfg = cfg or config
    det_csv = cfg.DRIVE_LOGS / "detection_log.csv"
    loc_csv = cfg.DRIVE_LOGS / "localization_log.csv"
    has_det = det_csv.exists(); has_loc = loc_csv.exists()
    if not has_det and not has_loc:
        print("❌ No training logs found in", cfg.DRIVE_LOGS); return

    n_plots = (2 if has_det else 0) + (2 if has_loc else 0)
    fig, axes = plt.subplots(1, n_plots, figsize=(6*n_plots, 5))
    if n_plots == 1: axes = [axes]
    _apply_dark_style(fig, list(axes))
    ax_idx = 0
    import csv

    if has_det:
        with open(det_csv, newline="") as f: rows = list(csv.DictReader(f))
        epochs  = [int(r["epoch"]) for r in rows]
        tr_loss = [float(r["tr_loss"]) for r in rows]
        tr_acc  = [float(r["tr_acc"])  for r in rows]
        val_acc = [float(r["val_acc"]) for r in rows]
        ax = axes[ax_idx]; ax_idx += 1
        ax.plot(epochs, tr_loss, "-o", color=PLOT_STYLE["accent"], ms=4, label="Train loss")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Cross-entropy loss")
        ax.set_title("Detection — Loss"); ax.legend(facecolor=PLOT_STYLE["panel"])
        ax = axes[ax_idx]; ax_idx += 1
        ax.plot(epochs, tr_acc,  "-o", color=PLOT_STYLE["ok"],   ms=4, label="Train acc %")
        ax.plot(epochs, val_acc, "-s", color=PLOT_STYLE["warn"], ms=4, label="Val acc %")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy (%)")
        ax.set_title("Detection — Accuracy"); ax.legend(facecolor=PLOT_STYLE["panel"])

    if has_loc:
        with open(loc_csv, newline="") as f: rows = list(csv.DictReader(f))
        epochs   = [int(r["epoch"])      for r in rows]
        tr_loss  = [float(r["tr_loss"])  for r in rows]
        val_loss = [float(r["val_loss"]) for r in rows]
        mae_az   = [float(r["mae_az"])   for r in rows]
        mae_dist = [float(r["mae_dist"]) for r in rows]
        mae_ht   = [float(r["mae_ht"])   for r in rows]
        ax = axes[ax_idx]; ax_idx += 1
        ax.plot(epochs, tr_loss,  "-o", color=PLOT_STYLE["accent"], ms=4, label="Train loss")
        ax.plot(epochs, val_loss, "-s", color=PLOT_STYLE["warn"],   ms=4, label="Val loss")
        ax.set_xlabel("Epoch"); ax.set_ylabel("MSE loss")
        ax.set_title("Localization — Loss"); ax.legend(facecolor=PLOT_STYLE["panel"])
        ax = axes[ax_idx]; ax_idx += 1
        ax.plot(epochs, mae_az,   "-o", color=PLOT_STYLE["err"],    ms=4, label="MAE az (°)")
        ax.plot(epochs, mae_dist, "-s", color=PLOT_STYLE["purple"], ms=4, label="MAE dist (m)")
        ax.plot(epochs, mae_ht,   "-^", color=PLOT_STYLE["ok"],     ms=4, label="MAE ht (m)")
        ax.set_xlabel("Epoch"); ax.set_ylabel("MAE")
        ax.set_title("Localization — MAE"); ax.legend(facecolor=PLOT_STYLE["panel"])

    plt.tight_layout()
    if save:
        out = cfg.DRIVE_LOGS / "training_curves.png"
        plt.savefig(str(out), dpi=150, bbox_inches="tight")
        print(f"💾 Plot saved: {out}")
    plt.show()


def plot_confusion_matrix_styled(cm: np.ndarray, labels: list, title: str = "Confusion Matrix",
                                  save_path: Path = None):
    """v13: Dark-themed confusion matrix with counts + percentages."""
    fig, ax = plt.subplots(figsize=(6, 5))
    _apply_dark_style(fig, [ax])
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("drone_cm", [PLOT_STYLE["panel"], PLOT_STYLE["accent"]])
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap)
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, color=PLOT_STYLE["text"])
    ax.set_yticklabels(labels, color=PLOT_STYLE["text"])
    total = cm.sum(axis=1, keepdims=True) + 1e-8
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            pct = 100.0 * cm[i,j] / total[i,0]
            ax.text(j, i, f"{cm[i,j]}\n({pct:.1f}%)", ha="center", va="center",
                    color=PLOT_STYLE["text"], fontsize=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(title, color=PLOT_STYLE["accent"])
    plt.tight_layout()
    if save_path:
        plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
        print(f"💾 Confusion matrix saved: {save_path}")
    plt.show()


def plot_localization_scatter(cfg: Config = None, n_samples: int = 200):
    """v13: 3-panel scatter: az scatter, dist scatter, az error histogram."""
    cfg = cfg or config
    try:
        model = load_localization_model(cfg)
    except FileNotFoundError:
        print("❌ No localization model — run train_localization() first."); return
    ds  = SyntheticLocDataset(cfg, n_samples=n_samples, augment=False)
    dev = torch.device(cfg.DEVICE)
    preds_az = []; true_az = []; preds_dist = []; true_dist = []
    model.eval()
    for i in range(len(ds)):
        mel_t, ipd_t, lbl_t = ds[i]
        with torch.no_grad():
            pred = model(mel_t.unsqueeze(0).to(dev), ipd_t.unsqueeze(0).to(dev))[0].cpu().numpy()
        preds_az.append(wrap_angle_deg(float(np.degrees(np.arctan2(pred[0], pred[1])))))
        true_az.append(wrap_angle_deg(float(np.degrees(np.arctan2(lbl_t[0].item(), lbl_t[1].item())))))
        preds_dist.append(float(abs(pred[2])*cfg.MAX_LOCALIZATION_DIST))
        true_dist.append(float(lbl_t[2].item()*cfg.MAX_LOCALIZATION_DIST))
    az_errs = angular_error_deg(np.array(preds_az), np.array(true_az))
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    _apply_dark_style(fig, axes)
    axes[0].scatter(true_az, preds_az, alpha=0.5, c=PLOT_STYLE["accent"], s=15)
    lim = (-180, 180); axes[0].plot(lim, lim, "--", color=PLOT_STYLE["warn"], lw=1)
    axes[0].set_xlabel("True Azimuth (°)"); axes[0].set_ylabel("Predicted Azimuth (°)")
    axes[0].set_title("Azimuth Scatter")
    axes[1].scatter(true_dist, preds_dist, alpha=0.5, c=PLOT_STYLE["ok"], s=15)
    m = max(max(true_dist), max(preds_dist)); axes[1].plot([0,m],[0,m],"--",color=PLOT_STYLE["warn"],lw=1)
    axes[1].set_xlabel("True Distance (m)"); axes[1].set_ylabel("Predicted Distance (m)")
    axes[1].set_title("Distance Scatter")
    axes[2].hist(az_errs, bins=30, color=PLOT_STYLE["purple"], edgecolor=PLOT_STYLE["bg"])
    axes[2].axvline(float(np.median(az_errs)), color=PLOT_STYLE["warn"], lw=2, label=f"Median={np.median(az_errs):.1f}°")
    axes[2].set_xlabel("Azimuth Error (°)"); axes[2].set_ylabel("Count")
    axes[2].set_title("Azimuth Error Distribution"); axes[2].legend(facecolor=PLOT_STYLE["panel"])
    plt.suptitle("v13 Localization Model — Synthetic Test Set", color=PLOT_STYLE["accent"], fontsize=13)
    plt.tight_layout()
    out = cfg.DRIVE_PLOTS / "localization_scatter.png"
    cfg.DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)
    plt.savefig(str(out), dpi=150, bbox_inches="tight")
    print(f"💾 Saved: {out}"); plt.show()


def plot_polar_azimuth(azimuth_degs: list, title: str = "Detected Drone Azimuths",
                        cfg: Config = None, save: bool = True):
    """v13: Compass-rose polar plot of detected azimuths."""
    cfg = cfg or config
    fig = plt.figure(figsize=(6, 6), facecolor=PLOT_STYLE["bg"])
    ax  = fig.add_subplot(111, projection="polar")
    ax.set_facecolor(PLOT_STYLE["panel"])
    ax.tick_params(colors=PLOT_STYLE["text"])
    ax.title.set_color(PLOT_STYLE["accent"])
    rads = np.radians([90 - a for a in azimuth_degs])   # N-up, clockwise
    counts, bin_edges = np.histogram(rads, bins=36, range=(-np.pi, np.pi))
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    ax.bar(bin_centers, counts, width=bin_edges[1]-bin_edges[0], alpha=0.8,
           color=PLOT_STYLE["accent"], edgecolor=PLOT_STYLE["bg"])
    ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
    ax.set_title(title, pad=12); ax.grid(color=PLOT_STYLE["grid"], alpha=0.4)
    plt.tight_layout()
    if save:
        out = (cfg.DRIVE_PLOTS if cfg else Path("/tmp")) / "polar_azimuth.png"
        plt.savefig(str(out), dpi=150, bbox_inches="tight"); print(f"💾 Saved: {out}")
    plt.show()


def plot_track_trajectory(tracks: list, cfg: Config = None, save: bool = True):
    """v13: Top-down track paths with plasma colormap."""
    cfg = cfg or config
    fig, ax = plt.subplots(figsize=(8, 8))
    _apply_dark_style(fig, [ax])
    mics = cfg.MIC_POSITIONS
    ax.scatter(mics[:,0], mics[:,1], marker="^", s=200, c=PLOT_STYLE["warn"], zorder=10, label="Mics")
    cmap_fn = plt.get_cmap("plasma")
    for idx, t in enumerate(tracks):
        pts = np.array(t.positions)
        if len(pts) < 2: continue
        colors = [cmap_fn(i / max(len(pts)-1,1)) for i in range(len(pts))]
        for i in range(len(pts)-1):
            ax.plot(pts[i:i+2,0], pts[i:i+2,1], "-", color=colors[i], lw=2.5)
        ax.scatter(*pts[0], s=100, c=[cmap_fn(0)], marker=">", zorder=8)
        ax.scatter(*pts[-1],s=100, c=[cmap_fn(1)], marker="s", zorder=8, label=f"Track #{t.track_id}")
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
    ax.set_title(f"Kalman Track Trajectories ({len(tracks)} tracks)")
    ax.legend(facecolor=PLOT_STYLE["panel"]); ax.set_aspect("equal")
    plt.tight_layout()
    if save:
        out = (cfg.DRIVE_PLOTS if cfg else Path("/tmp")) / "track_trajectory.png"
        cfg.DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)
        plt.savefig(str(out), dpi=150, bbox_inches="tight"); print(f"💾 Saved: {out}")
    plt.show()


def plot_multi_drone_positions(drones: list, cfg: Config = None, save: bool = True):
    """v13: Top-down scatter with confidence radius circles."""
    cfg = cfg or config
    fig, ax = plt.subplots(figsize=(7, 7))
    _apply_dark_style(fig, [ax])
    mics = cfg.MIC_POSITIONS
    ax.scatter(mics[:,0], mics[:,1], marker="^", s=200, c=PLOT_STYLE["warn"], zorder=10, label="Mics")
    colors = [PLOT_STYLE["accent"], PLOT_STYLE["ok"], PLOT_STYLE["err"], PLOT_STYLE["purple"]]
    for i, d in enumerate(drones):
        xy = d["xy_position"]; cr = d.get("confidence_radius", 0.0)
        col = colors[i % len(colors)]
        ax.scatter(*xy, s=200, c=[col], zorder=8, label=f"Drone {i+1} az={d['azimuth_deg']:.1f}°")
        if cr > 0 and not math.isnan(cr):
            circ = plt.Circle(xy, cr, color=col, alpha=0.15, fill=True)
            ax.add_patch(circ)
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
    ax.set_title(f"Multi-Drone Positions ({len(drones)} detected)")
    ax.legend(facecolor=PLOT_STYLE["panel"]); ax.set_aspect("equal")
    plt.tight_layout()
    if save:
        out = (cfg.DRIVE_PLOTS if cfg else Path("/tmp")) / "multi_drone_positions.png"
        plt.savefig(str(out), dpi=150, bbox_inches="tight"); print(f"💾 Saved: {out}")
    plt.show()


class DetectionTrainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg; self.dev = torch.device(cfg.DEVICE)
        self.model = DetectionCNN().to(self.dev); self._loaders = None

    def _set_loaders(self, tr_l, va_l, te_l): self._loaders = (tr_l, va_l, te_l)

    def run(self, epochs: int = None, resume: bool = True):
        epochs = epochs or self.cfg.NUM_EPOCHS
        _set_seed(self.cfg.SEED)
        if self._loaders is not None:
            tr_l, va_l, te_l = self._loaders
        else:
            data_root = self.cfg.PROCESSED_DIR / "detection"
            try:
                tr = DetectionDataset(data_root, "train", augment=True, cfg=self.cfg)
                va = DetectionDataset(data_root, "val",   augment=False, cfg=self.cfg)
                te = DetectionDataset(data_root, "test",  augment=False, cfg=self.cfg)
            except RuntimeError as e: print(f"❌ {e}"); return
            lbs = np.array(tr.labels); cnt = np.bincount(lbs); cnt[cnt==0] = 1
            wts = (1.0/cnt)[lbs]; sampler = WeightedRandomSampler(wts, len(wts), replacement=True)
            def _collate(batch):
                xs, ys = zip(*batch); return torch.stack(xs), torch.stack(ys)
            tr_l = DataLoader(tr, batch_size=self.cfg.BATCH_SIZE, sampler=sampler, collate_fn=_collate)
            va_l = DataLoader(va, batch_size=self.cfg.BATCH_SIZE, shuffle=False,   collate_fn=_collate)
            te_l = DataLoader(te, batch_size=self.cfg.BATCH_SIZE, shuffle=False,   collate_fn=_collate)

        opt   = torch.optim.AdamW(self.model.parameters(), lr=self.cfg.LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "max", patience=4, factor=0.5)
        crit  = nn.CrossEntropyLoss()
        scaler = torch.amp.GradScaler("cuda", enabled=self.cfg.USE_AMP and self.cfg.DEVICE=="cuda")
        logger = TrainingLogger(self.cfg.DRIVE_LOGS/"detection_log.csv",
                                columns=["epoch","tr_loss","tr_acc","val_acc"])
        logged_epochs = {int(r["epoch"]) for r in logger.rows}
        start_epoch = 1; best_acc = 0.0
        ckpt_path = self.cfg.DRIVE_MODELS/"best_detection.pth"
        latest_path = self.cfg.DRIVE_MODELS/"latest_detection.pth"
        if resume and latest_path.exists():
            ck = torch.load(latest_path, map_location=self.dev)
            self.model.load_state_dict(ck["model_state"])
            start_epoch = ck.get("epoch",1) + 1
            if ckpt_path.exists():
                best_ck = torch.load(ckpt_path, map_location=self.dev)
                best_acc = best_ck.get("best_val_acc", 0.0)
            print(f"▶️  Resuming detection from epoch {start_epoch} (best acc: {best_acc:.1f}%)")
        elif resume and ckpt_path.exists() and not latest_path.exists():
            ck = torch.load(ckpt_path, map_location=self.dev)
            self.model.load_state_dict(ck["model_state"])
            start_epoch = ck.get("epoch",1) + 1; best_acc = ck.get("best_val_acc",0.0)

        for ep in range(start_epoch, start_epoch+epochs):
            self.model.train(); loss_sum = correct = total = 0
            pbar = tqdm(tr_l, desc=f"Det train ep {ep}", leave=False)
            for X, y in pbar:
                X = X.to(self.dev, non_blocking=True); y = y.to(self.dev, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=self.cfg.USE_AMP and self.cfg.DEVICE=="cuda"):
                    out = self.model(X); loss = crit(out, y)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                loss_sum += loss.item()*X.size(0); correct += (out.argmax(1)==y).sum().item(); total += X.size(0)
                pbar.set_postfix({"loss":f"{loss_sum/max(total,1):.4f}","acc":f"{100*correct/max(total,1):.1f}%"})
            tr_loss = loss_sum/max(total,1); tr_acc = 100*correct/max(total,1)
            val_acc = self._eval(va_l, desc=f"Det val ep {ep}"); sched.step(val_acc)
            print(f"  Det Ep {ep:3d} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.1f}%  val_acc={val_acc:.1f}%")
            self._save(latest_path, ep, val_acc)
            if ep not in logged_epochs:
                logger.log(epoch=ep, tr_loss=round(tr_loss,6), tr_acc=round(tr_acc,3), val_acc=round(val_acc,3))
                logged_epochs.add(ep)
            if val_acc > best_acc:
                best_acc = val_acc; self._save(ckpt_path, ep, best_acc); print("   ✨ New best!")
        print("\n🎯 Final detection test:")
        self._eval(te_l, verbose=True, desc="Det test")

    def _eval(self, loader, verbose=False, desc="Eval"):
        self.model.eval(); correct = total = 0; preds, labs = [], []
        with torch.no_grad():
            for X, y in tqdm(loader, desc=desc, leave=False):
                X = X.to(self.dev, non_blocking=True)
                out = self.model(X).argmax(1).cpu()
                preds.extend(out.numpy()); labs.extend(y.numpy())
                correct += (out==y).sum().item(); total += y.size(0)
        if verbose: print(classification_report(labs, preds, target_names=["non_drone","drone"]))
        return 100*correct/max(total,1)

    def _save(self, path, epoch, metric):
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"model_state":self.model.state_dict(),"epoch":epoch,"best_val_acc":metric}, path)
        print(f"💾 Saved {path.name} (ep={epoch}, acc={metric:.1f}%)")


class LocalizationTrainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg; self.dev = torch.device(cfg.DEVICE)
        self.model = make_localization_model(cfg).to(self.dev)

    def run(self, data_root: Path, epochs: int = None, use_synthetic_fallback=True, resume: bool = True):
        epochs = epochs or self.cfg.NUM_EPOCHS; _set_seed(self.cfg.SEED)
        # Reduce batch size for Lite mode to save memory
        bs = self.cfg.BATCH_SIZE
        if getattr(self.cfg, "USE_LITE_LOC", False): bs = max(8, bs // 2)
        try:
            tr_real  = LocalizationDataset(data_root, "train", augment=True,  cfg=self.cfg)
            va       = LocalizationDataset(data_root, "val",   augment=False, cfg=self.cfg)
            te       = LocalizationDataset(data_root, "test",  augment=False, cfg=self.cfg)
            tr_synth = SyntheticLocDataset(self.cfg, n_samples=100, augment=True)
            tr       = torch.utils.data.ConcatDataset([tr_real, tr_synth])
            print(f"   📊 Train: {len(tr_real)} real + {len(tr_synth)} synthetic = {len(tr)} total")
        except RuntimeError as e:
            if not use_synthetic_fallback: print(f"❌ {e}"); return
            print("⚠️  Real data unavailable — falling back to synthetic only.")
            tr = SyntheticLocDataset(self.cfg, n_samples=500, augment=True)
            va = SyntheticLocDataset(self.cfg, n_samples=50,  augment=False)
            te = SyntheticLocDataset(self.cfg, n_samples=50,  augment=False)
        tr_l = DataLoader(tr, batch_size=bs, shuffle=True, drop_last=True)
        va_l = DataLoader(va, batch_size=bs, shuffle=False)
        te_l = DataLoader(te, batch_size=bs, shuffle=False)
        opt    = torch.optim.AdamW(self.model.parameters(), lr=self.cfg.LR, weight_decay=1e-4)
        sched  = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, "min", patience=4, factor=0.5)
        scaler = torch.amp.GradScaler("cuda", enabled=self.cfg.USE_AMP and self.cfg.DEVICE=="cuda")
        logger = TrainingLogger(self.cfg.DRIVE_LOGS/"localization_log.csv",
                                columns=["epoch","tr_loss","val_loss","mae_az","mae_dist","mae_ht"])
        logged_epochs = {int(r["epoch"]) for r in logger.rows}
        start_epoch = 1; best_val = 1e9
        ckpt_path = self.cfg.DRIVE_MODELS/"best_localization.pth"
        latest_path = self.cfg.DRIVE_MODELS/"latest_localization.pth"
        if resume and latest_path.exists():
            ck = torch.load(latest_path, map_location=self.dev)
            self.model.load_state_dict(ck["model_state"])
            start_epoch = ck.get("epoch",1) + 1
            if ckpt_path.exists():
                best_val = torch.load(ckpt_path, map_location=self.dev).get("best_val_loss",1e9)
            print(f"▶️  Resuming localization from epoch {start_epoch} (best val_loss: {best_val:.5f})")
        elif resume and ckpt_path.exists() and not latest_path.exists():
            ck = torch.load(ckpt_path, map_location=self.dev)
            self.model.load_state_dict(ck["model_state"])
            start_epoch = ck.get("epoch",1)+1; best_val = ck.get("best_val_loss",1e9)

        for ep in range(start_epoch, start_epoch+epochs):
            self.model.train(); loss_sum = n = 0
            for mel, ipd, lbl in tqdm(tr_l, desc=f"Loc train ep {ep}", leave=False):
                mel=mel.to(self.dev); ipd=ipd.to(self.dev); lbl=lbl.to(self.dev)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=self.cfg.USE_AMP and self.cfg.DEVICE=="cuda"):
                    pred=self.model(mel,ipd); loss=localization_loss(pred,lbl)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                loss_sum+=loss.item()*mel.size(0); n+=mel.size(0)
            tr_loss=loss_sum/max(n,1)
            val_loss, mae_az, mae_dist, mae_ht = self._eval(va_l)
            sched.step(val_loss)
            print(f"  Loc Ep {ep:3d} | tr_loss={tr_loss:.5f} val_loss={val_loss:.5f} "
                  f"mae_az={mae_az:.2f}° mae_dist={mae_dist:.3f}m mae_ht={mae_ht:.3f}m")
            self._save(latest_path, ep, val_loss)
            if ep not in logged_epochs:
                logger.log(epoch=ep, tr_loss=round(tr_loss,6), val_loss=round(val_loss,6),
                           mae_az=round(mae_az,4), mae_dist=round(mae_dist,4), mae_ht=round(mae_ht,4))
                logged_epochs.add(ep)
            if val_loss < best_val:
                best_val=val_loss; self._save(ckpt_path, ep, val_loss); print("   ✨ New best!")
        print("\n🎯 Final localization test:"); self._eval(te_l, verbose=True)

    def _eval(self, loader, verbose=False):
        self.model.eval()
        loss_sum=n=0; az_errs=[]; dist_errs=[]; ht_errs=[]
        with torch.no_grad():
            for mel, ipd, lbl in tqdm(loader, desc="Loc eval", leave=False):
                mel=mel.to(self.dev); ipd=ipd.to(self.dev); lbl=lbl.to(self.dev)
                pred=self.model(mel,ipd)
                loss_sum+=localization_loss(pred,lbl).item()*mel.size(0); n+=mel.size(0)
                p=pred.cpu().numpy(); t=lbl.cpu().numpy()
                p_az = np.degrees(np.arctan2(p[:,0], p[:,1]))
                t_az = np.degrees(np.arctan2(t[:,0], t[:,1]))
                az_errs.extend(angular_error_deg(p_az, t_az).tolist())
                dist_errs.extend(np.abs(p[:,2]-t[:,2]).tolist())
                ht_errs.extend(np.abs(p[:,3]-t[:,3]).tolist())
        val_loss = loss_sum/max(n,1)
        mae_az   = float(np.mean(az_errs))
        mae_dist = float(np.mean(dist_errs))*self.cfg.MAX_LOCALIZATION_DIST
        mae_ht   = float(np.mean(ht_errs))*self.cfg.MAX_LOCALIZATION_DIST
        if verbose:
            print(f"  Val loss={val_loss:.5f}  MAE az={mae_az:.2f}°  "
                  f"dist={mae_dist:.3f}m  ht={mae_ht:.3f}m")
        return val_loss, mae_az, mae_dist, mae_ht

    def _save(self, path, epoch, metric):
        path.parent.mkdir(parents=True, exist_ok=True)
        torch.save({"model_state":self.model.state_dict(),"epoch":epoch,"best_val_loss":metric}, path)
        print(f"💾 Saved {path.name} (ep={epoch}, val_loss={metric:.5f})")

# ══════════════════════════════════════════════════════════════════════════════
# CELL 7 – Model Loaders (inference)
# ══════════════════════════════════════════════════════════════════════════════

_det_model = None
_loc_model = None


def load_detection_model(cfg: Config) -> DetectionCNN:
    global _det_model
    if _det_model is not None: return _det_model
    ckpt = cfg.DRIVE_MODELS / "best_detection.pth"
    if not ckpt.exists(): raise FileNotFoundError(f"No detection checkpoint at {ckpt}. Run train_detection() first.")
    dev  = torch.device(cfg.DEVICE)
    data = torch.load(ckpt, map_location=dev)
    m    = DetectionCNN().to(dev); m.load_state_dict(data["model_state"]); m.eval()
    _det_model = m
    print(f"✅ Detection model loaded (ep={data.get('epoch','?')}, acc={data.get('best_val_acc','?'):.1f}%)")
    return m


def load_localization_model(cfg: Config):
    global _loc_model
    if _loc_model is not None: return _loc_model
    ckpt = cfg.DRIVE_MODELS / "best_localization.pth"
    if not ckpt.exists(): raise FileNotFoundError(f"No localization checkpoint at {ckpt}. Run train_localization() first.")
    dev  = torch.device(cfg.DEVICE)
    data = torch.load(ckpt, map_location=dev)
    m    = make_localization_model(cfg).to(dev)
    m.load_state_dict(data["model_state"]); m.eval()
    _loc_model = m
    print(f"✅ Localization model loaded (ep={data.get('epoch','?')}, val_loss={data.get('best_val_loss','?'):.5f})")
    return m


def reload_models(cfg: Config):
    global _det_model, _loc_model
    _det_model = None; _loc_model = None
    load_detection_model(cfg); load_localization_model(cfg)


def _mel_tensor(channels: list, cfg: Config) -> torch.Tensor:
    ap = AudioProcessor(cfg)
    mels = [ap.mel(ap.pad_or_truncate(c)) for c in channels]
    return torch.tensor(np.stack(mels, axis=0), dtype=torch.float32).unsqueeze(0).to(cfg.DEVICE)


def _ipd_tensor(channels: list, cfg: Config) -> torch.Tensor:
    ipd = compute_ipd_features(channels, cfg)
    return torch.tensor(ipd, dtype=torch.float32).unsqueeze(0).to(cfg.DEVICE)


def heuristic_detect(audio: np.ndarray, cfg: Config) -> dict:
    y = np.asarray(audio, dtype=np.float32)
    if len(y) == 0: return {"probability": 0.0, "label": "non_drone", "features": {}}
    rms = float(np.sqrt(np.mean(y**2)) + 1e-8)
    rms_db = 20.0 * math.log10(rms + 1e-8)
    try:
        S = np.abs(librosa.stft(y, n_fft=cfg.N_FFT, hop_length=cfg.HOP_LENGTH))
        centroid  = float(np.mean(librosa.feature.spectral_centroid(S=S, sr=cfg.SR)))
        rolloff   = float(np.mean(librosa.feature.spectral_rolloff(S=S, sr=cfg.SR, roll_percent=0.85)))
        bandwidth = float(np.mean(librosa.feature.spectral_bandwidth(S=S, sr=cfg.SR)))
    except Exception: centroid = rolloff = bandwidth = 0.0
    try:
        f0, _, _ = librosa.pyin(y, fmin=50.0, fmax=500.0, sr=cfg.SR, hop_length=cfg.HOP_LENGTH, fill_na=0.0)
        f0 = np.nan_to_num(f0, nan=0.0).astype(np.float32)
        voiced_ratio = float(np.mean(f0 > 0.0))
        voiced_f0 = f0[f0 > 0.0]
        f0_med = float(np.median(voiced_f0)) if len(voiced_f0) > 0 else 0.0
        f0_std = float(np.std(voiced_f0)) if len(voiced_f0) > 0 else 0.0
    except Exception: voiced_ratio = f0_med = f0_std = 0.0
    energy_score    = np.clip((rms_db + 45.0) / 25.0, 0.0, 1.0)
    f0_score        = 1.0 if 70.0 <= f0_med <= 260.0 else (0.6 if 50.0 <= f0_med <= 350.0 else 0.0)
    voiced_score    = np.clip(voiced_ratio / 0.50, 0.0, 1.0)
    stability_score = 1.0 - np.clip(f0_std / 80.0, 0.0, 1.0) if f0_med > 0 else 0.2
    centroid_score  = 1.0 if 150.0 <= centroid <= 3000.0 else 0.35
    rolloff_score   = 1.0 if 500.0 <= rolloff <= 6000.0 else 0.4
    bandwidth_score = 1.0 if 200.0 <= bandwidth <= 3000.0 else 0.5
    score = (0.22*energy_score + 0.22*f0_score + 0.16*voiced_score + 0.12*stability_score
             + 0.12*centroid_score + 0.08*rolloff_score + 0.08*bandwidth_score)
    prob = sigmoid(8.0*(score - 0.5))
    return {"probability": float(prob), "label": classify_detection_score(prob, cfg),
            "features": {"rms_db":rms_db,"centroid_hz":centroid,"rolloff_hz":rolloff,
                         "bandwidth_hz":bandwidth,"voiced_ratio":voiced_ratio,
                         "f0_median_hz":f0_med,"f0_std_hz":f0_std}}


def detect(channels: list, cfg: Config, use_hybrid: bool = True) -> dict:
    m = load_detection_model(cfg); ap = AudioProcessor(cfg)
    y0 = ap.pad_or_truncate(channels[0])
    mel0 = ap.mel(y0)
    m0 = torch.tensor(np.stack([mel0,mel0,mel0], axis=0), dtype=torch.float32).unsqueeze(0).to(cfg.DEVICE)
    with torch.no_grad(): cnn_prob = float(torch.softmax(m(m0), dim=1)[0,1].item())
    heur = heuristic_detect(y0, cfg); heuristic_prob = float(heur["probability"])
    if use_hybrid:
        fused_prob = float(cfg.CNN_WEIGHT)*cnn_prob + float(cfg.HEURISTIC_WEIGHT)*heuristic_prob
        if cnn_prob < 0.15 and heuristic_prob > 0.85: fused_prob = max(fused_prob, 0.55)
        if cnn_prob > 0.40 and heuristic_prob > 0.40: fused_prob = min(1.0, fused_prob + 0.08)
    else: fused_prob = cnn_prob
    label = classify_detection_score(fused_prob, cfg)
    return {"detected": bool(fused_prob >= cfg.DETECTION_THRESHOLD),
            "probability": float(fused_prob), "label": label,
            "cnn_probability": float(cnn_prob), "heuristic_probability": float(heuristic_prob),
            "heuristic_features": heur["features"]}


def localize(channels: list, cfg: Config) -> dict:
    m   = load_localization_model(cfg)
    mel = _mel_tensor(channels, cfg); ipd = _ipd_tensor(channels, cfg)
    with torch.no_grad(): pred = m(mel, ipd)[0].cpu().numpy()
    sin_az, cos_az, dist_raw, ht_raw = pred
    az_deg = wrap_angle_deg(float(np.degrees(np.arctan2(sin_az, cos_az))))
    dist_m = float(abs(dist_raw)*cfg.MAX_LOCALIZATION_DIST)
    ht_m   = float(abs(ht_raw)*cfg.MAX_LOCALIZATION_DIST)
    xy     = azimuth_deg_to_xy(az_deg, dist_m, cfg.ARRAY_CENTER)
    return {"azimuth_deg": az_deg, "distance_m": dist_m, "height_m": ht_m, "xy_position": xy}

# ══════════════════════════════════════════════════════════════════════════════
# CELL 8 – Kalman Filter tracker
# ══════════════════════════════════════════════════════════════════════════════

class KalmanTrack:
    _id_counter = 0
    def __init__(self, xy: np.ndarray, dt: float = 1.0, cfg: Config = None):
        KalmanTrack._id_counter += 1; self.track_id = KalmanTrack._id_counter
        cfg = cfg or config; self.cfg = cfg
        sigma_q = cfg.KF_PROCESS_NOISE; sigma_r = cfg.KF_MEASURE_NOISE
        self.P = np.diag([sigma_r**2, sigma_r**2, 1.0, 1.0]).astype(np.float64)
        self.x = np.array([xy[0], xy[1], 0.0, 0.0], dtype=np.float64)
        self.R = np.diag([sigma_r**2, sigma_r**2]).astype(np.float64)
        self.H = np.array([[1,0,0,0],[0,1,0,0]], dtype=np.float64)
        self.age = 0; self.hits = 1; self.positions = [xy.copy()]; self.timestamps = [time.time()]; self.active = True

    def _F(self, dt):
        return np.array([[1,0,dt,0],[0,1,0,dt],[0,0,1,0],[0,0,0,1]], dtype=np.float64)

    def _Q(self, dt):
        s = self.cfg.KF_PROCESS_NOISE; dt2=dt**2; dt3=dt**3; dt4=dt**4
        return np.array([[dt4/4,0,dt3/2,0],[0,dt4/4,0,dt3/2],[dt3/2,0,dt2,0],[0,dt3/2,0,dt2]],dtype=np.float64)*s**2

    def predict(self, dt=1.0):
        F=self._F(dt); Q=self._Q(dt); self.x=F@self.x; self.P=F@self.P@F.T+Q; self.age+=1; return self.x[:2].copy()

    def update(self, xy: np.ndarray, timestamp: float = None):
        z=xy.astype(np.float64); S=self.H@self.P@self.H.T+self.R; K=self.P@self.H.T@np.linalg.inv(S)
        self.x=self.x+K@(z-self.H@self.x); self.P=(np.eye(4)-K@self.H)@self.P
        self.positions.append(xy.copy()); self.timestamps.append(timestamp or time.time()); self.age=0; self.hits+=1

    def predicted_xy(self): return self.x[:2].copy()
    def velocity(self): return self.x[2:4].copy()
    def uncertainty_radius(self): return float(np.sqrt(np.trace(self.P[:2,:2])))
    def total_distance(self):
        pts=np.array(self.positions)
        return float(np.sum(np.linalg.norm(np.diff(pts,axis=0),axis=1))) if len(pts)>=2 else 0.0

    def to_dict(self):
        return {"track_id":self.track_id,"positions":[p.tolist() for p in self.positions],
                "timestamps":self.timestamps,"velocity_mps":self.velocity().tolist(),
                "total_dist_m":self.total_distance(),"active":self.active}


class KalmanTracker:
    def __init__(self, cfg: Config):
        self.cfg=cfg; self.tracks=[]; self.frame=0; self.dt=1.0

    def step(self, detections: list, timestamp: float = None):
        ts=timestamp or time.time()
        if self.tracks:
            prev_ts=self.tracks[0].timestamps[-1] if self.tracks[0].timestamps else ts
            self.dt=max(0.01, ts-prev_ts)
        predicted={}
        for t in self.tracks:
            if t.active: predicted[t.track_id]=t.predict(self.dt)
        active=[t for t in self.tracks if t.active]; unmatched=list(range(len(detections)))
        for track in sorted(active, key=lambda t:-t.hits):
            if not unmatched: break
            pred=predicted[track.track_id]
            dists=[(i, np.linalg.norm(pred-detections[i])) for i in unmatched]
            dists.sort(key=lambda d:d[1]); best_i, best_d = dists[0]
            if best_d <= self.cfg.KF_MATCH_GATE:
                track.update(detections[best_i], ts); unmatched.remove(best_i)
        for i in unmatched: self.tracks.append(KalmanTrack(detections[i], self.dt, self.cfg))
        for t in self.tracks:
            if t.age > self.cfg.KF_MAX_COAST: t.active = False
        self.frame+=1
        return [t for t in self.tracks if t.active and t.hits>=self.cfg.KF_MIN_HITS]

    def all_confirmed(self): return [t for t in self.tracks if t.hits>=self.cfg.KF_MIN_HITS]

    def save(self, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        with open(path,"w") as f: json.dump([t.to_dict() for t in self.tracks], f, indent=2)
        print(f"💾 Tracks saved: {path}")

# ══════════════════════════════════════════════════════════════════════════════
# CELL 9 – Multi-drone localization
# ══════════════════════════════════════════════════════════════════════════════

def _tdoa_residual(pos, tdoas, mics, c=343.0):
    pos = np.asarray(pos, dtype=float)
    d = np.linalg.norm(mics - pos[None,:], axis=1)
    tau12_pred = (d[1]-d[0]) / c
    tau13_pred = (d[2]-d[0]) / c
    tau23_pred = (d[2]-d[1]) / c
    return (tau12_pred-tdoas[0])**2 + (tau13_pred-tdoas[1])**2 + (tau23_pred-tdoas[2])**2


def _nelder_mead_localize(tdoas, mics, c, max_dist):
    cx, cy = mics.mean(axis=0)
    best_r, best_a, best_e = 1.0, 0.0, 1e12
    for r in [0.2, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0, 8.0]:
        for a in np.linspace(0, 2*np.pi, 24, endpoint=False):
            pos = np.array([cx+r*np.cos(a), cy+r*np.sin(a)])
            e = _tdoa_residual(pos, tdoas, mics, c)
            if e < best_e: best_e, best_r, best_a = e, r, a
    def obj(p):
        r = min(abs(p[0]), max_dist)
        return _tdoa_residual([cx+r*np.cos(p[1]), cy+r*np.sin(p[1])], tdoas, mics, c)
    res = scipy.optimize.minimize(obj, x0=[best_r, best_a], method="Nelder-Mead",
                                  options={"xatol":1e-7,"fatol":1e-15,"maxiter":10000})
    r_opt = min(abs(res.x[0]), max_dist)
    pos   = np.array([cx+r_opt*np.cos(res.x[1]), cy+r_opt*np.sin(res.x[1])], dtype=np.float32)
    err   = _tdoa_residual(pos, tdoas, mics, c)
    return pos, err


def localize_multi_drone(channels: list, cfg: Config, max_drones: int = None) -> list:
    max_drones = max_drones or cfg.MAX_DRONES
    mics = cfg.MIC_POSITIONS; c = cfg.SPEED_OF_SOUND; sr = cfg.SR
    max_tau = np.max([np.linalg.norm(mics[i]-mics[j])
                      for i in range(3) for j in range(i+1,3)]) / c * 1.5
    chs_bp = [bandpass(ch, sr, 200, 5000) for ch in channels]
    peaks12 = gcc_phat_peaks(chs_bp[1], chs_bp[0], sr, max_tau, max_drones+1)
    peaks13 = gcc_phat_peaks(chs_bp[2], chs_bp[0], sr, max_tau, max_drones+1)
    peaks23 = gcc_phat_peaks(chs_bp[2], chs_bp[1], sr, max_tau, max_drones+1)
    DEDUP = cfg.TDOA_DEDUP_MS
    candidates = []
    for tau12, s12 in peaks12:
        for tau13, s13 in peaks13:
            tau23_pred = tau13 - tau12
            best23 = min(peaks23, key=lambda x: abs(x[0]-tau23_pred))
            if abs(best23[0]-tau23_pred) < DEDUP*50:
                candidates.append((tau12, tau13, tau13-tau12, s12+s13))
    candidates.sort(key=lambda x: -x[3])
    drones = []; seen_pos = []; seen_tds = []
    for tau12, tau13, tau23, strength in candidates:
        if len(drones) >= max_drones: break
        if any(abs(tau12-st[0]) < DEDUP and abs(tau13-st[1]) < DEDUP for st in seen_tds): continue
        tdoas = np.array([tau12, tau13, tau23])
        pos, err = _nelder_mead_localize(tdoas, mics, c, cfg.MAX_LOCALIZATION_DIST)
        if err > 1e-6: continue
        if any(np.linalg.norm(pos-sp) < 0.15 for sp in seen_pos): continue
        eps = 1e-3
        try:
            hxx = (_tdoa_residual(pos+[eps,0],tdoas,mics,c)+_tdoa_residual(pos-[eps,0],tdoas,mics,c)-2*err)/eps**2
            hyy = (_tdoa_residual(pos+[0,eps],tdoas,mics,c)+_tdoa_residual(pos-[0,eps],tdoas,mics,c)-2*err)/eps**2
            cr  = float(min(np.sqrt(1/max(hxx,1e-10)+1/max(hyy,1e-10))*0.5, 20.0))
        except: cr = float("nan")
        cx_, cy_ = cfg.ARRAY_CENTER
        az_deg = math.degrees(math.atan2(pos[1]-cy_, pos[0]-cx_))
        dist_m = float(np.linalg.norm(pos - cfg.ARRAY_CENTER))
        drones.append({"xy_position":pos,"azimuth_deg":az_deg,"distance_m":dist_m,
                       "tdoa_residual":err,"confidence_radius":cr})
        seen_pos.append(pos.copy()); seen_tds.append((tau12, tau13))
    return drones

# ══════════════════════════════════════════════════════════════════════════════
# CELL 10 – Inference pipeline
# ══════════════════════════════════════════════════════════════════════════════

def load_3ch(paths: list, cfg: Config) -> list:
    ap = AudioProcessor(cfg)
    return [ap.pad_or_truncate(ap.load(p)) for p in paths]


def run_pipeline(wav_paths: list, cfg: Config, tracker: KalmanTracker = None,
                 multi_drone: bool = False, timestamp: float = None) -> dict:
    ts = timestamp or time.time()
    channels = load_3ch(wav_paths, cfg)
    det = detect(channels, cfg)
    result = {"detected": det["detected"], "probability": det["probability"],
              "cnn_probability": det.get("cnn_probability", float("nan")),
              "heuristic_probability": det.get("heuristic_probability", float("nan")),
              "drones": [], "tracks": []}
    if not det["detected"]:
        if tracker: result["tracks"] = tracker.step([], ts)
        return result
    if multi_drone:
        drone_locs = localize_multi_drone(channels, cfg)
        result["drones"] = drone_locs
        positions = [d["xy_position"] for d in drone_locs]
    else:
        loc = localize(channels, cfg)
        result["drones"] = [loc]
        positions = [loc["xy_position"]]
    if tracker and positions: result["tracks"] = tracker.step(positions, ts)
    return result

# ══════════════════════════════════════════════════════════════════════════════
# CELL 11 – Segment-based analysis  (v13 enhanced 6-panel dashboard)
# ══════════════════════════════════════════════════════════════════════════════

def synthesise_3ch(audio: np.ndarray, sr: int, drone_pos, mic_positions: np.ndarray) -> list:
    c = 343.0; src = np.array(drone_pos, dtype=float); n = len(audio)
    dists = np.linalg.norm(mic_positions - src[None,:], axis=1)
    rel   = (dists - dists.min()) / c * sr
    out   = []
    for i in range(len(mic_positions)):
        delayed = _fractional_delay(audio.copy(), rel[i])
        out.append(delayed + (0.003*np.random.randn(n)).astype(np.float32))
    return out


def _plot_analysis_report(segments, confirmed, cfg, title):
    """v13: 6-panel dark-themed analysis dashboard."""
    import matplotlib.gridspec as gridspec
    fig = plt.figure(figsize=(20, 10), facecolor=PLOT_STYLE["bg"])
    fig.suptitle(f"🚁 Drone Analysis — {title}", fontsize=14,
                 color=PLOT_STYLE["accent"], fontweight="bold", y=0.98)
    gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)
    axes = [fig.add_subplot(gs[r,c]) for r in range(2) for c in range(3)]
    _apply_dark_style(fig, axes)

    ts_list = [s["t_start"] for s in segments]

    # ── [0,0] Waveform ────────────────────────────────────────────────────
    ax = axes[0]
    if any("waveform" in s for s in segments):
        import itertools
        wave = list(itertools.chain.from_iterable(s.get("waveform",[]) for s in segments))
        t_wave = np.linspace(0, max(ts_list)+cfg.TARGET_DURATION if ts_list else 3.0, len(wave))
        ax.plot(t_wave, wave, color=PLOT_STYLE["accent"], lw=0.6, alpha=0.7)
        rms_vals = [s.get("rms_db", -60) for s in segments]
        ax2 = ax.twinx()
        ax2.plot(ts_list, rms_vals, "o-", color=PLOT_STYLE["warn"], ms=4, lw=1.5, label="RMS dB")
        ax2.tick_params(colors=PLOT_STYLE["text"]); ax2.yaxis.label.set_color(PLOT_STYLE["text"])
        ax2.set_ylabel("RMS (dB)", color=PLOT_STYLE["text"])
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude"); ax.set_title("Waveform + RMS")

    # ── [0,1] Mel spectrogram ─────────────────────────────────────────────
    ax = axes[1]
    if any("mel" in s for s in segments):
        mel_frames = np.concatenate([s["mel"] for s in segments if "mel" in s], axis=1)
        ax.imshow(mel_frames, aspect="auto", origin="lower", cmap="magma",
                  extent=[0, max(ts_list)+cfg.TARGET_DURATION if ts_list else 3.0,
                          0, cfg.SR//2 / 1000])
        plt.colorbar(ax.images[0], ax=ax, label="dB").ax.yaxis.label.set_color(PLOT_STYLE["text"])
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Freq (kHz)"); ax.set_title("Mel Spectrogram")

    # ── [0,2] Probability timeline ────────────────────────────────────────
    ax = axes[2]
    prbs  = [s["prob"]  for s in segments]
    cnns  = [s.get("cnn_probability",  float("nan")) for s in segments]
    heurs = [s.get("heuristic_probability", float("nan")) for s in segments]
    cols  = [PLOT_STYLE["ok"] if s["detected"] else PLOT_STYLE["err"] for s in segments]
    w = cfg.TARGET_DURATION * 0.8
    ax.bar(ts_list, prbs, width=w, color=cols, alpha=0.55, label="Hybrid")
    ax.fill_between(ts_list, prbs, alpha=0.15, color=PLOT_STYLE["accent"])
    if not all(math.isnan(v) for v in cnns):
        ax.plot(ts_list, cnns,  "-o", color=PLOT_STYLE["accent"], ms=4, lw=1.5, label="CNN")
    if not all(math.isnan(v) for v in heurs):
        ax.plot(ts_list, heurs, "--s", color=PLOT_STYLE["purple"], ms=4, lw=1.5, label="Heuristic")
    ax.axhline(cfg.DETECTION_THRESHOLD, color=PLOT_STYLE["warn"], lw=1.5, ls="--",
               label=f"Thr={cfg.DETECTION_THRESHOLD:.2f}")
    ax.set_xlim(left=0); ax.set_ylim(0, 1.05)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Probability"); ax.set_title("Detection Timeline")
    ax.legend(facecolor=PLOT_STYLE["panel"], fontsize=8)

    # ── [1,0] Localization bar chart ──────────────────────────────────────
    ax = axes[3]
    locs = [s for s in segments if s.get("loc") is not None]
    if locs:
        az_vals   = [s["loc"]["azimuth_deg"]  / 180.0 for s in locs]
        dist_vals = [s["loc"]["distance_m"]   / cfg.MAX_LOCALIZATION_DIST for s in locs]
        ht_vals   = [s["loc"]["height_m"]     / cfg.MAX_LOCALIZATION_DIST for s in locs]
        t_locs    = [s["t_start"] for s in locs]
        ax.bar(t_locs, az_vals,   width=w, color=PLOT_STYLE["accent"], alpha=0.7, label="Az/180°")
        ax.bar(t_locs, dist_vals, width=w, color=PLOT_STYLE["ok"],     alpha=0.7, label="Dist/MaxDist", bottom=az_vals)
    else:
        ax.text(0.5, 0.5, "No localization data", ha="center", va="center",
                color=PLOT_STYLE["muted"], transform=ax.transAxes)
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Normalised"); ax.set_title("Localization")
    if locs: ax.legend(facecolor=PLOT_STYLE["panel"], fontsize=8)

    # ── [1,1] Polar azimuth compass ───────────────────────────────────────
    axes[4].remove()
    ax_polar = fig.add_subplot(gs[1,1], projection="polar")
    ax_polar.set_facecolor(PLOT_STYLE["panel"])
    ax_polar.tick_params(colors=PLOT_STYLE["text"])
    az_degs = [s["loc"]["azimuth_deg"] for s in segments if s.get("loc") is not None]
    if az_degs:
        rads = np.radians([90 - a for a in az_degs])
        counts, bin_edges = np.histogram(rads, bins=24, range=(-np.pi, np.pi))
        bin_centers = 0.5*(bin_edges[:-1]+bin_edges[1:])
        ax_polar.bar(bin_centers, counts, width=bin_edges[1]-bin_edges[0],
                     alpha=0.8, color=PLOT_STYLE["accent"], edgecolor=PLOT_STYLE["bg"])
    ax_polar.set_theta_zero_location("N"); ax_polar.set_theta_direction(-1)
    ax_polar.set_title("Azimuth (N-up)", color=PLOT_STYLE["accent"], pad=12)
    ax_polar.grid(color=PLOT_STYLE["grid"], alpha=0.4)

    # ── [1,2] Detection score gauge ───────────────────────────────────────
    ax = axes[5]
    all_probs = [s["prob"] for s in segments]
    final_score = float(np.max(all_probs)) if all_probs else 0.0
    theta_range = np.linspace(np.pi, 0, 200)
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.1, 1.2)
    # background arc
    ax.plot(np.cos(theta_range), np.sin(theta_range), lw=18,
            color=PLOT_STYLE["panel"], solid_capstyle="round")
    # coloured fill
    fill_theta = np.linspace(np.pi, np.pi*(1 - final_score), 200)
    color = PLOT_STYLE["ok"] if final_score >= cfg.DETECTION_THRESHOLD else PLOT_STYLE["err"]
    ax.plot(np.cos(fill_theta), np.sin(fill_theta), lw=18, color=color, solid_capstyle="round")
    # needle
    needle_angle = np.pi*(1 - final_score)
    ax.annotate("", xy=(0.8*np.cos(needle_angle), 0.8*np.sin(needle_angle)), xytext=(0,0),
                arrowprops=dict(arrowstyle="-|>", color=PLOT_STYLE["text"], lw=2))
    ax.text(0, -0.08, f"{final_score:.3f}", ha="center", va="center",
            fontsize=16, fontweight="bold", color=color)
    ax.text(0, 0.6, "DRONE" if final_score >= cfg.DETECTION_THRESHOLD else "CLEAR",
            ha="center", va="center", fontsize=10, color=color)
    ax.text(-1.0, 0.0, "0.0", ha="center", color=PLOT_STYLE["muted"], fontsize=8)
    ax.text(1.0, 0.0, "1.0", ha="center", color=PLOT_STYLE["muted"], fontsize=8)
    ax.axis("off"); ax.set_title("Detection Score", color=PLOT_STYLE["accent"])

    # save
    slug = Path(title).stem
    cfg.DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)
    save_path = cfg.DRIVE_PLOTS / f"analysis_{slug}.png"
    plt.savefig(str(save_path), dpi=150, bbox_inches="tight")
    print(f"💾 Analysis dashboard saved: {save_path}")
    plt.show()


def _plot_analysis(segments, confirmed, cfg, title):
    """Legacy 3-panel analysis plot (kept for backward compatibility)."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Pipeline Analysis — {title}", fontsize=13, fontweight="bold")
    ax = axes[0]
    ts   = [s["t_start"] for s in segments]
    prbs = [s["prob"]    for s in segments]
    cols = ["#27ae60" if s["detected"] else "#e74c3c" for s in segments]
    ax.bar(ts, prbs, color=cols, width=cfg.TARGET_DURATION*0.8, alpha=0.75, label="Hybrid prob")
    cnn_probs  = [s.get("cnn_probability",  float("nan")) for s in segments]
    heur_probs = [s.get("heuristic_probability", float("nan")) for s in segments]
    if not np.all(np.isnan(cnn_probs)):  ax.plot(ts, cnn_probs,  "-o", label="CNN prob")
    if not np.all(np.isnan(heur_probs)): ax.plot(ts, heur_probs, "--s", label="Heuristic prob")
    ax.axhline(cfg.DETECTION_THRESHOLD, color="#e67e22", ls="--", lw=1.5, label=f"Thr={cfg.DETECTION_THRESHOLD:.2f}")
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Drone Probability")
    ax.set_title("Detection Confidence"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    ax = axes[1]
    mics = cfg.MIC_POSITIONS
    ax.scatter(mics[:,0], mics[:,1], marker="^", s=200, c="black", zorder=10, label="Mics")
    for i, m in enumerate(mics):
        ax.annotate(f"M{i+1}", m, textcoords="offset points", xytext=(5,5), fontsize=8)
    raw_xys = [s["xy"] for s in segments if s["xy"] is not None]
    if raw_xys:
        raw = np.array(raw_xys)
        ax.scatter(raw[:,0], raw[:,1], s=40, c="#e74c3c", alpha=0.4, label="Raw estimates")
    cmap = plt.colormaps["tab10"]
    for idx, t in enumerate(confirmed):
        pts = np.array(t.positions); c_col = cmap(idx%10)
        ax.plot(pts[:,0], pts[:,1], "-o", color=c_col, lw=2.5, ms=6, label=f"Track #{t.track_id}")
        ax.scatter(*pts[0],  s=120, c=[c_col], marker=">", zorder=8)
        ax.scatter(*pts[-1], s=120, c=[c_col], marker="s", zorder=8)
    ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
    ax.set_title(f"Trajectory ({len(confirmed)} confirmed track(s))")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3); ax.set_aspect("equal")
    ax = axes[2]
    for idx, t in enumerate(confirmed):
        cr = t.uncertainty_radius()
        ax.text(idx+1, cr, f"±{cr:.2f}m", ha="center", fontsize=9)
        ax.bar(idx+1, cr, color=cmap(idx%10), alpha=0.75)
    ax.set_xlabel("Track ID"); ax.set_ylabel("Final σ radius (m)")
    ax.set_title("Kalman Uncertainty"); ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    try: save_dir = cfg.DRIVE_ROOT; save_dir.mkdir(parents=True, exist_ok=True)
    except OSError: save_dir = Path("/tmp")
    save = save_dir / f"analysis_{Path(title).stem}.png"
    plt.savefig(str(save), dpi=150, bbox_inches="tight")
    print(f"💾 Analysis plot saved: {save}")
    plt.show()


def _plot_external_detection_scores(segment_results, threshold: float, cfg: Config, title: str):
    """Dark-themed segment probability chart for external audio analysis."""
    if not segment_results: return
    t     = [0.5*(s["t_start_s"]+s["t_end_s"]) for s in segment_results]
    fused = [s["probability"]          for s in segment_results]
    cnn   = [s["cnn_probability"]      for s in segment_results]
    heur  = [s["heuristic_probability"]for s in segment_results]
    cols  = [PLOT_STYLE["ok"] if s["detected_at_external_threshold"] else PLOT_STYLE["err"]
             for s in segment_results]
    fig, ax = plt.subplots(figsize=(12, 5))
    _apply_dark_style(fig, [ax])
    width = max((t[1]-t[0])*0.8 if len(t)>1 else 0.5, 0.15)
    ax.bar(t, fused, width=width, color=cols, alpha=0.55, label="Hybrid prob")
    ax.plot(t, cnn,  "-o",  color=PLOT_STYLE["accent"], label="CNN prob")
    ax.plot(t, heur, "--s", color=PLOT_STYLE["purple"], label="Heuristic prob")
    ax.axhline(threshold,             color=PLOT_STYLE["warn"], ls="--", lw=1.5, label=f"Ext thr={threshold:.2f}")
    ax.axhline(cfg.DETECTION_THRESHOLD, color=PLOT_STYLE["err"], ls=":",  lw=1.5, label=f"Main thr={cfg.DETECTION_THRESHOLD:.2f}")
    ax.set_title(f"Robust External Detection — {title}", color=PLOT_STYLE["accent"])
    ax.set_xlabel("Time (s)"); ax.set_ylabel("Probability"); ax.set_ylim(0, 1.05)
    ax.legend(facecolor=PLOT_STYLE["panel"]); plt.tight_layout(); plt.show()


def analyse_audio_file(audio_path: str, cfg: Config,
                       drone_pos=None, n_segments: int = 10,
                       threshold_override: float = None,
                       show_plot: bool = True,
                       use_synthesis: bool = False) -> dict:
    """
    v13: Enhanced segment-level analysis with 6-panel dark dashboard.
    use_synthesis=False (default): detects directly on raw mono — no domain corruption.
    use_synthesis=True: legacy path, simulates 3-mic geometry (for known drone pos testing).
    """
    if threshold_override is not None:
        old_thr = cfg.DETECTION_THRESHOLD; cfg.DETECTION_THRESHOLD = threshold_override
    ap = AudioProcessor(cfg); y_full = ap.load(audio_path, mono=True)
    total = len(y_full) / cfg.SR; seg_s = int(cfg.TARGET_DURATION * cfg.SR)
    hop   = max(seg_s, int((len(y_full)-seg_s) / max(n_segments-1,1)))
    dp    = drone_pos or [1.0, 0.8]
    load_detection_model(cfg)
    try: load_localization_model(cfg); can_localize = True
    except FileNotFoundError: can_localize = False
    tracker  = KalmanTracker(cfg); segments = []; base_ts = time.time()
    mode_str = "synthesis" if use_synthesis else "direct mono"
    print(f"\n🎵 {Path(audio_path).name}  ({total:.1f}s)  |  {n_segments} segments  |  mode={mode_str}")
    for seg_i in range(n_segments):
        start = min(seg_i * hop, max(0, len(y_full)-seg_s))
        audio = y_full[start:start+seg_s]
        if len(audio) < seg_s: audio = np.pad(audio, (0, seg_s-len(audio)))
        t_s = start / cfg.SR
        mel_frame = AudioProcessor(cfg).mel(ap.pad_or_truncate(audio))
        rms_db = float(20*np.log10(np.sqrt(np.mean(audio**2))+1e-8))
        if use_synthesis:
            chs = synthesise_3ch(audio, cfg.SR, dp, cfg.MIC_POSITIONS)
            det = detect(chs, cfg)
            tmp = []
            for ch in chs:
                tf = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
                sf.write(tf.name, ch, cfg.SR); tmp.append(tf.name)
            try:
                res = run_pipeline(tmp, cfg, tracker=tracker, timestamp=base_ts+t_s)
                res["cnn_probability"] = det.get("cnn_probability", float("nan"))
                res["heuristic_probability"] = det.get("heuristic_probability", float("nan"))
            finally:
                for p in tmp: os.unlink(p)
            drone_loc = res["drones"][0] if res["drones"] else None
        else:
            det = detect([audio, audio, audio], cfg)
            drone_loc = None
            if det["detected"] and can_localize:
                drone_loc = {"azimuth_deg":0.0,"distance_m":0.0,"height_m":0.0,
                             "xy_position":np.array(cfg.ARRAY_CENTER)}
            positions = [drone_loc["xy_position"]] if drone_loc else []
            tracks = tracker.step(positions, base_ts+t_s)
            res = {"detected": det["detected"], "probability": det["probability"],
                   "cnn_probability": det.get("cnn_probability", float("nan")),
                   "heuristic_probability": det.get("heuristic_probability", float("nan")),
                   "drones": [drone_loc] if drone_loc else [], "tracks": tracks}
        segments.append({
            "seg": seg_i+1, "t_start": t_s,
            "detected": res["detected"], "prob": res["probability"],
            "cnn_probability": res.get("cnn_probability", float("nan")),
            "heuristic_probability": res.get("heuristic_probability", float("nan")),
            "xy": res["drones"][0]["xy_position"] if res["drones"] else None,
            "loc": res["drones"][0] if res["drones"] else None,
            "mel": mel_frame, "rms_db": rms_db,
        })
        icon = "🚁" if res["detected"] else "🌳"
        print(f"  Seg {seg_i+1:3d}  {icon}  conf={res['probability']:.3f}  rms={rms_db:.1f}dB")
    confirmed = tracker.all_confirmed()
    n_det = sum(s["detected"] for s in segments)
    print(f"\n  📊 {n_det}/{n_segments} detected  |  {len(confirmed)} confirmed track(s)")
    if show_plot: _plot_analysis_report(segments, confirmed, cfg, Path(audio_path).name)
    if threshold_override is not None: cfg.DETECTION_THRESHOLD = old_thr
    return {"segments": segments, "tracker": tracker, "tracks": confirmed,
            "detected": n_det > 0, "probability": max((s["prob"] for s in segments), default=0.0),
            "duration_sec": total}


def analyse_external_audio_robust(audio_path: str, cfg: Config,
                                   segment_sec: float = None, overlap: float = None,
                                   threshold: float = None, min_pos_segments: int = None,
                                   agg_mode: str = None, topk: int = None,
                                   show_plot: bool = True) -> dict:
    segment_sec       = segment_sec or cfg.EXTERNAL_SEGMENT_SEC
    overlap           = cfg.EXTERNAL_SEGMENT_OVERLAP if overlap is None else overlap
    threshold         = threshold if threshold is not None else cfg.EXTERNAL_INFER_THRESHOLD
    min_pos_segments  = min_pos_segments or cfg.EXTERNAL_MIN_POS_SEGMENTS
    agg_mode          = agg_mode or cfg.EXTERNAL_AGG_MODE
    topk              = topk or cfg.EXTERNAL_TOPK
    ap = AudioProcessor(cfg); y = ap.load(audio_path, mono=True); total_s = len(y)/cfg.SR
    seg_len = max(1, int(segment_sec*cfg.SR)); hop_len = max(1, int(seg_len*(1.0-overlap)))
    starts = list(range(0, len(y)-seg_len+1, hop_len)) if len(y) > seg_len else [0]
    if starts and starts[-1] != len(y)-seg_len: starts.append(len(y)-seg_len)
    segment_results = []
    for i, start in enumerate(starts):
        seg = ap.pad_or_truncate(y[start:start+seg_len], seg_len)
        res = detect([seg,seg,seg], cfg, use_hybrid=True)
        t0 = start/cfg.SR; t1 = (start+seg_len)/cfg.SR
        segment_results.append({
            "segment_index": i+1, "t_start_s": float(t0), "t_end_s": float(t1),
            "probability": float(res["probability"]), "cnn_probability": float(res["cnn_probability"]),
            "heuristic_probability": float(res["heuristic_probability"]), "label": res["label"],
            "detected_at_main_threshold": bool(res["probability"] >= cfg.DETECTION_THRESHOLD),
            "detected_at_external_threshold": bool(res["probability"] >= threshold),
        })
    probs = [s["probability"] for s in segment_results]
    probs_sorted = sorted(probs, reverse=True)
    if agg_mode == "max": clip_score = float(max(probs)) if probs else 0.0
    elif agg_mode == "mean_topk": clip_score = safe_prob_average(probs_sorted[:max(1,topk)], default=0.0)
    elif agg_mode == "vote": clip_score = float(sum(p>=threshold for p in probs)/max(len(probs),1))
    else: clip_score = safe_prob_average(probs_sorted[:max(1,topk)], default=0.0)
    pos_count = sum(s["detected_at_external_threshold"] for s in segment_results)
    clip_detected = clip_score >= threshold or pos_count >= min_pos_segments
    clip_label = ("drone" if clip_detected and clip_score >= threshold
                  else "possible_drone" if pos_count >= 1 or clip_score >= cfg.DETECTION_THRESHOLD_WEAK
                  else "non_drone")
    print(f"\n🎧 Robust External Audio Analysis")
    print(f"🎵 File: {Path(audio_path).name} | duration={total_s:.2f}s | {len(segment_results)} segments")
    for s in segment_results:
        icon = "🚁" if s["detected_at_external_threshold"] else "🌳"
        print(f"  Seg {s['segment_index']:3d} {icon} prob={s['probability']:.3f} "
              f"cnn={s['cnn_probability']:.3f} heur={s['heuristic_probability']:.3f} "
              f"[{s['t_start_s']:.2f}-{s['t_end_s']:.2f}s]")
    print(f"\n📊 Clip score: {clip_score:.3f}  |  Positive: {pos_count}/{len(segment_results)}  |  Label: {clip_label}")
    if show_plot:
        t = [0.5*(s["t_start_s"]+s["t_end_s"]) for s in segment_results]
        fused = [s["probability"] for s in segment_results]
        cnn   = [s["cnn_probability"] for s in segment_results]
        heur  = [s["heuristic_probability"] for s in segment_results]
        cols  = [PLOT_STYLE["ok"] if s["detected_at_external_threshold"] else PLOT_STYLE["err"] for s in segment_results]
        fig, ax = plt.subplots(figsize=(12,5))
        _apply_dark_style(fig, [ax])
        width = max((t[1]-t[0])*0.8 if len(t)>1 else 0.5, 0.15)
        ax.bar(t, fused, width=width, color=cols, alpha=0.55, label="Hybrid prob")
        ax.plot(t, cnn,  "-o", color=PLOT_STYLE["accent"], label="CNN prob")
        ax.plot(t, heur, "--s", color=PLOT_STYLE["purple"], label="Heuristic prob")
        ax.axhline(threshold, color=PLOT_STYLE["warn"], ls="--", lw=1.5, label=f"Ext thr={threshold:.2f}")
        ax.axhline(cfg.DETECTION_THRESHOLD, color=PLOT_STYLE["err"], ls=":", lw=1.5, label=f"Main thr={cfg.DETECTION_THRESHOLD:.2f}")
        ax.set_title(f"Robust External Detection — {Path(audio_path).name}", color=PLOT_STYLE["accent"])
        ax.set_xlabel("Time (s)"); ax.set_ylabel("Probability"); ax.set_ylim(0,1.05)
        ax.legend(facecolor=PLOT_STYLE["panel"]); plt.tight_layout(); plt.show()
    return {"file": str(audio_path), "duration_s": float(total_s), "segment_results": segment_results,
            "clip_score": float(clip_score), "positive_segments": int(pos_count),
            "segments_total": int(len(segment_results)), "clip_detected": bool(clip_detected),
            "clip_label": clip_label, "aggregation_mode": agg_mode, "external_threshold": float(threshold)}

# ══════════════════════════════════════════════════════════════════════════════
# CELL 12 – Main training entry points
# ══════════════════════════════════════════════════════════════════════════════

def audit_localization_labels(cfg: Config = None):
    cfg = cfg or config
    proc = cfg.PROCESSED_DIR / "localization"; all_labels = []
    for split in ["train","val","test"]:
        for lf in (proc/split).glob("*_label.json"):
            d = json.loads(lf.read_text())
            all_labels.append((d["azimuth_deg"], d["distance_m"], d["height_m"]))
    if not all_labels: print("No labels found."); return
    az, dist, ht = zip(*all_labels)
    print(f"  Sessions : {len(all_labels)}")
    print(f"  Azimuth  : min={min(az):.1f}°  max={max(az):.1f}°  mean={np.mean(az):.1f}°  std={np.std(az):.1f}°")
    print(f"  Distance : min={min(dist):.2f}m  max={max(dist):.2f}m  mean={np.mean(dist):.2f}m")
    print(f"  Height   : min={min(ht):.2f}m  max={max(ht):.2f}m  mean={np.mean(ht):.2f}m")
    round_dist = sum(1 for d in dist if d == round(d, 0))
    if round_dist > len(all_labels)*0.3:
        print(f"  ⚠️  {round_dist}/{len(all_labels)} sessions have integer distance — possible placeholders")


def train_detection(cfg: Config = None, epochs: int = None, resume: bool = True,
                    force_rebuild_cache: bool = False, force_regen_mixed_audio: bool = False):
    cfg = cfg or config; cfg.ensure_dirs(); _set_seed(cfg.SEED)
    print("="*70); print("  STAGE 1 — Detection Model (real + scraped + mixed + synthetic)"); print("="*70)
    DroneAudioDatasetManager(cfg).prepare()
    AudioWebScraper(cfg).download(force=False)
    _incorporate_scraped_audio(cfg, force=False)
    generate_mixed_drone_training_audio(cfg, force=force_regen_mixed_audio)
    report_detection_split_counts(cfg)
    mcm = MelCacheManager(cfg); mcm.build(force=force_rebuild_cache)
    inject_synthetic_det_data(cfg, force=False)
    counts = mcm.count()
    print("\n📊 MEL CACHE CLASS BALANCE")
    for key, n in counts.items(): print(f"  {key:28s}  {n:6d}")
    print()
    try: tr_l, va_l, te_l = get_det_dataloaders(cfg)
    except RuntimeError as e: print(f"❌ Could not build dataloaders: {e}"); return
    tr = DetectionTrainer(cfg); tr._set_loaders(tr_l, va_l, te_l); tr.run(epochs=epochs, resume=resume)


def train_localization(cfg: Config = None, epochs: int = None,
                       use_synthetic_fallback: bool = True, resume: bool = True, reset_best: bool = False):
    cfg = cfg or config; cfg.ensure_dirs(); _set_seed(cfg.SEED)
    print("="*65); print("  STAGE 2 — Localization Model (UaVirBASE)"); print("="*65)
    um = UaVirBASEDatasetManager(cfg)
    try: um.prepare()
    except Exception as e: print(f"⚠️  UaVirBASE prepare failed: {e}")
    if reset_best:
        ckpt = cfg.DRIVE_MODELS / "best_localization.pth"
        if ckpt.exists():
            ck = torch.load(ckpt, map_location=cfg.DEVICE); ck["best_val_loss"] = 1e9
            torch.save(ck, ckpt); print("🔄 best_val_loss reset to 1e9")
    tr = LocalizationTrainer(cfg)
    tr.run(cfg.PROCESSED_DIR/"localization", epochs=epochs,
           use_synthetic_fallback=use_synthetic_fallback, resume=resume)


def train_all(cfg: Config = None, det_epochs: int = 5, loc_epochs: int = 5,
              use_synthetic_loc: bool = True, resume: bool = True, force_rebuild_cache: bool = False):
    """Run both training stages. Set resume=False to start from scratch."""
    train_detection(cfg, det_epochs, resume=resume,
                    force_rebuild_cache=force_rebuild_cache, force_regen_mixed_audio=False)
    train_localization(cfg, loc_epochs, use_synthetic_loc, resume=resume)
    print("\n✅ Both models trained.")

# ══════════════════════════════════════════════════════════════════════════════
# CELL 13 – Verification, demo & diagnostics
# ══════════════════════════════════════════════════════════════════════════════

def verify_tdoa_accuracy(cfg: Config = None):
    cfg = cfg or config; mics = cfg.MIC_POSITIONS; c = cfg.SPEED_OF_SOUND; sr = cfg.SR
    positions = [[1.0,0.8],[2.0,0.5],[-1.5,2.0],[0.3,0.2]]
    print("="*60); print("  TDOA ACCURACY VERIFICATION"); print("="*60)
    print(f"  {'Pos':15s}  {'Exp τ12 (ms)':>13s}  {'Meas τ12 (ms)':>14s}  {'Err':>8s}  Status")
    print("  "+"-"*57); all_ok = True
    ds = max(1, sr//4000); fs_ds = sr//ds; hi_ds = min(5000, fs_ds//2-100)
    max_tau = np.max([np.linalg.norm(mics[i]-mics[j]) for i in range(3) for j in range(i+1,3)]) / c * 1.5
    for pos in positions:
        src = np.array(pos); dists = np.linalg.norm(mics-src[None,:],axis=1); exp12 = (dists[1]-dists[0])/c
        chs = synthesise_drone(mics, pos, fundamental=100, noise_level=0.005)
        ap  = AudioProcessor(cfg)
        y1  = bandpass(ap.pad_or_truncate(chs[0]), sr, 80, hi_ds)[::ds]
        y2  = bandpass(ap.pad_or_truncate(chs[1]), sr, 80, hi_ds)[::ds]
        meas, _, _ = gcc_phat(y2, y1, fs=fs_ds, max_tau=max_tau, interp=16)
        err = abs(meas-exp12)*1000; ok = err < 0.1
        if not ok: all_ok = False
        print(f"  {str(pos):15s}  {exp12*1000:+12.4f}ms  {meas*1000:+13.4f}ms  {err:7.4f}ms  {'✅' if ok else '❌'}")
    print(); print("  ✅ All accurate." if all_ok else "  ❌ Some failed — check _fractional_delay()")
    print("="*60); return all_ok


def diagnose_uavirbase(cfg: Config = None, n_probe: int = 5):
    cfg = cfg or config; url = cfg.UAVIRBASE_ZIP_URL
    if url is None: print("❌ UAVIRBASE_ZIP_URL is not set."); return
    _ensure_remotezip()
    from remotezip import RemoteZip
    print(f"🔍 Diagnosing UaVirBASE archive …\n   URL: {url}\n")
    print("   Reading central directory …")
    with RemoteZip(url, initial_buffer_size=5*1024*1024) as rz: all_names = rz.namelist()
    print(f"   Total entries: {len(all_names)}")
    top_dirs = sorted(set("/".join(n.split("/")[:2]) for n in all_names if n.count("/")>=1))[:10]
    print("\n   Sample top-level folders:")
    for d in top_dirs: print(f"     {d}")
    audio_entries = [n for n in all_names if n.endswith("/output.wav") or n.endswith("/audio.wav")]
    label_entries = [n for n in all_names if n.endswith("/label.json") or n.endswith("/annotation.json")]
    print(f"\n   Audio files : {len(audio_entries)}  (e.g. {audio_entries[0] if audio_entries else 'none'})")
    print(f"   Label files : {len(label_entries)}  (e.g. {label_entries[0] if label_entries else 'none'})")
    if not label_entries:
        json_entries = [n for n in all_names if n.endswith(".json")][:10]
        print("\n   ⚠️  No label files found. All .json entries:")
        for e in json_entries: print(f"     {e}")
        return
    print(f"\n   Probing first {min(n_probe,len(label_entries))} label files …")
    with RemoteZip(url, initial_buffer_size=5*1024*1024) as rz:
        for entry in label_entries[:n_probe]:
            raw = rz.read(entry); schema = _probe_label_schema(raw); parsed = parse_label_json(raw)
            status = (f"✅ az={parsed[0]:.1f}°  dist={parsed[1]:.2f}m  ht={parsed[2]:.2f}m" if parsed else "❌ PARSE FAILED")
            print(f"\n   {entry}\n     Schema : {schema}\n     Result : {status}")
            if not parsed: print(f"     Raw    : {raw[:300]}")
    print("\n   ✅ Diagnosis complete.")


def quick_demo(cfg: Config = None):
    cfg = cfg or config
    print("\n🚁 QUICK DEMO — Multi-drone synthetic test"); print("─"*50)
    positions = [[1.5,0.4],[-1.0,1.8]]; funds = [100,280]
    chs_all = [np.zeros(int(cfg.SR*cfg.TARGET_DURATION)) for _ in range(3)]
    for pos, fund in zip(positions, funds):
        chs = synthesise_drone(cfg.MIC_POSITIONS, pos, fundamental=fund, noise_level=0.03)
        for i, ch in enumerate(chs): chs_all[i] = np.clip(chs_all[i]+ch, -1, 1).astype(np.float32)
    mx = max(np.abs(chs_all[i]).max() for i in range(3)) + 1e-8
    chs_all = [(c/mx).astype(np.float32) for c in chs_all]
    tmp_paths = []
    for ch in chs_all:
        tf = tempfile.NamedTemporaryFile(delete=False, suffix=".wav")
        sf.write(tf.name, ch, cfg.SR); tmp_paths.append(tf.name)
    tracker = KalmanTracker(cfg)
    result  = run_pipeline(tmp_paths, cfg, tracker=tracker, multi_drone=True)
    for p in tmp_paths: os.unlink(p)
    print(f"  Detected: {result['detected']}  (conf={result['probability']:.3f})")
    print(f"  Drones found: {len(result['drones'])}")
    for i, d in enumerate(result["drones"]):
        xy = d["xy_position"]; true = positions[i] if i < len(positions) else None
        err = f"  err={np.linalg.norm(xy-np.array(true)):.3f}m" if true else ""
        print(f"    Drone {i+1}: ({xy[0]:.2f},{xy[1]:.2f})m  az={d['azimuth_deg']:.1f}°  dist={d['distance_m']:.2f}m{err}")
    if result["drones"]: plot_multi_drone_positions(result["drones"], cfg)
    return result

# ══════════════════════════════════════════════════════════════════════════════
# CELL 14 – Colab UI  (v13 — updated header)
# ══════════════════════════════════════════════════════════════════════════════

def launch_ui(cfg: Config = None):
    if "google.colab" not in sys.modules:
        print("ℹ️  launch_ui() requires Google Colab."); return
    import ipywidgets as w
    from IPython.display import display, clear_output, Audio, HTML
    cfg = cfg or config
    AUDIO_EXTS = (".wav",".mp3",".ogg",".flac",".m4a")

    def _card(ok, title, rows):
        col = "#10b981" if ok else "#ef4444"; ic = "✅" if ok else "❌"
        trs = "".join(f"<tr><td style='color:#94a3b8;padding:2px 8px'>{k}</td>"
                      f"<td style='color:#e2e8f0;padding:2px 8px'>{v}</td></tr>" for k, v in rows)
        return HTML(f"<div style='background:#0f172a;border-left:3px solid {col};border-radius:8px;"
                    f"padding:12px 16px;margin:8px 0;font-family:monospace;font-size:12px'>"
                    f"<b style='color:{col}'>{ic} {title}</b>"
                    f"<table style='margin-top:8px;border-collapse:collapse'>{trs}</table></div>")

    state = {}

    # ── Tab 1: Single File ────────────────────────────────────────────────
    t1_up  = w.Button(description="📁 Upload Audio", button_style="primary", layout=w.Layout(width="190px"))
    t1_thr = w.FloatSlider(value=0.70, min=0.10, max=0.99, step=0.01, description="Threshold:",
                           layout=w.Layout(width="400px"), style={"description_width":"90px"}, continuous_update=False)
    t1_run = w.Button(description="🚁 Run", button_style="success", layout=w.Layout(width="120px"), disabled=True)
    t1_lbl = w.Label("Step 1: upload a file"); t1_out_audio = w.Output(); t1_out = w.Output()
    def _t1_up(_):
        from google.colab import files; up = files.upload()
        af = [p for p in up if p.lower().endswith(AUDIO_EXTS)]
        if not af: t1_lbl.value = "❌ No audio"; return
        p = af[0]; state["t1_audio"] = p; state["t1_path"] = p
        with t1_out_audio: clear_output(); display(Audio(p, autoplay=False))
        t1_run.disabled = False; t1_lbl.value = f"Ready: {p}"
    def _t1_run(_):
        if "t1_path" not in state: return
        cfg.DETECTION_THRESHOLD = t1_thr.value
        with t1_out:
            clear_output(wait=True)
            try:
                result = analyse_audio_file(state["t1_path"], cfg, show_plot=True, use_synthesis=False)
                rows = [("File", state["t1_path"]),
                        ("Prob", f"{result['probability']:.3f}"),
                        ("Detected", str(result['detected'])),
                        ("Duration", f"{result['duration_sec']:.2f}s")]
                display(_card(result["detected"], "Analysis Result", rows))
            except Exception as e:
                import traceback; traceback.print_exc()
                display(_card(False, f"Error: {e}", []))
    t1_up.on_click(_t1_up); t1_run.on_click(_t1_run)
    tab1 = w.VBox([w.HTML("<b style='color:#00d4ff;font-family:monospace'>Single-File Detection + Analysis</b>"),
                   w.HBox([t1_up, t1_run]), t1_thr, t1_lbl, t1_out_audio, t1_out])

    # ── Tab 2: 3-Mic Real ─────────────────────────────────────────────────
    t2_up  = w.Button(description="📁 Upload 3 Mic Files", button_style="primary", layout=w.Layout(width="210px"))
    t2_thr = w.FloatSlider(value=0.70, min=0.10, max=0.99, step=0.01, description="Threshold:",
                           layout=w.Layout(width="380px"), style={"description_width":"90px"}, continuous_update=False)
    t2_run = w.Button(description="📡 Localize", button_style="success", layout=w.Layout(width="130px"), disabled=True)
    t2_lbl = w.Label("Upload 3 files (sorted = Mic1, Mic2, Mic3)"); t2_out = w.Output()
    def _t2_up(_):
        from google.colab import files; up = files.upload()
        af = sorted([p for p in up if p.lower().endswith(AUDIO_EXTS)])[:3]
        if len(af)<3: t2_lbl.value = f"❌ Need 3, got {len(af)}"; return
        state["t2_paths"] = af; t2_run.disabled = False; t2_lbl.value = str([Path(p).name for p in af])
    def _t2_run(_):
        if "t2_paths" not in state: return
        cfg.DETECTION_THRESHOLD = t2_thr.value; chs = load_3ch(state["t2_paths"], cfg)
        with t2_out:
            clear_output(wait=True)
            try:
                det = detect(chs, cfg)
                rows = [("Detected", str(det["detected"])), ("Prob", f"{det['probability']:.3f}"),
                        ("CNN", f"{det['cnn_probability']:.3f}"), ("Heuristic", f"{det['heuristic_probability']:.3f}")]
                if det["detected"]:
                    loc = localize(chs, cfg)
                    rows += [("Azimuth", f"{loc['azimuth_deg']:.1f}°"),
                             ("Distance", f"{loc['distance_m']:.3f} m"),
                             ("XY", f"({loc['xy_position'][0]:.3f}, {loc['xy_position'][1]:.3f}) m")]
                    plot_polar_azimuth([loc["azimuth_deg"]], title="Detected Azimuth", cfg=cfg)
                display(_card(det["detected"], "3-Mic Real Result", rows))
            except Exception as e: display(_card(False, f"Error: {e}", []))
    t2_up.on_click(_t2_up); t2_run.on_click(_t2_run)
    tab2 = w.VBox([w.HTML("<b style='color:#00d4ff;font-family:monospace'>3-Mic Real Recording Localization</b>"),
                   w.HBox([t2_up, t2_run]), t2_thr, t2_lbl, t2_out])

    # ── Tab 3: Multi-Drone ────────────────────────────────────────────────
    t3_run = w.Button(description="🚁🚁 Run Synthetic Multi-Drone", button_style="success", layout=w.Layout(width="280px"))
    t3_max = w.IntSlider(value=2, min=1, max=3, step=1, description="Max drones:",
                         layout=w.Layout(width="300px"), style={"description_width":"100px"}, continuous_update=False)
    t3_out = w.Output()
    def _t3_run(_):
        with t3_out:
            clear_output(wait=True)
            try:
                true_pos = [[1.5,0.4],[-1.0,1.8]]
                chs_mix  = [np.zeros(int(cfg.SR*cfg.TARGET_DURATION)) for _ in range(3)]
                for pos, fund in zip(true_pos, [100,280]):
                    for i, ch in enumerate(synthesise_drone(cfg.MIC_POSITIONS, pos, fundamental=fund, noise_level=0.03)):
                        chs_mix[i] = np.clip(chs_mix[i]+ch, -1, 1).astype(np.float32)
                mx = max(np.abs(chs_mix[i]).max() for i in range(3)) + 1e-8
                chs_mix = [(c/mx).astype(np.float32) for c in chs_mix]
                drones = localize_multi_drone(chs_mix, cfg, t3_max.value)
                rows = [("True positions", str(true_pos)), ("Drones found", str(len(drones)))]
                for i, d in enumerate(drones):
                    xy = d["xy_position"]
                    rows.append((f"Drone #{i+1}", f"({xy[0]:.3f},{xy[1]:.3f})m  az={d['azimuth_deg']:.1f}°  ±{d.get('confidence_radius',0):.3f}m"))
                if drones: plot_multi_drone_positions(drones, cfg)
                display(_card(len(drones)>0, f"Multi-Drone: {len(drones)} found", rows))
            except Exception as e: display(_card(False, f"Error: {e}", []))
    t3_run.on_click(_t3_run)
    tab3 = w.VBox([w.HTML("<b style='color:#00d4ff;font-family:monospace'>Multi-Drone Detection & Localization</b>"),
                   t3_max, t3_run, t3_out])

    # ── Tab 4: Kalman Track ───────────────────────────────────────────────
    t4_n   = w.IntSlider(value=8, min=4, max=20, step=1, description="Waypoints:",
                         layout=w.Layout(width="360px"), style={"description_width":"100px"}, continuous_update=False)
    t4_sp  = w.FloatSlider(value=1.5, min=0.3, max=3.0, step=0.1, description="Spread (m):",
                           layout=w.Layout(width="360px"), style={"description_width":"100px"}, continuous_update=False)
    t4_run = w.Button(description="🛤️ Run Path Tracking", button_style="success", layout=w.Layout(width="200px"))
    t4_out = w.Output()
    def _t4_run(_):
        with t4_out:
            clear_output(wait=True)
            try:
                KalmanTrack._id_counter = 0; tracker = KalmanTracker(cfg)
                n = t4_n.value; spread = t4_sp.value
                angles = np.linspace(0, 2*np.pi*1.5, n); radii = np.linspace(0.3, spread, n)
                cx, cy = cfg.ARRAY_CENTER
                wpts = [(cx+r*np.cos(a), cy+r*np.sin(a)) for r,a in zip(radii, angles)]
                base_ts = time.time()
                for i, wp in enumerate(wpts):
                    chs = synthesise_drone(cfg.MIC_POSITIONS, wp, fundamental=random.choice([90,100,110]))
                    chs = [AudioProcessor(cfg).pad_or_truncate(c) for c in chs]
                    det = detect(chs, cfg)
                    if det["detected"]: loc = localize(chs, cfg); tracker.step([loc["xy_position"]], base_ts+i)
                confirmed = tracker.all_confirmed()
                rows = [("Waypoints", str(n)), ("Confirmed tracks", str(len(confirmed)))]
                for t in confirmed:
                    rows.append((f"Track #{t.track_id}", f"{len(t.positions)} pts  dist={t.total_distance():.2f}m  σ=±{t.uncertainty_radius():.3f}m"))
                if confirmed: plot_track_trajectory(confirmed, cfg)
                display(_card(len(confirmed)>0, "Path Tracking Result", rows))
            except Exception as e:
                import traceback; traceback.print_exc(); display(_card(False, f"Error: {e}", []))
    t4_run.on_click(_t4_run)
    tab4 = w.VBox([w.HTML("<b style='color:#00d4ff;font-family:monospace'>Kalman Path Tracking</b>"),
                   t4_n, t4_sp, t4_run, t4_out])

    # ── Tab 5: Training Plots ─────────────────────────────────────────────
    t5_run = w.Button(description="📈 Show Training Curves", button_style="info", layout=w.Layout(width="220px"))
    t5_sct = w.Button(description="🎯 Localization Scatter", button_style="info", layout=w.Layout(width="220px"))
    t5_out = w.Output()
    def _t5_run(_):
        with t5_out: clear_output(wait=True); plot_training_logs(cfg)
    def _t5_sct(_):
        with t5_out: clear_output(wait=True); plot_localization_scatter(cfg)
    t5_run.on_click(_t5_run); t5_sct.on_click(_t5_sct)
    tab5 = w.VBox([w.HTML("<b style='color:#00d4ff;font-family:monospace'>Training & Evaluation Plots</b>"),
                   w.HBox([t5_run, t5_sct]), t5_out])

    tabs = w.Tab(children=[tab1, tab2, tab3, tab4, tab5])
    for i, ttl in enumerate(["🎵 Single File","📡 3-Mic Real","🚁🚁 Multi-Drone","🛤️ Kalman Track","📈 Plots"]):
        tabs.set_title(i, ttl)

    header = w.HTML("""
<div style='background:linear-gradient(135deg,#0f172a,#1e3a5f);border-radius:10px;
            padding:16px 20px;margin-bottom:12px;font-family:monospace'>
  <div style='font-size:18px;font-weight:bold;color:#00d4ff'>
    🚁 Drone Detection v13 — Enhanced Localization &amp; Visualization
  </div>
  <div style='color:#64748b;font-size:11px;margin-top:4px'>
    ① Multi-source audio scraping (BBC, xeno-canto, SoundBible, FreeSound.io, yt-dlp)<br>
    ② Resource-aware localization: LocalizationCNNLite (Lite) or LocalizationCNN (Full)<br>
    ③ 6-panel dark dashboard · polar compass · track trajectory · score gauge
  </div>
</div>""")
    display(w.VBox([header, tabs]))



# ══════════════════════════════════════════════════════════════════════════════
# v14 GENERALIZATION PATCHES
# ══════════════════════════════════════════════════════════════════════════════

# These overrides focus on better real-world generalization for detection:
#   • richer 3-channel features: [log-mel, PCEN, delta-mel]
#   • stronger waveform and spectrogram domain randomization
#   • grouped split helpers to reduce leakage when used in future data prep
#   • focal loss + threshold search on validation
#   • richer evaluation metrics stored in checkpoints


def safe_standardize(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return ((x - x.mean()) / (x.std() + eps)).astype(np.float32)


def compute_delta_2d(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float32)
    return np.diff(x, axis=1, prepend=x[:, :1]).astype(np.float32)


def random_eq_tilt(y: np.ndarray, sr: int, strength: float = 0.35) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    spec = np.fft.rfft(y)
    freqs = np.fft.rfftfreq(len(y), d=1.0 / sr)
    freqs[0] = 1.0
    tilt = random.uniform(-strength, strength)
    mag = np.power(np.maximum(freqs / 1000.0, 1e-3), tilt)
    out = np.fft.irfft(spec * mag, n=len(y))
    return np.clip(out, -1.0, 1.0).astype(np.float32)


def random_bandlimit(y: np.ndarray, sr: int) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    lo = random.uniform(20, 250)
    hi = random.uniform(2500, min(8000, sr / 2 - 200))
    if hi <= lo + 300:
        return y
    sos = scipy.signal.butter(4, [lo / (sr / 2), hi / (sr / 2)], btype="band", output="sos")
    out = scipy.signal.sosfilt(sos, y)
    return np.clip(out, -1.0, 1.0).astype(np.float32)


def random_reverb(y: np.ndarray, sr: int) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    ir_len = int(random.uniform(0.03, 0.20) * sr)
    if ir_len < 8:
        return y
    t = np.linspace(0, 1, ir_len, endpoint=False)
    decay = np.exp(-t * random.uniform(8, 25))
    noise = np.random.randn(ir_len).astype(np.float32)
    ir = decay * noise
    ir[0] += 1.0
    ir /= np.max(np.abs(ir)) + 1e-8
    out = scipy.signal.fftconvolve(y, ir, mode="full")[:len(y)]
    return normalize_peak(out).astype(np.float32)


def random_codec_like(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    step = random.choice([64, 128, 256, 512])
    return np.clip(np.round(y * step) / step, -1.0, 1.0).astype(np.float32)


def random_dropout_chunks(y: np.ndarray, max_chunks: int = 3) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32).copy()
    n = len(y)
    for _ in range(random.randint(0, max_chunks)):
        w = random.randint(max(8, n // 100), max(16, n // 20))
        s = random.randint(0, max(0, n - w))
        y[s:s + w] *= random.uniform(0.0, 0.25)
    return y.astype(np.float32)


def augment_waveform(y: np.ndarray, cfg: Config) -> np.ndarray:
    y = np.asarray(y, dtype=np.float32)
    if random.random() < 0.90:
        y = np.clip(y * db_to_gain(random.uniform(-10, 10)), -1, 1).astype(np.float32)
    if random.random() < 0.60:
        y = AudioProcessor(cfg).add_noise(y, random.uniform(0, 25))
    if random.random() < 0.35:
        y = random_eq_tilt(y, cfg.SR, 0.45)
    if random.random() < 0.35:
        y = random_bandlimit(y, cfg.SR)
    if random.random() < 0.25:
        y = random_reverb(y, cfg.SR)
    if random.random() < 0.25:
        y = random_codec_like(y)
    if random.random() < 0.30:
        y = random_dropout_chunks(y)
    if random.random() < 0.50:
        y = np.roll(y, random.randint(0, len(y) // 4)).astype(np.float32)
    if random.random() < 0.20:
        clip_val = random.uniform(0.35, 0.9)
        y = np.clip(y, -clip_val, clip_val) / clip_val
    return np.clip(y, -1, 1).astype(np.float32)


def infer_group_id(path: Path) -> str:
    stem = path.stem.lower()
    patterns = [
        r"(session[_\-]?\d+)", r"(flight[_\-]?\d+)", r"(take[_\-]?\d+)",
        r"(clip[_\-]?\d+)", r"(yt[_\-]?[a-z0-9]+)", r"(fs[_\-]?\d+)",
        r"(bbc[_\-]?[a-z0-9]+)", r"(xc[_\-]?\d+)", r"(sb[_\-]?\d+)"
    ]
    for p in patterns:
        m = re.search(p, stem)
        if m:
            return m.group(1)
    parts = re.split(r"[_\-]", stem)
    return "_".join(parts[:2]) if len(parts) >= 2 else stem


def grouped_split_paths(files: List[Path], seed: int = 42) -> Dict[str, List[Path]]:
    rng = random.Random(seed)
    group_to_files: Dict[str, List[Path]] = {}
    for f in files:
        gid = infer_group_id(f)
        group_to_files.setdefault(gid, []).append(f)
    groups = list(group_to_files.items())
    rng.shuffle(groups)
    n = len(groups)
    n_tr = int(0.70 * n)
    n_val = int(0.15 * n)
    split_map = {"train": [], "val": [], "test": []}
    for i, (_, flist) in enumerate(groups):
        if i < n_tr:
            split_map["train"].extend(flist)
        elif i < n_tr + n_val:
            split_map["val"].extend(flist)
        else:
            split_map["test"].extend(flist)
    return split_map


class AudioProcessor(AudioProcessor):
    def mel_power(self, y):
        return librosa.feature.melspectrogram(
            y=y, sr=self.cfg.SR, n_fft=self.cfg.N_FFT,
            hop_length=self.cfg.HOP_LENGTH, n_mels=self.cfg.N_MELS,
            fmin=20, fmax=8000).astype(np.float32)

    def mel(self, y):
        M = self.mel_power(y)
        return librosa.power_to_db(M, ref=np.max).astype(np.float32)

    def pcen(self, y):
        M = self.mel_power(y)
        P = librosa.pcen(
            M, sr=self.cfg.SR, hop_length=self.cfg.HOP_LENGTH,
            gain=0.8, bias=10.0, power=0.25, time_constant=0.4, eps=1e-6)
        return P.astype(np.float32)

    def feature_stack(self, y):
        m = safe_standardize(self.mel(y))
        p = safe_standardize(self.pcen(y))
        d = safe_standardize(compute_delta_2d(m))
        return np.stack([m, p, d], axis=0).astype(np.float32)


class MelCacheManager(MelCacheManager):
    def build(self, force: bool = False):
        cache_root = self.cfg.MEL_CACHE_DIR
        n_existing = len(list(cache_root.rglob("*.npy")))
        if not force and n_existing > 100:
            print(f"✅ Mel cache already exists ({n_existing} files) — skipping.")
            return
        if force and cache_root.exists():
            shutil.rmtree(str(cache_root))
        print("🎵 Building mel cache from processed WAVs [v14 features] …")
        det_root = self.cfg.PROCESSED_DIR / "detection"
        wavs = []
        for split in ["train", "val", "test"]:
            for label in ["drone", "non_drone"]:
                src = det_root / split / label
                if src.exists():
                    for wav in src.glob("*.wav"):
                        wavs.append((split, label, wav))
        total = 0
        for split, label, wav in tqdm(wavs, desc="Mel cache"):
            dst = cache_root / split / label
            dst.mkdir(parents=True, exist_ok=True)
            out = dst / f"{wav.stem}.npy"
            if out.exists() and not force:
                continue
            try:
                y = self.ap.pad_or_truncate(self.ap.load(wav))
                np.save(str(out), self.ap.feature_stack(y))
                total += 1
            except Exception as e:
                print(f"   ⚠️  {wav.name}: {e}")
        print(f"✅ Mel cache built ({total} new files).")

    def build_detection_cache(self, force=False):
        det = self.cfg.PROCESSED_DIR / "detection"
        cache = self.cfg.MEL_CACHE_DIR / "detection"
        for split in ["train", "val", "test"]:
            for label in ["drone", "non_drone"]:
                src = det / split / label
                dst = cache / split / label
                if not src.exists():
                    continue
                dst.mkdir(parents=True, exist_ok=True)
                wavs = list(src.glob("*.wav"))
                print(f"  Caching {split}/{label}: {len(wavs)} files …")
                for wav in tqdm(wavs, desc=f"{split}/{label}", leave=False):
                    out = dst / (wav.stem + ".npy")
                    if out.exists() and not force:
                        continue
                    try:
                        y = self.ap.pad_or_truncate(self.ap.load(wav))
                        np.save(str(out), self.ap.feature_stack(y))
                    except Exception as e:
                        print(f"    ⚠️  {wav.name}: {e}")
        self._inject_synthetic(cache, force=force)
        print("✅ Detection mel cache built.")

    def _inject_synthetic(self, cache, force=False):
        n = self.cfg.SYNTHETIC_DET_SAMPLES
        if n <= 0:
            return
        out = cache / "train" / "drone"
        out.mkdir(parents=True, exist_ok=True)
        if len(list(out.glob("synth_det_*.npy"))) >= n and not force:
            print("  ✅ Synthetic injection already done.")
            return
        print(f"  🔬 Injecting {n} synthetic feature tensors …")
        rng = np.random.default_rng(self.cfg.SEED)
        cx, cy = self.cfg.ARRAY_CENTER
        ap = AudioProcessor(self.cfg)
        for i in tqdm(range(n), desc="SynthInject", leave=False):
            r = rng.uniform(0.5, self.cfg.MAX_LOCALIZATION_DIST)
            theta = rng.uniform(0, 2 * np.pi)
            xy = [cx + r * np.cos(theta), cy + r * np.sin(theta)]
            fund = int(rng.choice([80, 90, 100, 110, 120, 130]))
            chs = synthesise_drone(self.cfg.MIC_POSITIONS, xy, fundamental=fund,
                                   noise_level=float(rng.uniform(0.01, 0.08)))
            y = ap.pad_or_truncate(chs[0])
            np.save(str(out / f"synth_det_{i:06d}.npy"), ap.feature_stack(y))


def inject_synthetic_det_data(cfg: Config, force: bool = False):
    cache_dir = cfg.MEL_CACHE_DIR / "train" / "drone"
    cache_dir.mkdir(parents=True, exist_ok=True)
    n_samples = cfg.SYNTHETIC_DET_SAMPLES
    if n_samples <= 0:
        return
    existing = len(list(cache_dir.glob("synth_det_*.npy")))
    if existing >= n_samples and not force:
        print(f"✅ Synthetic detection injection already done ({existing} files).")
        return
    ap = AudioProcessor(cfg)
    rng = np.random.default_rng(cfg.SEED + 1)
    r = rng.uniform(0.3, cfg.MAX_LOCALIZATION_DIST, n_samples)
    theta = rng.uniform(0, 2 * np.pi, n_samples)
    funds = rng.choice([80, 90, 100, 110, 120, 130], n_samples)
    noises = rng.uniform(0.02, 0.10, n_samples)
    cx, cy = cfg.ARRAY_CENTER
    positions = np.stack([cx + r * np.cos(theta), cy + r * np.sin(theta)], axis=1)
    print(f"🔬 Injecting {n_samples} synthetic drone feature tensors into detection cache …")
    for i in tqdm(range(n_samples)):
        out_path = cache_dir / f"synth_det_{i:06d}.npy"
        if out_path.exists() and not force:
            continue
        chs = synthesise_drone(cfg.MIC_POSITIONS, positions[i], fundamental=int(funds[i]), noise_level=float(noises[i]))
        mono = np.mean(np.stack(chs, axis=0), axis=0)
        np.save(str(out_path), ap.feature_stack(ap.pad_or_truncate(mono)))
    print("✅ Synthetic injection done.")


class MelCachedDataset(Dataset):
    def __init__(self, cache_root, split, augment=False):
        self.augment = augment
        self.files = []
        self.labels = []
        for idx, cls in enumerate(["non_drone", "drone"]):
            d = cache_root / split / cls
            if d.exists():
                for f in d.glob("*.npy"):
                    self.files.append(f)
                    self.labels.append(idx)
        if not self.files:
            raise RuntimeError(f"No cached mels in {cache_root}/{split}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        x = np.load(str(self.files[idx])).astype(np.float32)
        if self.augment:
            if random.random() < 0.5:
                x *= random.uniform(0.85, 1.15)
            if random.random() < 0.35:
                t = random.randint(2, max(2, x.shape[-1] // 10))
                s = random.randint(0, max(0, x.shape[-1] - t))
                x[:, :, s:s + t] *= random.uniform(0.0, 0.2)
            if random.random() < 0.25:
                f = random.randint(2, max(2, x.shape[-2] // 8))
                s = random.randint(0, max(0, x.shape[-2] - f))
                x[:, s:s + f, :] *= random.uniform(0.0, 0.2)
            if random.random() < 0.30:
                x += np.random.randn(*x.shape).astype(np.float32) * 0.03
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)


class DetectionDataset(Dataset):
    def __init__(self, root: Path, split: str, augment=False, cfg: Config = None):
        self.ap = AudioProcessor(cfg or config)
        self.cfg = cfg or config
        self.augment = augment
        self.files = []
        self.labels = []
        for idx, cls in enumerate(["non_drone", "drone"]):
            d = root / split / cls
            if d.exists():
                for f in d.glob("*.wav"):
                    self.files.append(f)
                    self.labels.append(idx)
        if not self.files:
            raise RuntimeError(f"No files in {root}/{split}")

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        y = self.ap.pad_or_truncate(self.ap.load(self.files[idx]))
        if self.augment:
            y = augment_waveform(y, self.cfg)
        feat = self.ap.feature_stack(y)
        return torch.tensor(feat, dtype=torch.float32), torch.tensor(self.labels[idx], dtype=torch.long)


def perturb_multichannel(channels: List[np.ndarray], cfg: Config) -> List[np.ndarray]:
    out = []
    base_gain = random.uniform(-3.0, 3.0)
    ap = AudioProcessor(cfg)
    for ch in channels:
        x = np.asarray(ch, dtype=np.float32).copy()
        x *= db_to_gain(base_gain + random.uniform(-1.5, 1.5))
        if random.random() < 0.5:
            x = ap.add_noise(x, random.uniform(0, 20))
        if random.random() < 0.25:
            x = _fractional_delay(x, random.uniform(0.0, 2.0))
        if random.random() < 0.25:
            x = random_eq_tilt(x, cfg.SR, 0.25)
        out.append(np.clip(x, -1, 1).astype(np.float32))
    return out


class LocalizationDataset(LocalizationDataset):
    def __getitem__(self, idx):
        chs_paths, lf = self.sessions[idx]
        channels = [self.ap.pad_or_truncate(self.ap.load(p)) for p in chs_paths]
        if self.augment:
            channels = perturb_multichannel(channels, self.cfg)
        ipd_cache = lf.parent / (lf.stem.replace("_label", "") + "_ipd.npy")
        if (not self.augment) and ipd_cache.exists():
            ipd = np.load(str(ipd_cache))
        else:
            ipd = compute_ipd_features(channels, self.cfg)
            if not self.augment:
                try:
                    np.save(str(ipd_cache), ipd)
                except Exception:
                    pass
        mels = [self.ap.mel(c) for c in channels]
        mel_t = torch.tensor(np.stack(mels, axis=0), dtype=torch.float32)
        ipd_t = torch.tensor(ipd, dtype=torch.float32)
        label = json.loads(lf.read_text())
        az_deg = float(label["azimuth_deg"])
        di_m = float(label["distance_m"])
        ht_m = float(label["height_m"])
        if self.augment:
            az_deg = wrap_angle_deg(az_deg + random.gauss(0, 15.0))
            di_m = max(0.5, di_m + random.gauss(0, 1.5))
            ht_m = max(0.5, ht_m + random.gauss(0, 1.0))
        az_rad = math.radians(az_deg)
        max_dist = self.cfg.MAX_LOCALIZATION_DIST
        lbl_t = torch.tensor([
            math.sin(az_rad), math.cos(az_rad),
            np.clip(di_m / max_dist, 0, 1.5), np.clip(ht_m / max_dist, 0, 1.5)
        ], dtype=torch.float32)
        return mel_t, ipd_t, lbl_t


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none", label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        if self.alpha is not None:
            at = torch.where(targets == 1, self.alpha, 1 - self.alpha)
            focal = focal * at
        return focal.mean()


def collect_val_probs(model, loader, device):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy()
            ys.extend(y.numpy().tolist())
            ps.extend(probs.tolist())
    return np.asarray(ys), np.asarray(ps)


def find_best_threshold(y_true, y_prob, beta=1.0):
    best_t, best_score = 0.5, -1.0
    for t in np.linspace(0.05, 0.95, 181):
        pred = (y_prob >= t).astype(np.int64)
        tp = np.sum((pred == 1) & (y_true == 1))
        fp = np.sum((pred == 1) & (y_true == 0))
        fn = np.sum((pred == 0) & (y_true == 1))
        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)
        if precision == 0 and recall == 0:
            fbeta = 0.0
        else:
            fbeta = (1 + beta ** 2) * precision * recall / max(beta ** 2 * precision + recall, 1e-8)
        if fbeta > best_score:
            best_score = fbeta
            best_t = float(t)
    return best_t, best_score


def evaluate_binary_metrics(y_true, y_prob, threshold):
    pred = (y_prob >= threshold).astype(np.int64)
    tp = int(np.sum((pred == 1) & (y_true == 1)))
    fp = int(np.sum((pred == 1) & (y_true == 0)))
    tn = int(np.sum((pred == 0) & (y_true == 0)))
    fn = int(np.sum((pred == 0) & (y_true == 1)))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall / (precision + recall))
    acc = (tp + tn) / max(tp + tn + fp + fn, 1)
    try:
        auroc = float(__import__('sklearn.metrics').metrics.roc_auc_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float('nan')
    except Exception:
        auroc = float('nan')
    try:
        auprc = float(__import__('sklearn.metrics').metrics.average_precision_score(y_true, y_prob)) if len(np.unique(y_true)) > 1 else float('nan')
    except Exception:
        auprc = float('nan')
    return {
        "threshold": float(threshold), "accuracy": float(acc), "precision": float(precision),
        "recall": float(recall), "f1": float(f1), "auroc": auroc, "auprc": auprc,
        "tp": tp, "fp": fp, "tn": tn, "fn": fn,
    }


def print_detection_report(y_true, y_prob, threshold):
    metrics = evaluate_binary_metrics(y_true, y_prob, threshold)
    pred = (y_prob >= threshold).astype(np.int64)
    print("\n=== Detection validation report ===")
    print(f"Threshold : {metrics['threshold']:.3f}")
    print(f"Accuracy  : {metrics['accuracy']:.4f}")
    print(f"Precision : {metrics['precision']:.4f}")
    print(f"Recall    : {metrics['recall']:.4f}")
    print(f"F1        : {metrics['f1']:.4f}")
    print(f"AUROC     : {metrics['auroc']:.4f}")
    print(f"AUPRC     : {metrics['auprc']:.4f}")
    print(f"TP/FP/TN/FN = {metrics['tp']}/{metrics['fp']}/{metrics['tn']}/{metrics['fn']}")
    print(classification_report(y_true, pred, target_names=["non_drone", "drone"], digits=4))
    print(confusion_matrix(y_true, pred))


class DetectionTrainer:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.dev = torch.device(cfg.DEVICE)
        self.model = DetectionCNN().to(self.dev)
        self._loaders = None

    def _set_loaders(self, tr_l, va_l, te_l):
        self._loaders = (tr_l, va_l, te_l)

    def run(self, epochs: int = None, resume: bool = True):
        epochs = epochs or self.cfg.NUM_EPOCHS
        _set_seed(self.cfg.SEED)
        if self._loaders is not None:
            tr_l, va_l, te_l = self._loaders
        else:
            data_root = self.cfg.PROCESSED_DIR / "detection"
            try:
                tr = DetectionDataset(data_root, "train", augment=True, cfg=self.cfg)
                va = DetectionDataset(data_root, "val", augment=False, cfg=self.cfg)
                te = DetectionDataset(data_root, "test", augment=False, cfg=self.cfg)
            except RuntimeError as e:
                print(f"❌ {e}")
                return
            lbs = np.array(tr.labels)
            cnt = np.bincount(lbs)
            cnt[cnt == 0] = 1
            wts = (1.0 / cnt)[lbs]
            sampler = WeightedRandomSampler(wts, len(wts), replacement=True)
            def _collate(batch):
                xs, ys = zip(*batch)
                return torch.stack(xs), torch.stack(ys)
            tr_l = DataLoader(tr, batch_size=self.cfg.BATCH_SIZE, sampler=sampler, collate_fn=_collate)
            va_l = DataLoader(va, batch_size=self.cfg.BATCH_SIZE, shuffle=False, collate_fn=_collate)
            te_l = DataLoader(te, batch_size=self.cfg.BATCH_SIZE, shuffle=False, collate_fn=_collate)

        opt = torch.optim.AdamW(self.model.parameters(), lr=self.cfg.LR, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
        crit = FocalLoss(gamma=2.0, alpha=0.6, label_smoothing=0.02)
        scaler = torch.amp.GradScaler("cuda", enabled=self.cfg.USE_AMP and self.cfg.DEVICE == "cuda")
        logger = TrainingLogger(self.cfg.DRIVE_LOGS / "detection_log.csv",
                                columns=["epoch", "tr_loss", "tr_acc", "val_acc", "val_f1", "val_thr", "val_auprc"])
        logged_epochs = {int(r["epoch"]) for r in logger.rows}
        start_epoch = 1
        best_f1 = 0.0
        best_thr = self.cfg.DETECTION_THRESHOLD
        ckpt_path = self.cfg.DRIVE_MODELS / "best_detection.pth"
        latest_path = self.cfg.DRIVE_MODELS / "latest_detection.pth"
        if resume and latest_path.exists():
            ck = torch.load(latest_path, map_location=self.dev)
            self.model.load_state_dict(ck["model_state"])
            start_epoch = ck.get("epoch", 1) + 1
            best_f1 = float(ck.get("best_val_f1", 0.0))
            best_thr = float(ck.get("best_threshold", self.cfg.DETECTION_THRESHOLD))
            print(f"▶️  Resuming detection from epoch {start_epoch} (best f1: {best_f1:.4f}, thr: {best_thr:.3f})")
        elif resume and ckpt_path.exists() and not latest_path.exists():
            ck = torch.load(ckpt_path, map_location=self.dev)
            self.model.load_state_dict(ck["model_state"])
            start_epoch = ck.get("epoch", 1) + 1
            best_f1 = float(ck.get("best_val_f1", 0.0))
            best_thr = float(ck.get("best_threshold", self.cfg.DETECTION_THRESHOLD))

        for ep in range(start_epoch, start_epoch + epochs):
            self.model.train()
            loss_sum = correct = total = 0
            pbar = tqdm(tr_l, desc=f"Det train ep {ep}", leave=False)
            for X, y in pbar:
                X = X.to(self.dev, non_blocking=True)
                y = y.to(self.dev, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast("cuda", enabled=self.cfg.USE_AMP and self.cfg.DEVICE == "cuda"):
                    out = self.model(X)
                    loss = crit(out, y)
                scaler.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 2.0)
                scaler.step(opt)
                scaler.update()
                loss_sum += loss.item() * X.size(0)
                correct += (out.argmax(1) == y).sum().item()
                total += X.size(0)
                pbar.set_postfix({"loss": f"{loss_sum / max(total, 1):.4f}", "acc": f"{100 * correct / max(total, 1):.1f}%"})

            tr_loss = loss_sum / max(total, 1)
            tr_acc = 100 * correct / max(total, 1)
            y_val, p_val = collect_val_probs(self.model, va_l, self.dev)
            thr, _ = find_best_threshold(y_val, p_val, beta=1.0)
            val_metrics = evaluate_binary_metrics(y_val, p_val, thr)
            val_acc = 100.0 * val_metrics["accuracy"]
            val_f1 = val_metrics["f1"]
            sched.step()
            print(f"  Det Ep {ep:3d} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.1f}%  val_acc={val_acc:.1f}%  val_f1={val_f1:.4f} thr={thr:.3f}")
            self._save(latest_path, ep, val_metrics)
            if ep not in logged_epochs:
                logger.log(epoch=ep, tr_loss=round(tr_loss, 6), tr_acc=round(tr_acc, 3), val_acc=round(val_acc, 3),
                           val_f1=round(val_f1, 6), val_thr=round(thr, 4), val_auprc=round(val_metrics["auprc"], 6) if not math.isnan(val_metrics["auprc"]) else "nan")
                logged_epochs.add(ep)
            if val_f1 > best_f1:
                best_f1 = val_f1
                best_thr = thr
                self._save(ckpt_path, ep, val_metrics)
                print("   ✨ New best!")
                self.cfg.DETECTION_THRESHOLD = float(best_thr)

        print("\n🎯 Final detection test:")
        best_ck = torch.load(ckpt_path, map_location=self.dev) if ckpt_path.exists() else None
        if best_ck is not None:
            self.model.load_state_dict(best_ck["model_state"])
            test_thr = float(best_ck.get("best_threshold", self.cfg.DETECTION_THRESHOLD))
        else:
            test_thr = self.cfg.DETECTION_THRESHOLD
        y_test, p_test = collect_val_probs(self.model, te_l, self.dev)
        print_detection_report(y_test, p_test, test_thr)
        self.cfg.DETECTION_THRESHOLD = float(test_thr)

    def _save(self, path, epoch, metrics):
        path.parent.mkdir(parents=True, exist_ok=True)
        payload = {
            "model_state": self.model.state_dict(),
            "epoch": epoch,
            "best_val_acc": float(metrics.get("accuracy", 0.0) * 100.0),
            "best_val_f1": float(metrics.get("f1", 0.0)),
            "best_threshold": float(metrics.get("threshold", self.cfg.DETECTION_THRESHOLD)),
            "val_metrics": metrics,
        }
        torch.save(payload, path)
        print(f"💾 Saved {path.name} (ep={epoch}, f1={payload['best_val_f1']:.4f}, thr={payload['best_threshold']:.3f})")


def load_detection_model(cfg: Config) -> DetectionCNN:
    global _det_model
    if _det_model is not None:
        return _det_model
    ckpt = cfg.DRIVE_MODELS / "best_detection.pth"
    if not ckpt.exists():
        raise FileNotFoundError(f"No detection checkpoint at {ckpt}. Run train_detection() first.")
    dev = torch.device(cfg.DEVICE)
    data = torch.load(ckpt, map_location=dev)
    m = DetectionCNN().to(dev)
    m.load_state_dict(data["model_state"])
    m.eval()
    cfg.DETECTION_THRESHOLD = float(data.get("best_threshold", cfg.DETECTION_THRESHOLD))
    _det_model = m
    print(f"✅ Detection model loaded (ep={data.get('epoch', '?')}, f1={data.get('best_val_f1', '?')}, thr={cfg.DETECTION_THRESHOLD:.3f})")
    return m


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

config = Config()

if __name__ == "__main__":
    print("🚁 Drone Detection & Localization v14")
    print("=" * 65)
    print("This file is a safer, deployment-oriented upgrade of v13.")
    print("Suggested usage:")
    print("  train_all(det_epochs=5, loc_epochs=5, resume=False, force_rebuild_cache=True)")
    print("  plot_training_logs(config)")
    print("  analyse_audio_file('drone.mp3', config)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🚁 Drone Detection & Localization v14
This file is a safer, deployment-oriented upgrade of v13.
Suggested usage:
  train_all(det_epochs=5, loc_epochs=5, resume=False, force_rebuild_cache=True)
  plot_training_logs(config)
  analyse_audio_file('drone.mp3', config)


In [3]:
train_all(det_epochs=5, loc_epochs=5, resume=False, force_rebuild_cache=True)

  STAGE 1 — Detection Model (real + scraped + mixed + synthetic)
⚠️ Detection dataset incomplete. Rebuilding …
📥 Downloading DroneAudioDataset …
📦 Extracting …
✅ Detection dataset processed
🌐 Multi-source audio scraping …
  ℹ️  No FREESOUND_API_KEY — skipping Freesound.org
  🔎 FreeSound.io (key-free) …
    ⚠️  FSio (drone): HTTPSConnectionPool(host='freesound.io', port=443): Max retries exceeded with url: /api/sounds/search?query=drone&limit=20 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e295236b860>: Failed to resolve 'freesound.io' ([Errno -5] No address associated with hostname)"))
    ⚠️  FSio (uav): HTTPSConnectionPool(host='freesound.io', port=443): Max retries exceeded with url: /api/sounds/search?query=uav&limit=20 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7e2952415520>: Failed to resolve 'freesound.io' ([Errno -5] No address associated with hostname)"))
    ⚠️  FSio (quadcopter): HTTPSConnectionPool(hos

Mixing drone+background:   0%|          | 0/1200 [00:00<?, ?it/s]

✅ Mixed drone generation complete (1200 files).

📊 Detection WAV split counts
   train / drone     : 1952
   train / non_drone : 7264
   val   / drone     : 380
   val   / non_drone : 1557
   test  / drone     : 200
   test  / non_drone : 1556
🎵 Building mel cache from processed WAVs [v14 features] …


Mel cache:   0%|          | 0/12909 [00:00<?, ?it/s]

✅ Mel cache built (12909 new files).
🔬 Injecting 500 synthetic drone feature tensors into detection cache …


...

Loc train ep 5:   0%|          | 0/5 [00:00<?, ?it/s]

Loc eval:   0%|          | 0/1 [00:00<?, ?it/s]

  Loc Ep   5 | tr_loss=2.02259 val_loss=1.95179 mae_az=71.85° mae_dist=4.503m mae_ht=4.990m
💾 Saved latest_localization.pth (ep=5, val_loss=1.95179)

🎯 Final localization test:


Loc eval:   0%|          | 0/1 [00:00<?, ?it/s]

  Val loss=1.95171  MAE az=72.95°  dist=4.178m  ht=5.098m

✅ Both models trained.


In [4]:
plot_training_logs(config)

💾 Plot saved: /content/drive/MyDrive/drone_v14/logs/training_curves.png


In [6]:
analyse_audio_file('Drone 2.mp3', config)

✅ Detection model loaded (ep=5, f1=0.9907773386034255, thr=0.845)
🔧 Using LocalizationCNN (full-capacity mode)
✅ Localization model loaded (ep=4, val_loss=1.90863)

🎵 Drone 2.mp3  (2.1s)  |  10 segments  |  mode=direct mono
  Seg   1  🚁  conf=0.910  rms=-22.0dB
  Seg   2  🚁  conf=0.910  rms=-22.0dB
  Seg   3  🚁  conf=0.910  rms=-22.0dB
  Seg   4  🚁  conf=0.910  rms=-22.0dB
  Seg   5  🚁  conf=0.910  rms=-22.0dB
  Seg   6  🚁  conf=0.910  rms=-22.0dB
  Seg   7  🚁  conf=0.910  rms=-22.0dB
  Seg   8  🚁  conf=0.910  rms=-22.0dB
  Seg   9  🚁  conf=0.910  rms=-22.0dB
  Seg  10  🚁  conf=0.910  rms=-22.0dB

  📊 10/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_Drone 2.png


{'segments': [{'seg': 1,
   't_start': 0.0,
   'detected': True,
   'prob': 0.909678675802877,
   'cnn_probability': 0.9999829530715942,
   'heuristic_probability': 0.5133993037324022,
   'xy': array([0.1       , 0.05773333], dtype=float32),
   'loc': {'azimuth_deg': 0.0,
    'distance_m': 0.0,
    'height_m': 0.0,
    'xy_position': array([0.1       , 0.05773333], dtype=float32)},
   'mel': array([[-80.      , -36.04334 , -24.022257, ..., -80.      , -80.      ,
           -80.      ],
          [-80.      , -35.45653 , -23.663094, ..., -80.      , -80.      ,
           -80.      ],
          [-74.535034, -35.436375, -20.34646 , ..., -80.      , -80.      ,
           -80.      ],
          ...,
          [-80.      , -78.40657 , -60.756283, ..., -80.      , -80.      ,
           -80.      ],
          [-80.      , -75.85281 , -59.543434, ..., -80.      , -80.      ,
           -80.      ],
          [-80.      , -79.1535  , -60.73817 , ..., -80.      , -80.      ,
           -80.  

In [15]:
files = [
    'drilling_fold9_54976_5.wav',
    'engine_idling_fold9_39856_8.wav',
    'Drone_Noise_Test_-_DJI_Avata_vs_DJI_FPV_vs_DJI_Mavic_3_vs_DJI_Air_2S_compared_seg_012_0040548_0044548.wav',
    'Drone_Noise_Test_-_DJI_Avata_vs_DJI_FPV_vs_DJI_Mavic_3_vs_DJI_Air_2S_compared_seg_016_0055548_0059548.wav',
    'DJI drones noise comparson.mp3'
]
for f in files:
    analyse_audio_file(f, config)


🎵 drilling_fold9_54976_5.wav  (4.0s)  |  10 segments  |  mode=direct mono
  Seg   1  🚁  conf=0.903  rms=-23.0dB
  Seg   2  🚁  conf=1.000  rms=-23.5dB
  Seg   3  🚁  conf=1.000  rms=-23.5dB
  Seg   4  🚁  conf=1.000  rms=-23.5dB
  Seg   5  🚁  conf=1.000  rms=-23.5dB
  Seg   6  🚁  conf=1.000  rms=-23.5dB
  Seg   7  🚁  conf=1.000  rms=-23.5dB
  Seg   8  🚁  conf=1.000  rms=-23.5dB
  Seg   9  🚁  conf=1.000  rms=-23.5dB
  Seg  10  🚁  conf=1.000  rms=-23.5dB

  📊 10/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_drilling_fold9_54976_5.png

🎵 engine_idling_fold9_39856_8.wav  (4.0s)  |  10 segments  |  mode=direct mono
  Seg   1  🚁  conf=0.922  rms=-7.6dB
  Seg   2  🚁  conf=0.922  rms=-7.6dB
  Seg   3  🚁  conf=0.922  rms=-7.6dB
  Seg   4  🚁  conf=0.922  rms=-7.6dB
  Seg   5  🚁  conf=0.922  rms=-7.6dB
  Seg   6  🚁  conf=0.922  rms=-7.6dB
  Seg   7  🚁  conf=0.922  rms=-7.6dB
  Seg   8  🚁  conf=0.922  rms=-7.6dB
  Seg   9  🚁  conf=0.922  rms=-7.6dB
  Seg  10  🚁  conf=0.922  rms=-7.6dB

  📊 10/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_engine_idling_fold9_39856_8.png

🎵 Drone_Noise_Test_-_DJI_Avata_vs_DJI_FPV_vs_DJI_Mavic_3_vs_DJI_Air_2S_compared_seg_012_0040548_0044548.wav  (4.0s)  |  10 segments  |  mode=direct mono
  Seg   1  🚁  conf=1.000  rms=-15.9dB
  Seg   2  🚁  conf=1.000  rms=-16.2dB
  Seg   3  🚁  conf=1.000  rms=-16.2dB
  Seg   4  🚁  conf=1.000  rms=-16.2dB
  Seg   5  🚁  conf=1.000  rms=-16.2dB
  Seg   6  🚁  conf=1.000  rms=-16.2dB
  Seg   7  🚁  conf=1.000  rms=-16.2dB
  Seg   8  🚁  conf=1.000  rms=-16.2dB
  Seg   9  🚁  conf=1.000  rms=-16.2dB
  Seg  10  🚁  conf=1.000  rms=-16.2dB

  📊 10/10 detected  |  1 confirmed track(s)


...

💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_Drone_Noise_Test_-_DJI_Avata_vs_DJI_FPV_vs_DJI_Mavic_3_vs_DJI_Air_2S_compared_seg_012_0040548_0044548.png

🎵 Drone_Noise_Test_-_DJI_Avata_vs_DJI_FPV_vs_DJI_Mavic_3_vs_DJI_Air_2S_compared_seg_016_0055548_0059548.wav  (3.6s)  |  10 segments  |  mode=direct mono
  Seg   1  🚁  conf=1.000  rms=-14.5dB
  Seg   2  🚁  conf=1.000  rms=-14.5dB
  Seg   3  🚁  conf=1.000  rms=-14.5dB
  Seg   4  🚁  conf=1.000  rms=-14.5dB
  Seg   5  🚁  conf=1.000  rms=-14.5dB
  Seg   6  🚁  conf=1.000  rms=-14.5dB
  Seg   7  🚁  conf=1.000  rms=-14.5dB
  Seg   8  🚁  conf=1.000  rms=-14.5dB
  Seg   9  🚁  conf=1.000  rms=-14.5dB
  Seg  10  🚁  conf=1.000  rms=-14.5dB

  📊 10/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_Drone_Noise_Test_-_DJI_Avata_vs_DJI_FPV_vs_DJI_Mavic_3_vs_DJI_Air_2S_compared_seg_016_0055548_0059548.png

🎵 DJI drones noise comparson.mp3  (179.4s)  |  10 segments  |  mode=direct mono
  Seg   1  🌳  conf=0.060  rms=-60.1dB
  Seg   2  🌳  conf=0.218  rms=-29.8dB
  Seg   3  🌳  conf=0.267  rms=-20.2dB
  Seg   4  🚁  conf=0.893  rms=-24.7dB
  Seg   5  🌳  conf=0.271  rms=-18.1dB
  Seg   6  🌳  conf=0.138  rms=-18.3dB
  Seg   7  🌳  conf=0.104  rms=-28.1dB
  Seg   8  🌳  conf=0.254  rms=-41.5dB
  Seg   9  🌳  conf=0.295  rms=-51.1dB
  Seg  10  🌳  conf=0.550  rms=-33.2dB

  📊 1/10 detected  |  0 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_DJI drones noise comparson.png


In [17]:
analyse_audio_file('Drone Noise Test - DJI Avata vs DJI FPV vs DJI Mavic 3 vs DJI Air 2S compared.mp3', config)


🎵 Drone Noise Test - DJI Avata vs DJI FPV vs DJI Mavic 3 vs DJI Air 2S compared.mp3  (69.1s)  |  10 segments  |  mode=direct mono
  Seg   1  🌳  conf=0.184  rms=-35.1dB
  Seg   2  🌳  conf=0.294  rms=-14.1dB
  Seg   3  🌳  conf=0.550  rms=-21.4dB
  Seg   4  🌳  conf=0.550  rms=-12.4dB
  Seg   5  🌳  conf=0.206  rms=-12.0dB
  Seg   6  🚁  conf=1.000  rms=-27.0dB
  Seg   7  🌳  conf=0.290  rms=-28.0dB
  Seg   8  🚁  conf=1.000  rms=-27.4dB
  Seg   9  🌳  conf=0.092  rms=-37.4dB
  Seg  10  🌳  conf=0.024  rms=-160.0dB

  📊 2/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_Drone Noise Test - DJI Avata vs DJI FPV vs DJI Mavic 3 vs DJI Air 2S compared.png


{'segments': [{'seg': 1,
   't_start': 0.0,
   'detected': False,
   'prob': 0.1841397953725428,
   'cnn_probability': 1.1458119577575177e-20,
   'heuristic_probability': 0.526113701064408,
   'xy': None,
   'loc': None,
   'mel': array([[-80.      , -80.      , -80.      , ..., -36.405724, -37.098846,
           -34.613564],
          [-80.      , -80.      , -80.      , ..., -30.256763, -31.92704 ,
           -32.13351 ],
          [-80.      , -80.      , -80.      , ..., -22.72483 , -25.25614 ,
           -28.402271],
          ...,
          [-80.      , -80.      , -80.      , ..., -34.566372, -35.041775,
           -36.352173],
          [-80.      , -80.      , -80.      , ..., -33.404892, -34.96033 ,
           -37.820053],
          [-80.      , -80.      , -80.      , ..., -32.628704, -32.150345,
           -33.29374 ]], dtype=float32),
   'rms_db': -35.10289764404297},
  {'seg': 2,
   't_start': 7.347301587301588,
   'detected': False,
   'prob': 0.2938089517500638,
   'cnn

In [10]:
analyse_audio_file('dragon-studio-helicopter-sound-8d-372463.mp3', config)


🎵 dragon-studio-helicopter-sound-8d-372463.mp3  (10.0s)  |  10 segments  |  mode=direct mono
  Seg   1  🚁  conf=1.000  rms=-21.0dB
  Seg   2  🚁  conf=1.000  rms=-16.6dB
  Seg   3  🚁  conf=1.000  rms=-21.6dB
  Seg   4  🚁  conf=1.000  rms=-23.4dB
  Seg   5  🚁  conf=1.000  rms=-23.4dB
  Seg   6  🚁  conf=1.000  rms=-23.4dB
  Seg   7  🚁  conf=1.000  rms=-23.4dB
  Seg   8  🚁  conf=1.000  rms=-23.4dB
  Seg   9  🚁  conf=1.000  rms=-23.4dB
  Seg  10  🚁  conf=1.000  rms=-23.4dB

  📊 10/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_dragon-studio-helicopter-sound-8d-372463.png


{'segments': [{'seg': 1,
   't_start': 0.0,
   'detected': True,
   'prob': 1.0,
   'cnn_probability': 1.0,
   'heuristic_probability': 0.9606085737446045,
   'xy': array([0.1       , 0.05773333], dtype=float32),
   'loc': {'azimuth_deg': 0.0,
    'distance_m': 0.0,
    'height_m': 0.0,
    'xy_position': array([0.1       , 0.05773333], dtype=float32)},
   'mel': array([[-80.      , -80.      , -80.      , ..., -12.262793,  -4.200451,
            -4.640072],
          [-80.      , -80.      , -80.      , ..., -17.1117  , -14.248606,
           -17.178373],
          [-80.      , -80.      , -80.      , ..., -17.274876, -14.236057,
           -16.213417],
          ...,
          [-80.      , -80.      , -80.      , ..., -72.828354, -65.21892 ,
           -52.837605],
          [-80.      , -80.      , -80.      , ..., -72.35199 , -65.25768 ,
           -53.01282 ],
          [-80.      , -80.      , -80.      , ..., -73.09578 , -65.773315,
           -53.561226]], dtype=float32),
   'r

In [11]:
analyse_audio_file('klemenflerin-drone-fly-397287(1).mp3', config)


🎵 klemenflerin-drone-fly-397287(1).mp3  (247.9s)  |  10 segments  |  mode=direct mono
  Seg   1  🌳  conf=0.116  rms=-51.8dB
  Seg   2  🚁  conf=1.000  rms=-13.6dB
  Seg   3  🚁  conf=1.000  rms=-18.8dB
  Seg   4  🚁  conf=1.000  rms=-16.1dB
  Seg   5  🚁  conf=1.000  rms=-20.7dB
  Seg   6  🚁  conf=1.000  rms=-15.5dB
  Seg   7  🚁  conf=1.000  rms=-14.2dB
  Seg   8  🚁  conf=1.000  rms=-13.8dB
  Seg   9  🚁  conf=1.000  rms=-15.8dB
  Seg  10  🌳  conf=0.060  rms=-51.7dB

  📊 8/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_klemenflerin-drone-fly-397287(1).png


{'segments': [{'seg': 1,
   't_start': 0.0,
   'detected': False,
   'prob': 0.1163974086844633,
   'cnn_probability': 3.28810415050453e-23,
   'heuristic_probability': 0.3325640248127523,
   'xy': None,
   'loc': None,
   'mel': array([[-74.5444  , -15.672569,  -8.68976 , ..., -16.089039, -15.441456,
           -10.485872],
          [-80.      , -19.384956, -14.069431, ..., -29.05272 , -29.57539 ,
           -18.408863],
          [-80.      , -19.906937, -10.138813, ..., -35.519833, -30.331947,
           -20.278988],
          ...,
          [-80.      , -32.681473, -21.462494, ..., -43.842953, -44.28202 ,
           -46.997673],
          [-78.61818 , -32.97802 , -21.441807, ..., -44.472263, -49.778656,
           -49.879944],
          [-76.49312 , -36.182   , -22.563114, ..., -50.758064, -55.300613,
           -51.012222]], dtype=float32),
   'rms_db': -51.79428482055664},
  {'seg': 2,
   't_start': 27.213333333333335,
   'detected': True,
   'prob': 1.0,
   'cnn_probability': 1

In [8]:
analyse_audio_file('air_conditioner_fold8_162103_4.wav', config)


🎵 air_conditioner_fold8_162103_4.wav  (0.7s)  |  10 segments  |  mode=direct mono
  Seg   1  🌳  conf=0.550  rms=-32.1dB
  Seg   2  🌳  conf=0.550  rms=-32.1dB
  Seg   3  🌳  conf=0.550  rms=-32.1dB
  Seg   4  🌳  conf=0.550  rms=-32.1dB
  Seg   5  🌳  conf=0.550  rms=-32.1dB
  Seg   6  🌳  conf=0.550  rms=-32.1dB
  Seg   7  🌳  conf=0.550  rms=-32.1dB
  Seg   8  🌳  conf=0.550  rms=-32.1dB
  Seg   9  🌳  conf=0.550  rms=-32.1dB
  Seg  10  🌳  conf=0.550  rms=-32.1dB

  📊 0/10 detected  |  0 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_air_conditioner_fold8_162103_4.png


{'segments': [{'seg': 1,
   't_start': 0.0,
   'detected': False,
   'prob': 0.55,
   'cnn_probability': 3.5709057363164695e-23,
   'heuristic_probability': 0.857688037788734,
   'xy': None,
   'loc': None,
   'mel': array([[-22.374516 , -13.657793 ,  -5.752126 , ..., -80.       ,
           -80.       , -80.       ],
          [-21.279243 , -14.8174515,  -9.312254 , ..., -80.       ,
           -80.       , -80.       ],
          [-19.942854 , -14.1007   , -12.914326 , ..., -80.       ,
           -80.       , -80.       ],
          ...,
          [-44.29157  , -41.287354 , -43.804142 , ..., -80.       ,
           -80.       , -80.       ],
          [-44.390724 , -42.462093 , -42.355515 , ..., -80.       ,
           -80.       , -80.       ],
          [-47.166725 , -46.281868 , -44.5594   , ..., -80.       ,
           -80.       , -80.       ]], dtype=float32),
   'rms_db': -32.07848358154297},
  {'seg': 2,
   't_start': 0.0,
   'detected': False,
   'prob': 0.55,
   'cnn_proba

In [18]:
files = ['siren_fold9_66601_3.wav', 'siren_fold9_66601_5.wav']
for f in files:
    analyse_audio_file(f, config)


🎵 siren_fold9_66601_3.wav  (4.0s)  |  10 segments  |  mode=direct mono
  Seg   1  🚁  conf=0.765  rms=-32.8dB
  Seg   2  🚁  conf=0.762  rms=-33.4dB
  Seg   3  🚁  conf=0.762  rms=-33.4dB
  Seg   4  🚁  conf=0.762  rms=-33.4dB
  Seg   5  🚁  conf=0.762  rms=-33.4dB
  Seg   6  🚁  conf=0.762  rms=-33.4dB
  Seg   7  🚁  conf=0.762  rms=-33.4dB
  Seg   8  🚁  conf=0.762  rms=-33.4dB
  Seg   9  🚁  conf=0.762  rms=-33.4dB
  Seg  10  🚁  conf=0.762  rms=-33.4dB

  📊 10/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_siren_fold9_66601_3.png

🎵 siren_fold9_66601_5.wav  (4.0s)  |  10 segments  |  mode=direct mono
  Seg   1  🚁  conf=0.760  rms=-33.8dB
  Seg   2  🚁  conf=0.766  rms=-32.7dB
  Seg   3  🚁  conf=0.766  rms=-32.7dB
  Seg   4  🚁  conf=0.766  rms=-32.7dB
  Seg   5  🚁  conf=0.766  rms=-32.7dB
  Seg   6  🚁  conf=0.766  rms=-32.7dB
  Seg   7  🚁  conf=0.766  rms=-32.7dB
  Seg   8  🚁  conf=0.766  rms=-32.7dB
  Seg   9  🚁  conf=0.766  rms=-32.7dB
  Seg  10  🚁  conf=0.766  rms=-32.7dB

  📊 10/10 detected  |  1 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_siren_fold9_66601_5.png


In [9]:
analyse_audio_file('siren_fold9_159755_3.wav', config)


🎵 siren_fold9_159755_3.wav  (4.0s)  |  10 segments  |  mode=direct mono
  Seg   1  🌳  conf=0.225  rms=-36.2dB
  Seg   2  🌳  conf=0.258  rms=-30.9dB
  Seg   3  🌳  conf=0.258  rms=-30.9dB
  Seg   4  🌳  conf=0.258  rms=-30.9dB
  Seg   5  🌳  conf=0.258  rms=-30.9dB
  Seg   6  🌳  conf=0.258  rms=-30.9dB
  Seg   7  🌳  conf=0.258  rms=-30.9dB
  Seg   8  🌳  conf=0.258  rms=-30.9dB
  Seg   9  🌳  conf=0.258  rms=-30.9dB
  Seg  10  🌳  conf=0.258  rms=-30.9dB

  📊 0/10 detected  |  0 confirmed track(s)


/tmp/ipykernel_2817/4200806280.py:2351: UserWarning: Glyph 128641 (\N{HELICOPTER}) missing from font(s) DejaVu Sans.
  plt.savefig(str(save_path), dpi=150, bbox_inches="tight")


💾 Analysis dashboard saved: /content/drive/MyDrive/drone_v14/logs/plots/analysis_siren_fold9_159755_3.png


{'segments': [{'seg': 1,
   't_start': 0.0,
   'detected': False,
   'prob': 0.225472931169933,
   'cnn_probability': 1.9919297556798246e-19,
   'heuristic_probability': 0.6442083747712372,
   'xy': None,
   'loc': None,
   'mel': array([[-51.017017, -49.031338, -45.021355, ..., -42.798298, -45.923847,
           -45.383762],
          [-45.314114, -42.977818, -40.477024, ..., -37.643505, -43.849087,
           -45.826588],
          [-41.421814, -39.114365, -40.202877, ..., -36.684727, -42.520973,
           -42.027695],
          ...,
          [-80.      , -80.      , -80.      , ..., -80.      , -71.774826,
           -59.14508 ],
          [-80.      , -80.      , -80.      , ..., -57.205082, -58.555748,
           -58.141476],
          [-80.      , -80.      , -80.      , ..., -56.99724 , -57.478436,
           -57.318867]], dtype=float32),
   'rms_db': -36.20369338989258},
  {'seg': 2,
   't_start': 1.0,
   'detected': False,
   'prob': 0.2583074510607386,
   'cnn_probability':

In [12]:
quick_demo(cfg=config)


🚁 QUICK DEMO — Multi-drone synthetic test
──────────────────────────────────────────────────
  Detected: True  (conf=1.000)
  Drones found: 3
    Drone 1: (0.10,0.05)m  az=-90.0°  dist=0.01m  err=1.444m
    Drone 2: (0.10,25.06)m  az=90.0°  dist=25.00m  err=23.284m
    Drone 3: (0.10,0.36)m  az=90.0°  dist=0.31m
💾 Saved: /content/drive/MyDrive/drone_v14/logs/plots/multi_drone_positions.png


{'detected': True,
 'probability': 1.0,
 'cnn_probability': 1.0,
 'heuristic_probability': 0.9817799186925428,
 'drones': [{'xy_position': array([0.1       , 0.04710528], dtype=float32),
   'azimuth_deg': -90.0,
   'distance_m': 0.010628055781126022,
   'tdoa_residual': np.float64(2.5503242603162995e-24),
   'confidence_radius': 20.0},
  {'xy_position': array([ 0.09999928, 25.057734  ], dtype=float32),
   'azimuth_deg': 90.00000165632088,
   'distance_m': 25.0,
   'tdoa_residual': np.float64(1.896367877590738e-08),
   'confidence_radius': 20.0},
  {'xy_position': array([0.1       , 0.36455366], dtype=float32),
   'azimuth_deg': 90.0,
   'distance_m': 0.3068203330039978,
   'tdoa_residual': np.float64(4.1565545323749426e-23),
   'confidence_radius': 20.0}],
 'tracks': []}

In [13]:
launch_ui(cfg=config)

Saving drilling_fold9_146249_0.wav to drilling_fold9_146249_0.wav
Saving drilling_fold9_14111_6.wav to drilling_fold9_14111_6.wav


Saving gd_salman-helicopter-ambience-353004.mp3 to gd_salman-helicopter-ambience-353004.mp3
Saving 11325622-helicopter-sound-effect-241421.mp3 to 11325622-helicopter-sound-effect-241421.mp3
Saving freesound_community-rumbling-drone-59370.mp3 to freesound_community-rumbling-drone-59370.mp3
